# SE-RDHEI — reproduce every result of the CAEE manuscript

This notebook recomputes **every measured number and every data figure** in the manuscript from the images you upload.

**This is the final run (block size B = 2).** It writes to a new output folder (`results_B2`, or `SE-RDHEI-results-B2` on Drive) and refuses to reuse results of earlier runs. All ciphertext statistics and key-sensitivity tests are now measured on the specified AES-256-CTR construction. Please include **all four** datasets: USC-SIPI, BOSSbase, **BOWS-2** and UCID.

| Output | Manuscript |
|---|---|
| capacity, side information, container, effective rate, exactness | Table 3, Fig. 5 |
| per-dataset means (USC-SIPI, BOSSbase, BOWS-2, UCID) | Table 4, Figs. 5–6 |
| matched comparison with the reimplemented BPC method (covers **and** database samples) | Table 5 |
| block-size sweep (B = 2–32) and level sweep | Fig. 7 |
| ablations (overflow under the classic ordering, modular-lifting room) | Table 6, Fig. 8 |
| runtime | Section 6.6 |
| ciphertext entropy, χ², correlation, histograms, 3-D correlation plots | Table 7, Figs. 9–10 |
| NPCR/UACI with Wu–Noonan–Agaian tests (master key of the specified construction; A1, A2 on the base) | Fig. 11 |
| *t*-map leakage, wrong-key recovery, credential ablation | Section 7 |
| specified construction: round trip, tampering, AES-256-CTR statistics | Section 4.6, Algorithm 3, Section 7 |
| block-minimum efficiency, leakage-reduced descriptor, 20-key statistics | Table 8, Proposition 1 |
| visual results montage (cover, encrypted, marked, recovered) | Fig. 4 |

**How to use:** run the cells from top to bottom (*Runtime → Run all*). Step 2 asks for your data. When it finishes, the last cell downloads **`results_SE-RDHEI.zip`**. Send that zip back and the manuscript will be updated from it.

**Time:** about 2–3 hours on a standard Colab CPU for 12 covers plus 3 × 500 database images (set `N_PER_DB = 100` for a 40–60 minute run with wider error bars). No GPU is needed. Results are cached per image, so if Colab disconnects, re-running resumes where it stopped (keep the output folder on Google Drive for this).

**Checks built in:** section 0 of the report compares every key-independent value (capacity, side information, sweeps, room) against your recorded MATLAB run, recomputed at the MATLAB block size B = 4 for this check. All rows should read *match*. A mismatch means a cover file differs from the one used in MATLAB.

## 1. Setup
Installs the few missing packages and the Times-like font used in the figures, then writes the source files. The code cells in this step are hidden; you don't need to open them.

In [ ]:
#@title Install packages and fonts
!pip -q install numba soundfile cryptography
!apt-get -qq install -y fonts-texgyre > /dev/null 2>&1 || echo "font install skipped (figures fall back to a similar serif font)"
!rm -rf ~/.cache/matplotlib
import os
os.makedirs('/content/se_rdhei', exist_ok=True)
print('ready')

In [ ]:
%%writefile /content/se_rdhei/se_full.py
"""SE-RDHEI measurement engine: a line-by-line Python port of the batch mode of
SE_RDHEI.m (functions keys, keystream, permutation, encrypt_embed,
extract_decrypt, tmap_pack, one_image and the helpers they call).

Capacity, side information, container size and exactness do not depend on
the key, so they reproduce the MATLAB run bit for bit on the same cover file.
Ciphertext statistics depend on the key, the nonce and the two credentials
A1, A2; they reproduce MATLAB exactly only when the same audio clips are
supplied (export handel.mat and gong.mat from MATLAB), and otherwise agree
statistically.
"""
import hashlib, math, time
import numpy as np
from scipy.stats import chi2 as _chi2
from scipy.special import erfcinv

try:
    from numba import njit
except Exception:                                    # numba is optional
    def njit(*a, **k):
        if a and callable(a[0]):
            return a[0]
        return lambda f: f

SUB = ['LL', 'HL', 'LH', 'HH']
import os as _os
# Block size B of the block minimum (Eq. 7). The manuscript uses B = 2; set the
# environment variable SE_BLOCK to run another size.
BLOCK = int(_os.environ.get('SE_BLOCK', '2'))
DEFAULT_KEY = bytes(range(32))
DEFAULT_NONCE = b'0123456789ab'
AUDIO_LEN = 160000                                   # 10 s at 16 kHz


# ------------------------------------------------------------- 5/3 lifting
def f1(x):
    x = x.astype(np.int64)
    e, o = x[:, 0::2], x[:, 1::2]
    en = np.concatenate([e[:, 1:], e[:, -1:]], axis=1)
    d = o - np.floor_divide(e + en, 2)
    dp = np.concatenate([d[:, :1], d[:, :-1]], axis=1)
    s = e + np.floor_divide(dp + d + 2, 4)
    return s, d


def i1(s, d):
    dp = np.concatenate([d[:, :1], d[:, :-1]], axis=1)
    e = s - np.floor_divide(dp + d + 2, 4)
    en = np.concatenate([e[:, 1:], e[:, -1:]], axis=1)
    o = d + np.floor_divide(e + en, 2)
    x = np.zeros((e.shape[0], 2 * e.shape[1]), dtype=np.int64)
    x[:, 0::2], x[:, 1::2] = e, o
    return x


def iwt2(I):
    L, H = f1(I)
    LL, LH = f1(L.T)
    HL, HH = f1(H.T)
    return dict(LL=LL.T, LH=LH.T, HL=HL.T, HH=HH.T)


def iiwt2_raw(B):
    L = i1(B['LL'].T, B['LH'].T).T
    H = i1(B['HL'].T, B['HH'].T).T
    return i1(L, H)


def max_levels(shape, cap=4):
    M, N = shape
    L = 0
    while L < cap and M % 2 == 0 and N % 2 == 0 and min(M, N) >= 2:
        M //= 2; N //= 2; L += 1
    return max(L, 1)


def band_names(L):
    det = []
    for lv in range(1, L + 1):
        det += ['HL%d' % lv, 'LH%d' % lv, 'HH%d' % lv]
    return ['LL'] + det, det


def band_id(nm):
    if nm == 'LL':
        return 0
    return (int(nm[2:]) << 4) | (['HL', 'LH', 'HH'].index(nm[:2]) + 1)


def sub_of(nm):
    return 'LL' if nm == 'LL' else nm[:2]


def iwt2_multi(I, L):
    B, cur = {}, I.astype(np.int64)
    for lv in range(1, L + 1):
        Q = iwt2(cur)
        B['HL%d' % lv], B['LH%d' % lv], B['HH%d' % lv] = Q['HL'], Q['LH'], Q['HH']
        cur = Q['LL']
    B['LL'] = cur
    return B


def iiwt2_multi(B, L):
    cur = B['LL']
    for lv in range(L, 0, -1):
        cur = iiwt2_raw(dict(LL=cur, HL=B['HL%d' % lv], LH=B['LH%d' % lv], HH=B['HH%d' % lv]))
    return np.clip(cur, 0, 255).astype(np.uint8)


# ------------------------------------------ field coding and reservation
def band_width(v):
    lo, hi = int(v.min()), int(v.max())
    W = 2
    while not (-(2 ** (W - 1)) <= lo and hi <= 2 ** (W - 1) - 1):
        W += 1
    return W


def lead_ident(u, W):
    sg = (u >> (W - 1)) & 1
    n = np.ones(u.shape, dtype=np.int64)
    for k in range(W - 1, 0, -1):
        n += (((u >> (k - 1)) & 1) == sg) & (n == (W - k))
    return n


def block_t(u, W, tmax, BLK):
    t = np.minimum(lead_ident(u, W) - 1, tmax)
    hh, ww = t.shape
    bh, bw = hh // BLK, ww // BLK
    if bh == 0 or bw == 0:
        return np.zeros((0, 0), dtype=np.int64)
    return t[:bh * BLK, :bw * BLK].reshape(bh, BLK, bw, BLK).min(axis=(1, 3))


def expand_t(tm, shape, BLK):
    f = np.zeros(shape, dtype=np.int64)
    if tm.size:
        f[:tm.shape[0] * BLK, :tm.shape[1] * BLK] = np.kron(tm, np.ones((BLK, BLK), dtype=np.int64))
    return f


def vacate(u, t, W):
    return u & ((1 << (W - t)) - 1)


def restore(u, t, W):
    carrier = (u >> (W - t - 1)) & 1
    return (u & ((1 << (W - t)) - 1)) + ((1 << t) - 1) * (1 << (W - t)) * carrier


def slots(t, W):
    """Row-major flat index and 0-based bit position of every vacated slot."""
    f = t.reshape(-1)
    ii, bb = [], []
    for k in range(1, int(f.max(initial=0)) + 1):
        s = np.nonzero(f >= k)[0]
        ii.append(s); bb.append(np.full(s.size, W - k, dtype=np.int64))
    if not ii:
        return np.zeros(0, np.int64), np.zeros(0, np.int64)
    return np.concatenate(ii), np.concatenate(bb)


# ---------------------------------------------------- SHA-256 / CTR / HKDF
def sha(b):
    return hashlib.sha256(bytes(b)).digest()


def ctr(key, nbytes):
    key = bytes(key)
    nb = -(-nbytes // 32)
    return b''.join(hashlib.sha256(key + i.to_bytes(4, 'big')).digest() for i in range(nb))[:nbytes]


def hmac256(key, msg):
    key = bytes(key)
    if len(key) > 64:
        key = sha(key)
    k = key + b'\x00' * (64 - len(key))
    return sha(bytes(x ^ 92 for x in k) + sha(bytes(x ^ 54 for x in k) + bytes(msg)))


def hkdf(prk, info, n):
    out, t, i = b'', b'', 1
    while len(out) < n:
        t = hmac256(prk, t + bytes(info) + bytes([i])); out += t; i += 1
    return out[:n]


# ------------------------------------------------------ audio credentials
def canon_audio(y, fs, L=AUDIO_LEN):
    """fetch_audio(): mono, linear resample to 16 kHz, loop, crop, peak to int16."""
    y = np.asarray(y, dtype=float)
    if y.ndim > 1:
        y = y.mean(axis=1) if y.shape[0] > y.shape[1] else y.mean(axis=0)
    y = y.ravel()
    if fs != 16000:
        n2 = max(int(round(y.size * 16000 / fs)), 2)
        y = np.interp(np.linspace(0, 1, n2), np.linspace(0, 1, y.size), y)
    if y.size < L:
        y = np.tile(y, -(-L // y.size))
    y = y[:L]
    m = np.abs(y).max() or 1.0
    return np.floor(y / m * 32767 + 0.5).astype(np.int16)


def synthetic_audio(k, L=AUDIO_LEN):
    t = np.arange(L) / 16000
    if k == 1:
        y = np.sin(2 * np.pi * 220 * t) * (1 + 0.5 * np.sin(2 * np.pi * 3 * t))
    else:
        y = np.sign(np.sin(2 * np.pi * 90 * t)) * np.exp(-np.mod(t, 0.5) * 6)
    return canon_audio(y, 16000, L)


def load_audio(path):
    """WAV/FLAC/OGG (via soundfile or scipy) or a MATLAB .mat with y and Fs."""
    p = str(path).lower()
    if p.endswith('.mat'):
        from scipy.io import loadmat
        S = loadmat(path)
        return canon_audio(S['y'], float(np.squeeze(S['Fs'])))
    try:
        import soundfile as sf
        y, fs = sf.read(path, dtype='float64')
    except Exception:
        from scipy.io import wavfile
        fs, y = wavfile.read(path)
        if y.dtype.kind == 'i':
            y = y / float(2 ** (8 * y.dtype.itemsize - 1))
        elif y.dtype.kind == 'u':
            y = (y.astype(float) - 128) / 128
    return canon_audio(y, fs)


def _stft(a, Wn=1024, Hp=256):
    x = a.astype(float) / 32768
    win = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(Wn) / Wn)
    T = 1 + (x.size - Wn) // Hp
    idx = np.arange(T)[:, None] * Hp + np.arange(Wn)[None, :]
    return np.fft.fft(x[idx] * win, axis=1)[:, :Wn // 2 + 1]


def _round(x):                                       # MATLAB round, half away from 0
    return np.sign(x) * np.floor(np.abs(x) + 0.5)


def _pct(x, p):
    x = np.sort(x.ravel())
    return x[int(max(1, min(x.size, _round(p / 100 * x.size)))) - 1]


_STFT_CACHE = {}


def _stft_cached(a):
    h = sha(a.astype('<i2').tobytes())
    if h not in _STFT_CACHE:
        if len(_STFT_CACHE) > 8:
            _STFT_CACHE.clear()
        _STFT_CACHE[h] = _stft(a)
    return _STFT_CACHE[h]


def descriptor(a1, a2, alpha, theta):
    X1, X2 = _stft_cached(a1), _stft_cached(a2)
    Ln = np.log(1 + np.abs(X1 + alpha * X2 * np.exp(1j * theta)) ** 2)
    mag = np.abs(X1)
    f = np.arange(mag.shape[1], dtype=float)
    cen = (mag @ f) / (mag.sum(axis=1) + 1e-12)
    en = np.log(1 + (mag ** 2).sum(axis=1))
    return np.array([Ln.mean(), Ln.std(), Ln.max(), cen.mean(), cen.std(),
                     en.mean(), en.std(), _pct(Ln, 10), _pct(Ln, 50), _pct(Ln, 90)])


def qdesc(v):
    q = _round(np.clip(v, -1e4, 1e4) * 128)
    return b''.join((int(x) & (2 ** 64 - 1)).to_bytes(8, 'big') for x in q)


def keys(km, a1, a2, nonce, use_credentials=True):
    K = dict(nonce=bytes(nonce))
    K['h1'] = sha(a1.astype('<i2').tobytes()) if use_credentials else bytes(32)
    K['h2'] = sha(a2.astype('<i2').tobytes()) if use_credentials else bytes(32)
    prk = hmac256(b'SE-RDHEI-v2', km)
    for nm, lab in [('enc', b'ENC'), ('prm', b'PERM'), ('par', b'PARAM'), ('dh', b'DH'),
                    ('mac', b'MAC'), ('pad', b'PAD'), ('tmap', b'TMAP')]:
        K[nm] = hkdf(prk, lab, 32)
    K['F'] = {}
    for b, nm in enumerate(SUB, start=1):
        r = hkdf(K['par'], bytes([b]), 8)
        al = 0.2 + 1.6 * int.from_bytes(r[:4], 'big') / 2 ** 32
        th = 2 * np.pi * int.from_bytes(r[4:], 'big') / 2 ** 32
        K['F'][nm] = sha(qdesc(descriptor(a1, a2, al, th)))[:16] if use_credentials else bytes(16)
    return K


def keystream(K, nm, n, W):
    k = hkdf(K['enc'], bytes([band_id(nm)]) + K['h1'] + K['h2'] + K['F'][sub_of(nm)] + K['nonce'], 32)
    raw = np.frombuffer(ctr(k, 2 * n), dtype='<u2').astype(np.int64)
    return raw[:n] & ((1 << W) - 1)


@njit(cache=True)
def _fisher_yates(r):
    n = r.size
    p = np.arange(n)
    for i in range(n - 1, 0, -1):
        j = r[i] % np.uint64(i + 1)
        t = p[i]; p[i] = p[j]; p[j] = t
    return p


def permutation(K, nm, n):
    raw = ctr(bytes(K['prm']) + b'PERM' + bytes([band_id(nm)]) + K['nonce'], 8 * n)
    r = np.frombuffer(raw, dtype='>u8').astype(np.uint64)
    return _fisher_yates(r)


# ------------------------------------------------------- Huffman side info
def huff_total(cnt):
    """Total bits of an optimal Huffman code (any tie-break gives the same)."""
    import heapq
    w = [int(c) for c in cnt if c > 0]
    if not w:
        return 0
    if len(w) == 1:
        return w[0]
    heapq.heapify(w)
    tot = 0
    while len(w) > 1:
        a = heapq.heappop(w); b = heapq.heappop(w)
        tot += a + b; heapq.heappush(w, a + b)
    return tot


def tmap_pack(tm, det):
    tmax = max(2, max(int(tm[b].max(initial=0)) for b in det))
    alpha = tmax + 2
    sym = np.concatenate([tm[b].ravel() for b in det]) if det else np.zeros(0, np.int64)
    if sym.size == 0:
        return 0
    return huff_total(np.bincount(sym, minlength=alpha)) + 4 * alpha


# ------------------------------------------------- Algorithm 1 / Algorithm 2
def capacity(I, BLK=BLOCK, levels=0):
    """Key-independent quantities of encrypt_embed: raw, side, cap, container."""
    L = min(levels, max_levels(I.shape)) if levels > 0 else max_levels(I.shape)
    B = iwt2_multi(I, L)
    names, det = band_names(L)
    total, tm, cont, wd = 0, {}, 0, {}
    for nm in names:
        W = band_width(B[nm]); cont += W * B[nm].size; rb = 0
        if nm in det:
            u = np.mod(B[nm], 1 << W)
            tm[nm] = block_t(u, W, W - 2, BLK)
            rb = int(tm[nm].sum()) * BLK * BLK; total += rb
        wd[nm] = [int(W), int(B[nm].size), rb]      # width, coefficients, room bits
    side = tmap_pack(tm, det)
    return dict(L=L, raw=total, side=side, cap=max(total - side, 0), cont=cont, tm=tm, bands=wd)


def encrypt_embed(I, km, a1, a2, nonce, payload=None, BLK=BLOCK, levels=0, K=None,
                  rng=None, return_plain_cipher=False):
    L = min(levels, max_levels(I.shape)) if levels > 0 else max_levels(I.shape)
    B = iwt2_multi(I, L)
    names, det = band_names(L)
    if K is None:
        K = keys(km, a1, a2, nonce)
    Wd, shapes, tm, C, sl, total = {}, {}, {}, {}, {}, 0
    for nm in names:
        W = band_width(B[nm]); Wd[nm] = W; shapes[nm] = B[nm].shape
        u = np.mod(B[nm], 1 << W)
        if nm in det:
            tm[nm] = block_t(u, W, W - 2, BLK)
            tf = expand_t(tm[nm], u.shape, BLK)
            u = vacate(u, tf, W)
        n = u.size
        e = u.reshape(-1) ^ keystream(K, nm, n, W)
        p = permutation(K, nm, n)
        C[nm] = e[p]
        if nm in det:
            ii, bb = slots(tf, W)
            pinv = np.empty(n, np.int64); pinv[p] = np.arange(n)
            sl[nm] = (pinv[ii], bb); total += ii.size
    side = tmap_pack(tm, det)
    cap = max(total - side, 0)
    if payload is None:
        rng = rng or np.random.default_rng()
        payload = rng.integers(0, 2, cap).astype(np.uint8)
    if payload.size > total:
        raise ValueError('payload exceeds room')
    npad = total - payload.size
    if npad > 0:
        pb = np.unpackbits(np.frombuffer(ctr(K['pad'] + bytes(nonce), -(-npad // 8)), np.uint8))
        bits = np.concatenate([payload.astype(np.int64), pb[:npad].astype(np.int64)])
    else:
        bits = payload.astype(np.int64)
    Cplain = {nm: C[nm].copy() for nm in names} if return_plain_cipher else None
    off = 0
    for nm in det:
        pos, pb_ = sl[nm]; k = pos.size
        if k == 0:
            continue
        np.bitwise_and.at(C[nm], pos, ~(np.int64(1) << pb_))
        np.bitwise_or.at(C[nm], pos, bits[off:off + k] << pb_)
        off += k
    h = dict(tm=tm, Wd=Wd, nonce=bytes(nonce), shape=I.shape, levels=L, shapes=shapes,
             names=names, det=det, raw=total, side=side, cap=cap, total=total, blk=BLK)
    if return_plain_cipher:
        return C, h, payload, Cplain
    return C, h, payload


def extract_decrypt(C, h, km, a1, a2, K=None):
    BLK = h['blk']
    if K is None:
        K = keys(km, a1, a2, h['nonce'])
    out, Bh = [], {}
    for nm in h['names']:
        W, n, shp = h['Wd'][nm], C[nm].size, h['shapes'][nm]
        p = permutation(K, nm, n)
        isdet = nm in h['det']
        if isdet:
            tf = expand_t(h['tm'][nm], shp, BLK)
            ii, bb = slots(tf, W)
            if ii.size:
                pinv = np.empty(n, np.int64); pinv[p] = np.arange(n)
                out.append((C[nm][pinv[ii]] >> bb) & 1)
        e = np.empty(n, np.int64); e[p] = C[nm]
        u = (e ^ keystream(K, nm, n, W)).reshape(shp)
        if isdet:
            u = restore(u, tf, W)
        Bh[nm] = u - (1 << W) * (u >= (1 << (W - 1)))
    I = iiwt2_multi(Bh, h['levels'])
    pay = np.concatenate(out).astype(np.uint8) if out else np.zeros(0, np.uint8)
    return I, pay


# ------------------------------------------------------------ measurements
def packbits_cipher(C, Wd, names):
    parts = []
    for nm in names:
        W, v = Wd[nm], C[nm]
        m = ((v[:, None] >> np.arange(W - 1, -1, -1)[None, :]) & 1).astype(np.uint8)
        parts.append(m.reshape(-1))
    b = np.concatenate(parts)
    return np.packbits(b[:b.size // 8 * 8])


def entropy8(x):
    p = np.bincount(np.asarray(x).ravel(), minlength=256) / np.asarray(x).size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def chi2stat(x):
    o = np.bincount(np.asarray(x).ravel(), minlength=256)
    e = np.asarray(x).size / 256
    return float(((o - e) ** 2 / e).sum())


def square(x):
    x = np.asarray(x)
    if x.ndim == 1:
        s = int(math.isqrt(x.size)); x = x[:s * s].reshape(s, s)
    return x


def corr3(x):
    x = square(x).astype(float)
    out = []
    for dy, dx in [(0, 1), (1, 0), (1, 1)]:
        a = x[:x.shape[0] - dy, :x.shape[1] - dx].ravel()
        b = x[dy:, dx:].ravel()
        out.append(float((np.mean(a * b) - a.mean() * b.mean()) / (a.std() * b.std() + 2.2e-16)))
    return out


def npcr_uaci(a, b):
    """NPCR and UACI over the serialised ciphertext bytes. Both ciphertexts of a
    pair have the same length (widths and room depend on the cover only)."""
    assert a.size == b.size, 'ciphertexts of one pair must have equal length'
    a = a.astype(float); b = b.astype(float)
    return float(np.mean(a != b) * 100), float(np.mean(np.abs(a - b)) / 255 * 100)


def npcr_test(v, T, alpha=0.05):
    F = 255; z = -math.sqrt(2) * erfcinv(2 * alpha)
    crit = (F + z * math.sqrt(F / T)) / (F + 1) * 100
    return v > crit, crit


def uaci_test(v, T, alpha=0.05):
    F = 255; mu = (F + 2) / (3 * F + 3)
    sd = math.sqrt((F + 2) * (F ** 2 + 2 * F + 3) / (18 * (F + 1) ** 2 * F * T))
    z = math.sqrt(2) * erfcinv(alpha)
    lo, hi = (mu - z * sd) * 100, (mu + z * sd) * 100
    return lo < v < hi, lo, hi


def cipher_ref(I, km, a1, a2, nonce, BLK, seed, K=None):
    """Ciphertext with the room completely filled by a seeded random payload."""
    cp = capacity(I, BLK)
    full = np.random.default_rng(seed).integers(0, 2, cp['raw']).astype(np.uint8)
    C, h, _ = encrypt_embed(I, km, a1, a2, nonce, full, BLK, K=K)
    return packbits_cipher(C, h['Wd'], h['names'])


def miwt2(I):
    def f1m(x):
        M = 256
        e = np.mod(x[:, 0::2], M); o = np.mod(x[:, 1::2], M)
        en = np.concatenate([e[:, 1:], e[:, -1:]], axis=1)
        d = np.mod(o - np.floor_divide(e + en, 2), M)
        dp = np.concatenate([d[:, :1], d[:, :-1]], axis=1)
        return np.mod(e + np.floor_divide(dp + d + 2, 4), M), d
    L, H = f1m(I.astype(np.int64))
    LL, LH = f1m(L.T); HL, HH = f1m(H.T)
    return dict(LL=LL.T, LH=LH.T, HL=HL.T, HH=HH.T)


def ablation(I, BLK, raw, seed=0):
    rng = np.random.default_rng(seed)
    Bs = iwt2(I); Bq = dict(Bs)
    for nm in ['HL', 'LH', 'HH']:
        W = band_width(Bs[nm]); u = np.mod(Bs[nm], 1 << W)
        tv = expand_t(block_t(u, W, W - 2, BLK), u.shape, BLK)
        u = vacate(u, tv, W) + np.floor(rng.random(u.shape) * 2.0 ** tv).astype(np.int64) * (1 << (W - tv))
        Bq[nm] = u - (1 << W) * (u >= (1 << (W - 1)))
    X = iiwt2_raw(Bq)
    ovf = int(((X < 0) | (X > 255)).sum())
    Bm = miwt2(I)
    mod = sum(int(block_t(Bm[nm], 8, 6, BLK).sum()) * BLK * BLK for nm in ['HL', 'LH', 'HH'])
    return [ovf, 0, mod, int(raw)]


def tmap_leak(h, K):
    tb = np.concatenate([h['tm'][b].T.ravel() for b in h['det']]).astype(np.uint8)
    ks = np.frombuffer(ctr(K['tmap'] + h['nonce'], tb.size), np.uint8)
    return entropy8(tb), entropy8(tb ^ ks)


def display(E, M):
    s = min(M, int(math.isqrt(E.size)))
    return E[:s * s].reshape(s, s)


BSWEEP = (2, 4, 8, 16, 32)


def spec_cipher_bytes(C, H):
    """Serialised ciphertext of the specified construction: every band's W_b-bit
    fields in band order, packed MSB first, final partial byte dropped."""
    return packbits_cipher(C, H['W'], H['names'])


def spec_security(I, K, nonce, KH, BLK, trials, rng, keep=False):
    """Ciphertext statistics and key sensitivity of the specified (AES-256-CTR)
    construction, the one Propositions 1-3 cover. The marked ciphertext carries a
    payload filling the whole room. Key sensitivity: one bit of the master key K
    flipped; nonce, data-hiding key K_H and payload unchanged."""
    import serdhei as SD
    I64 = I.astype(np.int64)

    def marked(Kx, D=None):
        C, H0, t1 = SD.spec_owner(I64, Kx, nonce, BLK=BLK)
        kt = SD.owner_keys(Kx, nonce)['tmap']
        if D is None:
            T = SD._decrypt_tmap(H0, kt)
            room = int(sum(int(T[b].sum()) * BLK * BLK for b in T))
            D = np.random.default_rng(1000).integers(0, 2, room).astype(np.uint8)
        Cm, H1, t2 = SD.spec_hider(C, H0, t1, kt, KH, D)
        return C, H0, Cm, H1, t1, t2, D
    C, H0, Cm, H1, t1, t2, D = marked(K)
    E = spec_cipher_bytes(Cm, H1)
    out = dict(Hciph=entropy8(E), chi2=chi2stat(E))
    out['pchi'] = float(_chi2.sf(out['chi2'], 255)); out['cc'] = corr3(E)
    R = SD.spec_recover(Cm, H1, t1, K)
    out['exact'] = bool(R is not None and np.array_equal(np.asarray(R).astype(np.int64), I64))
    bad = bytearray(K); bad[0] ^= 1
    out['wrong_key_rejected'] = SD.spec_recover(Cm, H1, t1, bytes(bad)) is None
    if keep:
        out['img'] = dict(enc=display(spec_cipher_bytes(C, H0), I.shape[0]), mark=display(E, I.shape[0]),
                          ciph=display(E, I.shape[0]), rec=np.asarray(R).astype(np.uint8))
    if trials > 0:
        # key sensitivity is measured on the content owner's encrypted image, the output
        # K controls; the marked images (same K_H and payload) are recorded as well
        E0 = spec_cipher_bytes(C, H0)

        def wna(A, Bs):
            T = int(A.size); nn = [npcr_uaci(A, b)[0] for b in Bs]; uu = [npcr_uaci(A, b)[1] for b in Bs]
            npass, nstar = npcr_test(float(np.mean(nn)), T); upass, ulo, uhi = uaci_test(float(np.mean(uu)), T)
            return dict(npcr=float(np.mean(nn)), npcr_sd=float(np.std(nn, ddof=1)), uaci=float(np.mean(uu)),
                        uaci_sd=float(np.std(uu, ddof=1)), npcr_trials=nn, uaci_trials=uu,
                        npass_trials=int(sum(bool(npcr_test(x, T)[0]) for x in nn)),
                        upass_trials=int(sum(bool(uaci_test(x, T)[0]) for x in uu)),
                        T=T, nstar=float(nstar), ulo=float(ulo), uhi=float(uhi),
                        npass=bool(npass), upass=bool(upass), trials=len(nn))
        enc, mk = [], []
        for t in range(trials):
            k = bytearray(K); bp = int(rng.integers(0, 256)); k[bp // 8] ^= 1 << (bp % 8)
            Ck, Hk, Cmk, H1k = marked(bytes(k), D)[:4]
            enc.append(spec_cipher_bytes(Ck, Hk)); mk.append(spec_cipher_bytes(Cmk, H1k))
        out['sens'] = wna(E0, enc)
        out['sens_marked'] = wna(E, mk)
        # plaintext sensitivity: one pixel changed by +-1. Under a fresh nonce, as the scheme
        # requires, and, as a diagnostic of the forbidden nonce reuse, under the same nonce.
        fresh, same = [], []
        for t in range(trials):
            for _ in range(50):
                i, j = int(rng.integers(0, I.shape[0])), int(rng.integers(0, I.shape[1]))
                I2 = I64.copy(); I2[i, j] += 1 if I2[i, j] < 255 else -1
                C2, H2, _t = SD.spec_owner(I2, K, nonce, BLK=BLK)
                if H2['W'] == H0['W']:          # same widths, so the two arrays align field by field
                    break
            else:
                continue
            same.append(spec_cipher_bytes(C2, H2))
            C3, H3, _t = SD.spec_owner(I2, K, rng.bytes(12), BLK=BLK)
            fresh.append(spec_cipher_bytes(C3, H3))
        if len(fresh) >= 2:
            out['psens_fresh'] = wna(E0, fresh)
            out['psens_same'] = dict(npcr=float(np.mean([npcr_uaci(E0, b)[0] for b in same])),
                                     uaci=float(np.mean([npcr_uaci(E0, b)[1] for b in same])), trials=len(same))
    return out


def one_image(I, a1, a2, km=DEFAULT_KEY, nonce=DEFAULT_NONCE, BLK=BLOCK, trials=10,
              sweeps=True, seed=0, keep_images=False):
    """Everything one_image() in SE_RDHEI.m reports that the manuscript uses."""
    rng = np.random.default_rng(seed)
    MN = I.size
    r = dict(M=int(I.shape[0]), N=int(I.shape[1]), Hplain=entropy8(I))
    K = keys(km, a1, a2, nonce)
    t0 = time.perf_counter()
    C, h, pl, Cplain = encrypt_embed(I, km, a1, a2, nonce, None, BLK, K=K, rng=rng,
                                     return_plain_cipher=True)
    r['tenc'] = time.perf_counter() - t0
    t0 = time.perf_counter()
    Ir, pe = extract_decrypt(C, h, km, a1, a2, K=K)
    r['tdec'] = time.perf_counter() - t0
    r['exact'] = bool(np.array_equal(Ir, I) and np.array_equal(pe[:pl.size], pl))
    r['maxerr'] = int(np.abs(I.astype(int) - Ir.astype(int)).max())
    r['levels'] = h['levels']
    r['raw'], r['side'], r['net'] = h['raw'] / MN, h['side'] / MN, h['cap'] / MN
    r['cont'] = sum(h['Wd'][b] * C[b].size for b in h['names']) / MN
    r['eff'] = r['net'] - max(r['cont'] - 8, 0)
    r['Wd'] = h['Wd']
    E = packbits_cipher(C, h['Wd'], h['names'])
    r['Hciph'] = entropy8(E); r['chi2'] = chi2stat(E)
    r['pchi'] = float(_chi2.sf(r['chi2'], 255))
    r['cp'] = corr3(I); r['cc'] = corr3(E)
    r['tleak'] = list(tmap_leak(h, K))
    # one-bit wrong key: the recovered image must be unrelated to the cover
    bad = bytearray(km); bad[0] ^= 1
    Iw, _ = extract_decrypt(C, h, bytes(bad), a1, a2)
    r['wrong_match_pct'] = float(np.mean(Iw == I) * 100)
    if keep_images:
        r['img'] = dict(cover=I.astype(np.uint8),
                        enc=display(packbits_cipher(Cplain, h['Wd'], h['names']), I.shape[0]),
                        mark=display(E, I.shape[0]), rec=Ir, ciph=display(E, I.shape[0]))
    # ---- sensitivity: one-bit change of the key, of A1 and of A2
    if trials > 0:
        base = cipher_ref(I, km, a1, a2, nonce, BLK, 1000, K=K)
        for lab in ['key', 'A1', 'A2']:
            nn, uu, npt, upt = [], [], 0, 0
            T = int(base.size)            # number of compared bytes, not M*N
            for t in range(trials):
                k, x1, x2 = bytearray(km), a1.copy(), a2.copy()
                if lab == 'key':
                    bp = int(rng.integers(0, 256)); k[bp // 8] ^= 1 << (bp % 8)
                elif lab == 'A1':
                    j = int(rng.integers(0, x1.size)); x1[j] ^= 1
                else:
                    j = int(rng.integers(0, x2.size)); x2[j] ^= 1
                # same payload and same nonce as the reference: only the key or credential bit differs
                Ec = cipher_ref(I, bytes(k), x1, x2, nonce, BLK, 1000)
                n_, u_ = npcr_uaci(base, Ec); nn.append(n_); uu.append(u_)
                npt += bool(npcr_test(n_, T)[0]); upt += bool(uaci_test(u_, T)[0])
            r['npcr_' + lab] = float(np.mean(nn)); r['npcr_%s_sd' % lab] = float(np.std(nn, ddof=1))
            r['uaci_' + lab] = float(np.mean(uu)); r['uaci_%s_sd' % lab] = float(np.std(uu, ddof=1))
            r['npcr_%s_trials' % lab] = nn; r['uaci_%s_trials' % lab] = uu
            r['npcr_%s_tpass' % lab] = npt; r['uaci_%s_tpass' % lab] = upt
        # Wu-Noonan-Agaian critical values for the number of compared bytes T
        npass, nstar = npcr_test(r['npcr_key'], T)
        upass, ulo, uhi = uaci_test(r['uaci_key'], T)
        r['wu'] = dict(T=T, nstar=nstar, npass=bool(npass), ulo=ulo, uhi=uhi, upass=bool(upass),
                       trials=trials, npass_trials=r['npcr_key_tpass'], upass_trials=r['uaci_key_tpass'])
    # ---- specified construction: ciphertext statistics and key sensitivity
    r['spec'] = spec_security(I, km, nonce, bytes(range(32, 64)), BLK, trials, rng, keep=keep_images)
    if keep_images and 'img' in r['spec']:
        sp = r['spec'].pop('img')
        r['img'].update({'spec_' + k: v for k, v in sp.items()})
    # ---- block-size and level sweeps (key independent)
    if sweeps:
        r['Bnet'] = [capacity(I, b)['cap'] / MN for b in BSWEEP]
        r['Bside'] = [capacity(I, b)['side'] / MN for b in BSWEEP]
        # key-independent values at B = 4, the block size of the recorded MATLAB run,
        # so that the run can still be validated against it
        c4 = capacity(I, 4)
        lv4 = [capacity(I, 4, lv) for lv in (1, 2, 3, 4)]
        det4 = band_names(c4['L'])[1]
        tb4 = np.concatenate([c4['tm'][b].T.ravel() for b in det4]).astype(np.uint8)
        r['val4'] = dict(levels=c4['L'], raw=c4['raw'], side=c4['side'], cap=c4['cap'], cont=c4['cont'],
                         Leff=[(c['cap'] - max(c['cont'] - 8 * MN, 0)) / MN for c in lv4],
                         abl=ablation(I, 4, c4['raw'])[2:], tleak0=entropy8(tb4))
        r['bands'] = capacity(I, BLK)['bands']   # per band: width, coefficients, room
        r['Leff'], r['Lnet'] = [], []
        for lv in (1, 2, 3, 4):
            c = capacity(I, BLK, lv)
            r['Lnet'].append(c['cap'] / MN)
            r['Leff'].append(c['cap'] / MN - max(c['cont'] / MN - 8, 0))
        r['abl'] = ablation(I, BLK, h['raw'])
    return r


def credential_ablation_Hc(I, a1, a2, km=DEFAULT_KEY, nonce=DEFAULT_NONCE, BLK=BLOCK, seed=0):
    """Ciphertext entropy with the credentials on and with them removed."""
    out = []
    for use in (True, False):
        K = keys(km, a1, a2, nonce, use_credentials=use)
        C, h, _ = encrypt_embed(I, km, a1, a2, nonce, None, BLK, K=K,
                                rng=np.random.default_rng(seed))
        out.append(entropy8(packbits_cipher(C, h['Wd'], h['names'])))
    return out


def to_gray(arr):
    """load_cover(): uint8(0.299R + 0.587G + 0.114B), crop to a multiple of 16."""
    a = np.asarray(arr)
    if a.ndim == 3:
        a = a[:, :, :3].astype(float)
        a = np.floor(0.299 * a[:, :, 0] + 0.587 * a[:, :, 1] + 0.114 * a[:, :, 2] + 0.5)
    a = np.clip(a, 0, 255).astype(np.uint8)
    return a[:16 * (a.shape[0] // 16), :16 * (a.shape[1] // 16)]


def read_cover(path):
    from PIL import Image
    im = Image.open(path)
    if im.mode in ('RGB', 'RGBA', 'P', 'CMYK', 'YCbCr', 'LA'):
        im = im.convert('RGB')
    elif im.mode not in ('L', '1'):
        a = np.array(im)
        if a.dtype != np.uint8:                       # 16-bit greyscale: MATLAB uint8() saturates
            a = np.clip(a, 0, 255)
        return to_gray(a)
    return to_gray(np.array(im))


def classic_overflow(I, a1, a2, km=DEFAULT_KEY, nonce=DEFAULT_NONCE, BLK=BLOCK, seed=0):
    """Classic ordering through the full pipeline: the same L-level transform,
    the same reservation and the same pseudorandom payload as the proposed
    scheme; the receiver decrypts and synthesises *before* restoring the
    vacated bits.  Returns the number of pixels outside [0, 255] and, as a
    check, the count under deferred synthesis (always 0)."""
    K = keys(km, a1, a2, nonce)
    C, h, _ = encrypt_embed(I, km, a1, a2, nonce, None, BLK, K=K, rng=np.random.default_rng(seed))
    Bc, Bd = {}, {}
    for nm in h['names']:
        W, n, shp = h['Wd'][nm], C[nm].size, h['shapes'][nm]
        p = permutation(K, nm, n)
        e = np.empty(n, np.int64); e[p] = C[nm]
        u = (e ^ keystream(K, nm, n, W)).reshape(shp)            # decrypted, payload still in place
        Bc[nm] = u - (1 << W) * (u >= (1 << (W - 1)))
        if nm in h['det']:
            u = restore(u, expand_t(h['tm'][nm], shp, BLK), W)
        Bd[nm] = u - (1 << W) * (u >= (1 << (W - 1)))
    def raw_synth(B):
        cur = B['LL']
        for lv in range(h['levels'], 0, -1):
            cur = iiwt2_raw(dict(LL=cur, HL=B['HL%d' % lv], LH=B['LH%d' % lv], HH=B['HH%d' % lv]))
        return cur
    Xc, Xd = raw_synth(Bc), raw_synth(Bd)
    return int(((Xc < 0) | (Xc > 255)).sum()), int(((Xd < 0) | (Xd > 255)).sum()), bool(np.array_equal(Xd, I))


def modular_room(I, BLK=BLOCK, levels=0):
    """Room under modular Z_256 lifting at the same depth as the scheme: every
    detail band of every level, 8-bit fields, block minimum, t capped at 6."""
    L = min(levels, max_levels(I.shape)) if levels > 0 else max_levels(I.shape)
    cur, room = I.astype(np.int64), 0
    for _ in range(L):
        Bm = miwt2(cur)
        room += sum(int(block_t(Bm[nm], 8, 6, BLK).sum()) * BLK * BLK for nm in ['HL', 'LH', 'HH'])
        cur = Bm['LL']
    return room


def ablation_full(I, a1, a2, BLK=BLOCK, seeds=(0, 1, 2)):
    """Ablation at the scheme's own depth L, own reservation and real payload."""
    cl = [classic_overflow(I, a1, a2, BLK=BLK, seed=s) for s in seeds]
    assert all(c[1] == 0 and c[2] for c in cl), 'deferred synthesis must give zero and exact recovery'
    return dict(levels=max_levels(I.shape), classic=[c[0] for c in cl], deferred=0,
                modular_room=modular_room(I, BLK), proposed_room=capacity(I, BLK)['raw'])

In [ ]:
%%writefile /content/se_rdhei/run_all.py
#!/usr/bin/env python3
"""SE-RDHEI: reproduce every measured result of the CAEE manuscript.

    python run_all.py --data DATA_DIR --out OUT_DIR [--n 100] [--trials 10]
                      [--a1 A1.wav --a2 A2.wav] [--stage all]

DATA_DIR holds one folder per dataset (USC-SIPI, BOSSbase, BOWS-2, UCID; any
other folder is processed as an extra dataset). Images may sit in sub-folders.
Per-image results are cached in OUT_DIR/cache, so an interrupted run resumes
where it stopped. Outputs:

  data.json      capacity, statistics, NPCR/UACI, sweeps, ablation, runtime,
                 per-dataset means              (Tables 3, 4, 6, 7; Figs 4-11)
  extra.json     block-minimum efficiency, leakage-reduced descriptor,
                 20-key statistics                         (Table 8, Sec. 7)
  spec.json      specified construction: round trip, tampering, AES statistics
  matched.json   matched comparison with the reimplemented BPC method (Table 5)
  matched_db.json  the same comparison on the database samples
  figures/       every data figure of the manuscript (PDF + PNG)
  REPORT.md      all tables in readable form, plus validation against the
                 recorded MATLAB run
  results_SE-RDHEI.zip   everything above, to send back
"""
import argparse, json, os, re, shutil, subprocess, sys, time, zipfile, platform
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import se_full as S

IMG_EXT = ('.png', '.tif', '.tiff', '.bmp', '.pgm', '.jpg', '.jpeg', '.ppm', '.gif')
ORDER = ['Airplane', 'Baboon', 'Jetplane', 'Lena', 'Man', 'Tiffany', 'barbara',
         'boat', 'goldhill', 'lake', 'livingroom', 'peppers']
ALIASES = {
    'Airplane': ['airplane', '7.1.02'],
    'Baboon': ['baboon', 'mandrill', 'mandril', '4.2.03'],
    'Jetplane': ['jetplane', 'jet', 'f16', 'f-16', '4.2.05'],
    'Lena': ['lena', 'lenna', '4.2.04'],
    'Man': ['man', '5.3.01'],
    'Tiffany': ['tiffany', '4.2.02'],
    'barbara': ['barbara'],
    'boat': ['boat', 'boats', 'boat.512'],
    'goldhill': ['goldhill', 'gold_hill', 'gold-hill'],
    'lake': ['lake', '4.2.06'],
    'livingroom': ['livingroom', 'living_room', 'living-room'],
    'peppers': ['peppers', 'pepper', '4.2.07'],
}
# BOWS-2 is tested before BOSSbase so that a folder named BOSS-2 (how the MATLAB run
# stored its BOWS-2 copy) is not taken for BOSSbase
DS_ALIASES = {'USC-SIPI': ['usc-sipi', 'usc_sipi', 'sipi', 'usc', 'standard'],
              'BOWS-2': ['bows-2', 'bows2', 'bows_2', 'bows', 'boss-2', 'boss2', 'boss_2', 'bose-2', 'bose2'],
              'BOSSbase': ['bossbase', 'boss'],
              'UCID': ['ucid']}
MONTAGE = ['Airplane', 'barbara', 'livingroom']


def canon_ds(folder):
    f = folder.lower()
    for k, al in DS_ALIASES.items():
        if any(f.startswith(a) for a in al):
            return k
    return folder


def canon_cover(stem, rename):
    if stem in rename:
        return rename[stem]
    s = stem.lower()
    tok = [t for t in re.split(r'[_\- ]+', s) if t]
    cands = [s] + ([tok[0], tok[0].split('.')[0]] if tok else [])
    for c in cands:
        for k, al in ALIASES.items():
            if c in al:
                return k
    return stem


def discover(data, n_per_db, seed, rename, dsmap=None):
    """Group images into datasets.  dsmap maps a folder name found anywhere in a
    file's path (case-insensitive) to a dataset name, e.g. {"BOSS-2": "BOWS-2"}."""
    dsmap = {k.lower(): v for k, v in (dsmap or {}).items()}
    sets, origin = {}, {}
    for d in sorted(os.listdir(data)):
        p = os.path.join(data, d)
        if not os.path.isdir(p) or d.startswith(('.', '__')):
            continue
        for root, _, fs in os.walk(p, followlinks=True):
            for f in fs:
                if not f.lower().endswith(IMG_EXT) or f.startswith('._'):
                    continue
                fp = os.path.join(root, f)
                parts = os.path.relpath(fp, data).replace(os.sep, '/').split('/')[:-1]
                ds = next((dsmap[c.lower()] for c in reversed(parts) if c.lower() in dsmap), None)
                if ds is not None and ds.strip().lower() in ('', 'skip', 'exclude'):
                    continue
                ds = ds or canon_ds(d)
                sets.setdefault(ds, []).append(fp)
                origin.setdefault(ds, {}).setdefault('/'.join(parts), 0)
                origin[ds]['/'.join(parts)] += 1
    # a folder of standard covers under any name (e.g. "Image") is the USC-SIPI set
    for ds in list(sets):
        if ds in DS_ALIASES:
            continue
        hits = [f for f in sets[ds] if canon_cover(os.path.splitext(os.path.basename(f))[0], rename) in ORDER]
        if len(hits) >= 3:
            print('  folder "%s" holds standard covers: treated as USC-SIPI' % ds)
            sets.setdefault('USC-SIPI', []).extend(sets.pop(ds))
            o = origin.pop(ds)
            for k, v in o.items():
                origin.setdefault('USC-SIPI', {})[k] = v
    for ds, o in origin.items():
        if len(o) > 1 and ds != 'USC-SIPI':
            print('  ! %s mixes images from %d folders: %s' % (ds, len(o), ', '.join('%s (%d)' % kv for kv in sorted(o.items()))))
            print('    if they are different datasets, name each one in DATASET_MAP, e.g. {"<folder name>": "<dataset name>"}')
    out = {}
    for ds, files in sets.items():
        files = sorted(set(files))
        if ds == 'USC-SIPI':
            m = {}
            for f in files:
                nm = canon_cover(os.path.splitext(os.path.basename(f))[0], rename)
                if nm in m:
                    print('  ! duplicate cover %s: keeping %s, ignoring %s' % (nm, m[nm], f))
                    continue
                m[nm] = f
            out[ds] = dict(sorted(m.items(), key=lambda kv: (ORDER.index(kv[0]) if kv[0] in ORDER else 99, kv[0])))
        else:
            if n_per_db and len(files) > n_per_db:
                idx = np.random.default_rng(seed).choice(len(files), n_per_db, replace=False)
                files = [files[i] for i in sorted(idx)]
            out[ds] = {os.path.relpath(f, data).replace(os.sep, '/'): f for f in files}
    return out


def jdump(o, p):
    def conv(x):
        if isinstance(x, (np.integer,)): return int(x)
        if isinstance(x, (np.floating,)): return float(x)
        if isinstance(x, np.bool_): return bool(x)
        if isinstance(x, np.ndarray): return x.tolist()
        raise TypeError(type(x))
    with open(p, 'w') as f:
        json.dump(o, f, indent=1, default=conv)


def safe(s):
    return re.sub(r'[^A-Za-z0-9._-]', '_', s)


# ------------------------------------------------------------------ stages
def stage_main(cfg, sets, a1, a2):
    cache = os.path.join(cfg.out, 'cache'); os.makedirs(cache, exist_ok=True)
    usc = sets.get('USC-SIPI', {})
    mont = [c for c in MONTAGE if c in usc] + [c for c in usc if c not in MONTAGE]
    mont = mont[:3]
    for ds, items in sets.items():
        full = ds == 'USC-SIPI'
        d = os.path.join(cache, safe(ds)); os.makedirs(d, exist_ok=True)
        t_ds = time.time()
        for i, (nm, path) in enumerate(items.items()):
            cj = os.path.join(d, safe(nm) + '.json')
            if os.path.exists(cj):
                continue
            t0 = time.time()
            try:
                I = S.read_cover(path)
            except Exception as e:
                print('  skip unreadable %s (%s)' % (path, e)); continue
            r = S.one_image(I, a1, a2, BLK=S.BLOCK, trials=cfg.trials if full else 0,
                            sweeps=full, seed=i, keep_images=(full and nm in mont))
            if full:
                r['cred_Hc'] = S.credential_ablation_Hc(I, a1, a2, seed=i)
            if 'img' in r:
                np.savez_compressed(os.path.join(d, safe(nm) + '_img.npz'), **r.pop('img'))
            r['file'] = path
            jdump(r, cj)
            print('  [%s %d/%d] %-28s %4dx%-4d net %.4f eff %.4f exact %s  %.1fs'
                  % (ds, i + 1, len(items), nm[-28:], r['M'], r['N'], r['net'], r['eff'],
                     'YES' if r['exact'] else 'NO', time.time() - t0), flush=True)
        print('  %s done (%.0fs)' % (ds, time.time() - t_ds), flush=True)


def load_cache(cfg, sets):
    R = {}
    for ds, items in sets.items():
        d = os.path.join(cfg.out, 'cache', safe(ds))
        R[ds] = {}
        for nm in items:
            p = os.path.join(d, safe(nm) + '.json')
            if os.path.exists(p):
                R[ds][nm] = json.load(open(p))
    return R


def r4(x): return round(float(x), 4)


def assemble(cfg, sets, R, audio_src):
    D = dict(cap={}, stats={}, stats_base={}, val4={}, npcr_base={}, npcr_marked={}, psens={}, bands={}, blk={}, lvl={}, abl={}, rt={}, npcr={}, ds={}, size={},
             npcr_cred={}, tleak={}, wrong_key={}, cred_ablation={}, files={}, meta={})
    usc = R.get('USC-SIPI', {})
    for nm, r in usc.items():
        D['cap'][nm] = [r['levels'], r4(r['raw']), r4(r['side']), r4(r['net']),
                        round(r['cont'], 3), r4(r['eff'])]
        sp = r['spec']          # ciphertext statistics: specified (AES-256-CTR) construction
        D['stats'][nm] = dict(Hp=r4(r['Hplain']), Hc=r4(sp['Hciph']), chi=round(sp['chi2'], 1),
                              p=round(sp['pchi'], 3), plain=[r4(x) for x in r['cp']],
                              ciph=[r4(x) for x in sp['cc']])
        D['stats_base'][nm] = dict(Hc=r4(r['Hciph']), chi=round(r['chi2'], 1), p=round(r['pchi'], 3),
                                   ciph=[r4(x) for x in r['cc']])
        if 'val4' in r:
            v = r['val4']; MN = r['M'] * r['N']
            D['val4'][nm] = dict(cap=[v['levels'], r4(v['raw'] / MN), r4(v['side'] / MN), r4(v['cap'] / MN),
                                      round(v['cont'] / MN, 3), r4(v['cap'] / MN - max(v['cont'] / MN - 8, 0))],
                                 lvl=[r4(x) for x in v['Leff']], abl=v['abl'], tleak0=r4(v['tleak0']))
        D['blk'][nm] = [r4(x) for x in r['Bnet']]
        D['lvl'][nm] = [r4(x) for x in r['Leff']]
        if 'bands' in r:
            D['bands'][nm] = r['bands']
        D['abl'][nm] = r['abl']
        D['rt'][nm] = [r['M'] * r['N'], round(r['tenc'], 3), round(r['tdec'], 3)]
        D['size'][nm] = [r['M'], r['N']]
        if 'sens' in r['spec']:
            q = r['spec']['sens']          # key sensitivity of the specified construction
            D['npcr'][nm] = dict(n=r4(q['npcr']), nstar=r4(q['nstar']), npass='YES' if q['npass'] else 'NO',
                                 u=r4(q['uaci']), ulo=r4(q['ulo']), uhi=r4(q['uhi']), upass='YES' if q['upass'] else 'NO',
                                 nsd=r4(q['npcr_sd']), usd=r4(q['uaci_sd']),
                                 nsd1=r4(q['npcr_sd']), usd1=r4(q['uaci_sd']),
                                 nmin=r4(min(q['npcr_trials'])), nmax=r4(max(q['npcr_trials'])),
                                 umin=r4(min(q['uaci_trials'])), umax=r4(max(q['uaci_trials'])),
                                 T=q['T'], trials=q['trials'], npass_trials=q['npass_trials'], upass_trials=q['upass_trials'])
            if 'psens_fresh' in r['spec']:
                f_, s_ = r['spec']['psens_fresh'], r['spec']['psens_same']
                D['psens'][nm] = dict(n=r4(f_['npcr']), u=r4(f_['uaci']), nsd1=r4(f_['npcr_sd']), usd1=r4(f_['uaci_sd']),
                                      npass_trials=f_['npass_trials'], upass_trials=f_['upass_trials'], trials=f_['trials'],
                                      n_same=round(s_['npcr'], 5), u_same=round(s_['uaci'], 6))
            m = r['spec']['sens_marked']   # marked images, same K_H and payload, one bit of K flipped
            D['npcr_marked'][nm] = dict(n=r4(m['npcr']), nsd1=r4(m['npcr_sd']), u=r4(m['uaci']), usd1=r4(m['uaci_sd']),
                                        npass_trials=m['npass_trials'], upass_trials=m['upass_trials'], T=m['T'],
                                        npass='YES' if m['npass'] else 'NO', upass='YES' if m['upass'] else 'NO')
        if 'wu' in r:
            w = r['wu']
            D['npcr_base'][nm] = dict(n=r4(r['npcr_key']), nstar=r4(w['nstar']),
                                 npass='YES' if w['npass'] else 'NO', u=r4(r['uaci_key']),
                                 ulo=r4(w['ulo']), uhi=r4(w['uhi']), upass='YES' if w['upass'] else 'NO',
                                 nsd=r4(r['npcr_key_sd']), usd=r4(r['uaci_key_sd']),
                                 nsd1=r4(r['npcr_key_sd']), usd1=r4(r['uaci_key_sd']),   # sample SD (ddof=1)
                                 nmin=r4(min(r['npcr_key_trials'])), nmax=r4(max(r['npcr_key_trials'])),
                                 umin=r4(min(r['uaci_key_trials'])), umax=r4(max(r['uaci_key_trials'])),
                                 T=w.get('T'), trials=w.get('trials'),
                                 npass_trials=w.get('npass_trials'), upass_trials=w.get('upass_trials'))
            D['npcr_cred'][nm] = {q: dict(n=r4(r['npcr_' + q]), u=r4(r['uaci_' + q]),
                                          nsd=r4(r['npcr_%s_sd' % q]), usd=r4(r['uaci_%s_sd' % q]))
                                  for q in ('A1', 'A2')}
        D['tleak'][nm] = [r4(x) for x in r['tleak']]
        D['wrong_key'][nm] = r4(r['wrong_match_pct'])
    if usc:
        cw = [r['cred_Hc'][0] for r in usc.values()]
        cn = [r['cred_Hc'][1] for r in usc.values()]
        D['cred_ablation'] = dict(with_credentials=round(float(np.mean(cw)), 5),
                                  without_credentials=round(float(np.mean(cn)), 5), n=len(cw))
    for ds, items in R.items():
        if not items:
            continue
        net = np.array([r['net'] for r in items.values()])
        eff = np.array([r['eff'] for r in items.values()])
        D['ds'][ds] = dict(N=len(items), net=r4(net.mean()), nsd=r4(net.std()),
                           eff=r4(eff.mean()), esd=r4(eff.std()),
                           H=r4(np.mean([r['spec']['Hciph'] for r in items.values()])),
                           rho=r4(np.mean([max(abs(x) for x in r['spec']['cc']) for r in items.values()])),
                           chi_rej=int(sum(r['spec']['pchi'] < 0.05 for r in items.values())),
                           Hmin=r4(min(r['spec']['Hciph'] for r in items.values())),
                           Hmax=r4(max(r['spec']['Hciph'] for r in items.values())),
                           rho_max=r4(max(max(abs(x) for x in r['spec']['cc']) for r in items.values())),
                           exact=int(sum(r['exact'] and r['spec']['exact'] for r in items.values())),
                           net_min=r4(net.min()), net_max=r4(net.max()),
                           sizes=sorted({'%dx%d' % (r['M'], r['N']) for r in items.values()}))
        D['files'][ds] = {nm: r['file'] for nm, r in items.items()}
    D['meta'] = dict(date=time.strftime('%Y-%m-%d %H:%M:%S'), python=platform.python_version(),
                     machine=platform.processor() or platform.machine(), numpy=np.__version__,
                     key='bytes 0..31', nonce='0123456789ab', block=S.BLOCK, trials=cfg.trials,
                     n_per_db=cfg.n, sample_seed=cfg.seed, audio=audio_src,
                     note='capacity values are key independent; statistics depend on key, nonce and A1/A2')
    return D


def stage_ablation(cfg, sets, a1, a2):
    """Ablation at the scheme's own depth, reservation and payload (Table 6, Fig. 8)."""
    p = os.path.join(cfg.out, 'ablation.json')
    out = json.load(open(p)) if os.path.exists(p) else {}
    for nm, path in sets.get('USC-SIPI', {}).items():
        if nm in out:
            continue
        t0 = time.time()
        out[nm] = S.ablation_full(S.read_cover(path), a1, a2, BLK=S.BLOCK)
        jdump(out, p)
        c = out[nm]['classic']
        print('  [ablation] %-11s L=%d classic %d..%d px out of range, deferred 0, modular room %d, proposed room %d  %.1fs'
              % (nm, out[nm]['levels'], min(c), max(c), out[nm]['modular_room'], out[nm]['proposed_room'], time.time() - t0), flush=True)


def stage_spec_all(cfg, sets):
    """Specified construction (AES-256-CTR, block permutation, independent K_H)
    end to end on every image: payload and cover exact, room equal to the base."""
    import serdhei as SD
    p = os.path.join(cfg.out, 'spec_all.json')
    out = json.load(open(p)) if os.path.exists(p) else {}
    rng = np.random.default_rng(cfg.seed)
    for ds, items in sets.items():
        rows = out.setdefault(ds, {})
        for i, (nm, path) in enumerate(items.items()):
            if nm in rows:
                continue
            try:
                I = S.read_cover(path).astype(np.int64)
            except Exception as e:
                print('  skip unreadable %s (%s)' % (path, e)); continue
            t0 = time.time()
            K, KH, N = rng.bytes(32), rng.bytes(32), rng.bytes(12)
            C, H0, t1 = SD.spec_owner(I, K, N, BLK=S.BLOCK)
            kt = SD.owner_keys(K, N)['tmap']
            T = SD._decrypt_tmap(H0, kt)
            room = int(sum(int(T[b].sum()) * H0['B'] ** 2 for b in T))
            D = rng.integers(0, 2, room).astype(np.uint8)          # fill the room completely
            Cm, H1, t2 = SD.spec_hider(C, H0, t1, kt, KH, D)
            got = SD.spec_extract(Cm, H1, t1, t2, kt, KH)
            R = SD.spec_recover(Cm, H1, t1, K)
            cp = S.capacity(I.astype(np.uint8), S.BLOCK)
            hdr_bits = 8 * (len(SD._ser_header(H1)) + 64)                 # header plus both tags
            rows[nm] = dict(room=room, base_room=cp['raw'], Rt_bits=int(cp['side']),
                            tmap_stored_bits=8 * len(H0['Tcode']), header_and_tags_bits=hdr_bits,
                            meta_bits=hdr_bits - int(cp['side']),
                            payload_exact=bool(got is not None and np.array_equal(got, D)),
                            cover_exact=bool(R is not None and np.array_equal(np.asarray(R).astype(np.int64), I)),
                            seconds=round(time.time() - t0, 2))
            if (i + 1) % 25 == 0 or i + 1 == len(items):
                jdump(out, p)
                print('  [spec] %s %d/%d' % (ds, i + 1, len(items)), flush=True)
    jdump(out, p)
    for ds, rows in out.items():
        n = len(rows)
        print('  %-9s %d images: payload exact %d, cover exact %d, room = base room %d'
              % (ds, n, sum(r['payload_exact'] for r in rows.values()), sum(r['cover_exact'] for r in rows.values()),
                 sum(r['room'] == r['base_room'] for r in rows.values())))


def stage_extra(cfg, sets):
    """Run the manuscript's own experiment scripts on the uploaded covers."""
    w = os.path.join(cfg.out, 'work'); os.makedirs(w, exist_ok=True)
    for f in ['serdhei.py', 'bpc.py', 'extra_experiments.py', 'spec_experiments.py',
              'matched_comparison.py', 'mkfigs.py']:
        shutil.copy(os.path.join(HERE, f), w)
    shutil.copy(os.path.join(cfg.out, 'data.json'), w)
    jdump({nm: os.path.abspath(p) for nm, p in sets.get('USC-SIPI', {}).items()},
          os.path.join(w, 'verified_covers.json'))
    for sc in ['extra_experiments.py', 'spec_experiments.py', 'matched_comparison.py']:
        out = sc.replace('_experiments.py', '.json').replace('matched_comparison.py', 'matched.json')
        if os.path.exists(os.path.join(cfg.out, out)) and not cfg.force:
            print('  %s exists, skipping %s' % (out, sc)); continue
        print('  running %s ...' % sc, flush=True); t0 = time.time()
        subprocess.run([sys.executable, sc], cwd=w, check=True)
        shutil.copy(os.path.join(w, out), cfg.out)
        print('  %s written (%.0fs)' % (out, time.time() - t0), flush=True)


def stage_matched_db(cfg, sets, R):
    import math
    import bpc
    p = os.path.join(cfg.out, 'matched_db.json')
    out = json.load(open(p)) if os.path.exists(p) else {'rows': {}, 'summary': {}}
    for ds, items in sets.items():
        if ds == 'USC-SIPI':
            continue
        rows = out['rows'].setdefault(ds, {})
        for i, (nm, path) in enumerate(items.items()):
            if nm in rows or nm not in R.get(ds, {}):
                continue
            I = S.read_cover(path).astype(np.int64)
            room, bs, lf, lens, ov = bpc.compress(I, border='reference')
            aux = 18 + math.ceil(math.log2(I.size)) * ov + 4 + 3 + 3
            rows[nm] = dict(bpc_net=(room - aux) / I.size, overflow=int(ov),
                            prop_net=R[ds][nm]['net'], prop_eff=R[ds][nm]['eff'])
            if (i + 1) % 25 == 0:
                print('  matched %s %d/%d' % (ds, i + 1, len(items)), flush=True); jdump(out, p)
        if rows:
            b = np.array([r['bpc_net'] for r in rows.values()])
            pn = np.array([r['prop_net'] for r in rows.values()])
            pe = np.array([r['prop_eff'] for r in rows.values()])
            out['summary'][ds] = dict(N=len(rows), bpc_net=r4(b.mean()), bpc_sd=r4(b.std()),
                                      prop_net=r4(pn.mean()), prop_eff=r4(pe.mean()),
                                      prop_net_wins=int((pn > b).sum()), prop_eff_wins=int((pe > b).sum()))
    jdump(out, p)
    for ds, s in out['summary'].items():
        print('  %-9s N=%d  BPC %.4f  proposed net %.4f eff %.4f  (net wins %d, eff wins %d)'
              % (ds, s['N'], s['bpc_net'], s['prop_net'], s['prop_eff'], s['prop_net_wins'], s['prop_eff_wins']))


def stage_xiao(cfg, sets, R):
    """Identical-image comparison with the reimplemented ABCS method of Xiao et al.
    (2024), t = 8, T searched per image as in the paper. Covers: the twelve
    standard covers. Databases: exactly the images of matched_db.json, so the
    BPC, ABCS and proposed columns refer to the same files."""
    import abcs
    p = os.path.join(cfg.out, 'xiao.json')
    out = json.load(open(p)) if os.path.exists(p) else {'rows': {}, 'summary': {}}
    md = json.load(open(os.path.join(cfg.out, 'matched_db.json'))) if os.path.exists(
        os.path.join(cfg.out, 'matched_db.json')) else {'rows': {}}
    for ds, items in sets.items():
        names = list(items) if ds == 'USC-SIPI' else [n for n in items if n in md['rows'].get(ds, {})]
        rows = out['rows'].setdefault(ds, {})
        for i, nm in enumerate(names):
            if nm in rows or nm not in R.get(ds, {}):
                continue
            I = S.read_cover(items[nm])
            r = abcs.rate(I, 8)
            rows[nm] = dict(xiao_net=float(r['er']), T=int(r['T']), coding2_blocks=int(r['coding2_blocks']),
                            prop_net=R[ds][nm]['net'], prop_eff=R[ds][nm]['eff'])
            if (i + 1) % 25 == 0:
                print('  xiao %s %d/%d' % (ds, i + 1, len(names)), flush=True); jdump(out, p)
        if rows:
            x = np.array([r['xiao_net'] for r in rows.values()])
            pn = np.array([r['prop_net'] for r in rows.values()])
            pe = np.array([r['prop_eff'] for r in rows.values()])
            out['summary'][ds] = dict(N=len(rows), xiao_net=r4(x.mean()), xiao_sd=r4(x.std()),
                                      prop_net=r4(pn.mean()), prop_eff=r4(pe.mean()),
                                      prop_net_wins=int((pn > x).sum()), prop_eff_wins=int((pe > x).sum()))
    jdump(out, p)
    for ds, s_ in out['summary'].items():
        print('  %-9s N=%d  ABCS %.4f  proposed net %.4f eff %.4f  (net wins %d, eff wins %d)'
              % (ds, s_['N'], s_['xiao_net'], s_['prop_net'], s_['prop_eff'], s_['prop_net_wins'], s_['prop_eff_wins']))


def stage_figs(cfg, sets):
    w = os.path.join(cfg.out, 'work'); os.makedirs(w, exist_ok=True)
    shutil.copy(os.path.join(HERE, 'mkfigs.py'), w)
    shutil.copy(os.path.join(cfg.out, 'data.json'), w)
    d = os.path.join(cfg.out, 'cache', 'USC-SIPI')
    sec, mon = {}, {}
    for f in sorted(os.listdir(d)) if os.path.isdir(d) else []:
        if f.endswith('_img.npz'):
            nm = f[:-8]; Z = np.load(os.path.join(d, f))
            g = lambda k: Z['spec_' + k] if 'spec_' + k in Z.files else Z[k]   # specified construction
            sec[nm + '_cover'] = Z['cover']; sec[nm + '_ciph'] = g('ciph')
            mon['%s_cover' % nm] = Z['cover']
            for k in ('enc', 'mark', 'rec'):
                mon['%s_%s' % (nm, k)] = g(k)
    if sec:
        np.savez_compressed(os.path.join(w, 'secdata.npz'), **sec)
        np.savez_compressed(os.path.join(w, 'montage.npz'), **mon)
        shutil.copy(os.path.join(w, 'secdata.npz'), cfg.out)
        shutil.copy(os.path.join(w, 'montage.npz'), cfg.out)
    subprocess.run([sys.executable, 'mkfigs.py'], cwd=w, check=True)
    fd = os.path.join(cfg.out, 'figures'); os.makedirs(fd, exist_ok=True)
    for f in os.listdir(w):
        if f.startswith('fig_') and f.endswith('.pdf') or f.startswith('p_fig_') and f.endswith('.png'):
            shutil.copy(os.path.join(w, f), fd)
    print('  figures ->', fd)


# ------------------------------------------------------------------ report
def table(head, rows):
    s = '| ' + ' | '.join(head) + ' |\n|' + '---|' * len(head) + '\n'
    for r in rows:
        s += '| ' + ' | '.join(str(x) for x in r) + ' |\n'
    return s + '\n'


def f4(x): return '%.4f' % x


def report(cfg, sets, D):
    rec = json.load(open(os.path.join(HERE, 'recorded_matlab.json')))
    L = ['# SE-RDHEI results\n', 'Generated %s on %s, Python %s.  Key bytes 0..31, nonce "0123456789ab", '
         'B = %d, %d sensitivity trials.  Credentials A1/A2: %s.\n\n'
         % (D['meta']['date'], D['meta']['machine'], D['meta']['python'], S.BLOCK, cfg.trials, D['meta']['audio'])]
    covers = list(D['cap'])
    # validation
    if covers:
        L.append('## 0. Validation against the recorded MATLAB run (key-independent quantities, at its block size B = 4)\n')
        rows, allok = [], True
        for nm in covers:
            if nm not in rec['cap']:
                rows.append([nm, 'not in recorded run', '', '', '', '']); continue
            v4 = D['val4'].get(nm)
            if v4 is None:
                rows.append([nm, 'no B = 4 values', '', '', '', '']); allok = False; continue
            ok = [v4['cap'] == rec['cap'][nm], D['blk'][nm][1:] == rec['blk'][nm],
                  v4['lvl'] == rec['lvl'][nm], v4['abl'] == [rec['modroom'][nm], rec['ourroom'][nm]],
                  abs(v4['tleak0'] - rec['tleak'][nm][0]) < 1e-4]
            allok &= all(ok)
            rows.append([nm] + ['match' if o else '**DIFFERENT**' for o in ok])
        L.append(table(['cover', 'capacity row', 'B sweep', 'L sweep', 'room (ablation)', 't-map entropy'], rows))
        L.append('All key-independent values %s the recorded run on the same cover files. '
                 'A difference means the uploaded file is not the file used in the MATLAB run.\n\n'
                 % ('reproduce' if allok else 'do NOT all reproduce'))
        # capacity
        L.append('## 1. Capacity and reversibility (Table 3, Fig. 5)\n')
        rows = [[nm, D['cap'][nm][0], 'YES' if R_exact(cfg, nm) else 'NO'] + [f4(x) for x in D['cap'][nm][1:4]]
                + ['%.3f' % D['cap'][nm][4], f4(D['cap'][nm][5])] for nm in covers]
        m = np.array([D['cap'][nm][1:] for nm in covers]).mean(0)
        rows.append(['MEAN', '', ''] + [f4(x) for x in m[:3]] + ['%.3f' % m[3], f4(m[4])])
        L.append(table(['image', 'L', 'exact', 'raw', 'side', 'net', 'container', 'eff'], rows))
    L.append('## 2. Per-dataset rates (Table 4, Figs. 5-6)\n')
    L.append(table(['dataset', 'N', 'sizes', 'exact', 'net mean', 'net sd', 'net min', 'net max', 'eff mean', 'eff sd', 'H cipher', 'mean max|rho|', 'chi2 rejections'],
                   [[ds, v['N'], ' '.join(v['sizes']), '%d/%d' % (v['exact'], v['N']), f4(v['net']), f4(v['nsd']),
                     f4(v['net_min']), f4(v['net_max']), f4(v['eff']), f4(v['esd']), f4(v['H']), f4(v['rho']),
                     '%s/%d' % (v.get('chi_rej', '-'), v['N'])]
                    for ds, v in D['ds'].items()]))
    if covers:
        L.append('## 3. Ciphertext statistics (Table 7, Figs. 9-10)\n')
        L.append(table(['image', 'H plain', 'H cipher', 'chi2', 'p', 'plain H,V,D', 'cipher H,V,D'],
                       [[nm, f4(s['Hp']), f4(s['Hc']), s['chi'], s['p'], ', '.join('%+.4f' % x for x in s['plain']),
                         ', '.join('%+.4f' % x for x in s['ciph'])] for nm, s in D['stats'].items()]))
        if D['npcr']:
            L.append('## 4. Key and credential sensitivity, Wu-Noonan-Agaian tests at alpha = 0.05 (Fig. 11)\n')
            L.append('Key: one bit of the master key K of the specified construction flipped, encrypted images compared. '
                     'A1/A2: credentials of the base construction.\n')
            L.append(table(['image', 'NPCR key', 'sd', 'N*', 'pass', 'UACI key', 'sd', 'U*-', 'U*+', 'pass', 'NPCR/UACI A1', 'NPCR/UACI A2'],
                           [[nm, f4(v['n']), f4(v['nsd']), f4(v['nstar']), v['npass'], f4(v['u']), f4(v['usd']), f4(v['ulo']), f4(v['uhi']), v['upass'],
                             '%.4f / %.4f' % (D['npcr_cred'][nm]['A1']['n'], D['npcr_cred'][nm]['A1']['u']),
                             '%.4f / %.4f' % (D['npcr_cred'][nm]['A2']['n'], D['npcr_cred'][nm]['A2']['u'])]
                            for nm, v in D['npcr'].items()]))
            n = [v['n'] for v in D['npcr'].values()]; u = [v['u'] for v in D['npcr'].values()]
            L.append('NPCR mean %.4f (range %.4f to %.4f), UACI mean %.4f (range %.4f to %.4f); NPCR passed %d/%d, UACI passed %d/%d.\n\n'
                     % (np.mean(n), min(n), max(n), np.mean(u), min(u), max(u),
                        sum(v['npass'] == 'YES' for v in D['npcr'].values()), len(n),
                        sum(v['upass'] == 'YES' for v in D['npcr'].values()), len(n)))
        if D.get('psens'):
            L.append('## 4b. Plaintext sensitivity, one pixel +-1 (specified construction)\n')
            L.append(table(['image', 'NPCR fresh nonce', 'UACI fresh nonce', 'passed N/U', 'NPCR same nonce', 'UACI same nonce'],
                           [[nm, f4(v['n']), f4(v['u']), '%d/%d, %d/%d' % (v['npass_trials'], v['trials'], v['upass_trials'], v['trials']),
                             '%.5f' % v['n_same'], '%.6f' % v['u_same']] for nm, v in D['psens'].items()]))
        if D.get('npcr_marked'):
            L.append('## 4c. Key sensitivity on marked images (same K_H and payload)\n')
            L.append(table(['image', 'NPCR', 'UACI', 'passed N/U'],
                           [[nm, f4(v['n']), f4(v['u']), '%d, %d' % (v['npass_trials'], v['upass_trials'])] for nm, v in D['npcr_marked'].items()]))
        L.append('## 5. Block-size sweep, net bpp (Fig. 7)\n')
        L.append(table(['image'] + ['B=%d' % b for b in S.BSWEEP], [[nm] + [f4(x) for x in v] for nm, v in D['blk'].items()]))
        L.append('## 6. Decomposition-level sweep, effective bpp (Fig. 7)\n')
        L.append(table(['image', 'L=1', 'L=2', 'L=3', 'L=4'], [[nm] + [f4(x) for x in v] for nm, v in D['lvl'].items()]))
        L.append('## 7. Ablations (Table 6, Fig. 8)\n')
        L.append(table(['image', 'overflow, classic order', 'overflow, ours', 'room, modular lifting', 'room, ours'],
                       [[nm] + v for nm, v in D['abl'].items()]))
        L.append('## 8. Runtime (Section 6.6), this machine\n')
        L.append(table(['image', 'pixels', 'encrypt s', 'decrypt s'], [[nm] + v for nm, v in D['rt'].items()]))
        s512 = [v for v in D['rt'].values() if v[0] == 262144]
        if s512:
            L.append('512x512 mean: encrypt %.3f s, decrypt %.3f s.\n\n' % (np.mean([v[1] for v in s512]), np.mean([v[2] for v in s512])))
        L.append('## 9. t-map leakage and wrong-key recovery (Section 7)\n')
        L.append(table(['image', 't-map entropy plain', 'encrypted', 'pixels equal to cover after 1-bit wrong key (%)'],
                       [[nm, f4(v[0]), f4(v[1]), f4(D['wrong_key'][nm])] for nm, v in D['tleak'].items()]))
        if D['cred_ablation']:
            c = D['cred_ablation']
            L.append('## 10. Credential ablation (Section 7.6)\n\nMean ciphertext entropy over %d covers: with credentials %.5f, '
                     'with the credential terms removed %.5f.\n\n' % (c['n'], c['with_credentials'], c['without_credentials']))
    A_ = os.path.join(cfg.out, 'ablation.json')
    if os.path.exists(A_):
        A = json.load(open(A_))
        L.append('## 7b. Ablation at full depth L, same reservation and payload as the scheme (three payloads)\n')
        L.append(table(['image', 'L', 'classic ordering: px out of range (min..max)', 'deferred', 'modular room', 'proposed room'],
                       [[nm, a['levels'], '%d..%d' % (min(a['classic']), max(a['classic'])), 0, a['modular_room'], a['proposed_room']]
                        for nm, a in A.items()]))
    SA = os.path.join(cfg.out, 'spec_all.json')
    if os.path.exists(SA):
        SA = json.load(open(SA))
        L.append('## 14b. Specified construction end to end on every image (AES-256-CTR, full payload)\n')
        L.append('Stored header and tags minus the Huffman t-map cost R_t (bits): %s\n\n'
                 % ', '.join('%s %d' % (n, v['meta_bits']) for r in SA.values() for n, v in r.items() if 'meta_bits' in v))
        L.append(table(['dataset', 'images', 'payload exact', 'cover exact', 'room = base room', 'mean s'],
                       [[ds, len(r), sum(v['payload_exact'] for v in r.values()), sum(v['cover_exact'] for v in r.values()),
                         sum(v['room'] == v['base_room'] for v in r.values()), '%.2f' % np.mean([v['seconds'] for v in r.values()])]
                        for ds, r in SA.items() if r]))
    X = os.path.join(cfg.out, 'extra.json')
    if os.path.exists(X):
        X = json.load(open(X))
        L.append('## 11. Block-minimum efficiency (Table 8)\n')
        L.append(table(['image', 'sum t (bpp)', 'room (bpp)', 'eta', 'eta B=2', 'eta B=4', 'eta B=8', 'oracle net'],
                       [[nm, f4(e['sum_t']), f4(e['room']), '%.3f' % e['eta']]
                        + ['%.3f' % e.get('eta_by_B', {}).get(b, float('nan')) for b in ('2', '4', '8')]
                        + [f4(e['oracle_net'])] for nm, e in X['eff'].items()]))
        L.append('## 12. Leakage-reduced descriptor (Proposition 1)\n')
        L.append(table(['image', 'room', 'payload exact', 'cover exact', 't-map length same', 'rho spatial', 'rho permuted'],
                       [[nm, f4(l['room']), l['payload_exact'], l['cover_exact'], l['tmap_len_same'],
                         ', '.join('%.3f' % x for x in l['rho_spatial']), ', '.join('%.3f' % x for x in l['rho_permuted'])]
                        for nm, l in X['lr'].items()]))
        mk = X['multikey']; P = [p for m in mk.values() for p in m['p']]
        L.append('## 13. Statistics under 20 independent keys per cover (base construction)\n\n'
                 'H cipher %.5f to %.5f; chi-square rejections %d of %d at alpha = 0.05 (expected %.1f); max |rho| %.4f.\n\n'
                 % (min(min(m['Hc']) for m in mk.values()), max(max(m['Hc']) for m in mk.values()),
                    sum(p < 0.05 for p in P), len(P), 0.05 * len(P), max(max(m['rho']) for m in mk.values())))
    P_ = os.path.join(cfg.out, 'spec.json')
    if os.path.exists(P_):
        SP = json.load(open(P_))
        L.append('## 14. Specified construction (Section 4.6, Algorithm 3)\n\nSelf-tests: HKDF RFC 5869 %s, AES SP 800-38A %s.\n\n'
                 % (SP['selftest']['hkdf_rfc5869'], SP['selftest']['aes_sp800_38a']))
        L.append(table(['image', 'room bpp', 'raw (base)', 'payload exact', 'cover exact', 'seconds'],
                       [[nm, f4(r['room_bpp']), f4(r['recorded_raw']), r['payload_exact'], r['cover_exact'], r['seconds']]
                        for nm, r in SP['roundtrip'].items()]))
        kinds = list(next(iter(SP['tamper'].values())))
        L.append(table(['tampering', 'tau2 accepts', 'tau1 accepts', 'cover exact if accepted', 'identical on all covers'],
                       [[k] + [str(next(iter(SP['tamper'].values()))[k][q]) for q in ('tau2_accepts', 'tau1_accepts', 'cover_exact_if_accepted')]
                        + [len({json.dumps(SP['tamper'][n][k], sort_keys=True) for n in SP['tamper']}) == 1] for k in kinds]))
        st = SP['stats']; P = [p for m in st.values() for p in m['p']]
        L.append('AES-256-CTR, 20 keys per cover: H cipher %.5f to %.5f; chi-square rejections %d of %d (expected %.1f); max |rho| %.4f.\n\n'
                 % (min(min(m['Hc']) for m in st.values()), max(max(m['Hc']) for m in st.values()),
                    sum(p < 0.05 for p in P), len(P), 0.05 * len(P), max(max(m['rho']) for m in st.values())))
    M_ = os.path.join(cfg.out, 'matched.json')
    if os.path.exists(M_):
        MT = json.load(open(M_))
        L.append('## 15. Matched comparison with reimplemented BPC (Table 5)\n')
        if MT['validation']:
            L.append(table(['cover', 'published BPC net', 'reimplemented', 'room deviation %'],
                           [[nm, f4(v['published_net']), f4(v['reimpl_net']), '%+.2f' % v['room_dev_pct']] for nm, v in MT['validation'].items()]))
        L.append(table(['cover', 'BPC net', 'overflow px', 'bs', 'Lfix', 'proposed net', 'proposed eff'],
                       [[nm, f4(r['bpc_net']), r['overflow'], r['bs'], r['Lfix'], f4(r.get('prop_net', float('nan'))), f4(r.get('prop_eff', float('nan')))]
                        for nm, r in MT['rows'].items()]))
    Q = os.path.join(cfg.out, 'matched_db.json')
    if os.path.exists(Q):
        Q = json.load(open(Q))
        if Q['summary']:
            L.append('## 16. Matched comparison on the database samples\n')
            L.append(table(['dataset', 'N', 'BPC net', 'BPC sd', 'proposed net', 'proposed eff', 'net wins', 'eff wins'],
                           [[ds, s['N'], f4(s['bpc_net']), f4(s['bpc_sd']), f4(s['prop_net']), f4(s['prop_eff']), s['prop_net_wins'], s['prop_eff_wins']]
                            for ds, s in Q['summary'].items()]))
    X = os.path.join(cfg.out, 'xiao.json')
    if os.path.exists(X):
        X = json.load(open(X))
        if X['summary']:
            L.append('## 17. Identical-image comparison with Xiao et al. (2024), reimplemented (t = 8)\n')
            L.append(table(['dataset', 'N', 'ABCS net', 'ABCS sd', 'proposed net', 'proposed eff', 'net wins', 'eff wins'],
                           [[ds, s_['N'], f4(s_['xiao_net']), f4(s_['xiao_sd']), f4(s_['prop_net']), f4(s_['prop_eff']),
                             s_['prop_net_wins'], s_['prop_eff_wins']] for ds, s_ in X['summary'].items()]))
            if 'USC-SIPI' in X['rows']:
                L.append(table(['cover', 'ABCS net', 'T', 'proposed net', 'proposed eff'],
                               [[nm, f4(r['xiao_net']), r['T'], f4(r['prop_net']), f4(r['prop_eff'])]
                                for nm, r in X['rows']['USC-SIPI'].items()]))
    L.append('## Files processed\n')
    for ds, m in D['files'].items():
        L.append('- %s: %d images\n' % (ds, len(m)))
    txt = re.sub(r'^(#+ [^\n]*)\n(?!\n)', r'\1\n\n', ''.join(L), flags=re.M)
    open(os.path.join(cfg.out, 'REPORT.md'), 'w').write(txt)
    # per-image CSV for every dataset
    import csv
    with open(os.path.join(cfg.out, 'per_image.csv'), 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['dataset', 'image', 'file', 'M', 'N', 'levels', 'exact', 'raw', 'side', 'net', 'container', 'eff',
                    'H_plain', 'H_cipher', 'chi2', 'p', 'cc_H', 'cc_V', 'cc_D'])
        for ds, items in _R.items():
            for nm, r in items.items():
                w.writerow([ds, nm, r['file'], r['M'], r['N'], r['levels'], r['exact']] +
                           ['%.6f' % r[k] for k in ('raw', 'side', 'net', 'cont', 'eff', 'Hplain', 'Hciph', 'chi2', 'pchi')] +
                           ['%.6f' % x for x in r['cc']])


_R = {}


def R_exact(cfg, nm):
    return _R.get('USC-SIPI', {}).get(nm, {}).get('exact', False)


def bundle(cfg):
    z = os.path.join(cfg.out, 'results_SE-RDHEI.zip')
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as Z:
        for f in ['REPORT.md', 'per_image.csv', 'data.json', 'ablation.json', 'spec_all.json', 'extra.json', 'spec.json', 'matched.json',
                  'matched_db.json', 'xiao.json', 'secdata.npz', 'montage.npz', 'run_log.txt']:
            p = os.path.join(cfg.out, f)
            if os.path.exists(p):
                Z.write(p, f)
        for sub in ['figures', 'cache']:
            for root, _, fs in os.walk(os.path.join(cfg.out, sub)):
                for f in fs:
                    if f.endswith('_img.npz'):
                        continue
                    p = os.path.join(root, f); Z.write(p, os.path.relpath(p, cfg.out))
    print('  bundle ->', z, '(%.1f MB)' % (os.path.getsize(z) / 2 ** 20))
    return z


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--n', type=int, default=100, help='images sampled per database (0 = all)')
    ap.add_argument('--seed', type=int, default=2026)
    ap.add_argument('--trials', type=int, default=10)
    ap.add_argument('--a1'); ap.add_argument('--a2')
    ap.add_argument('--rename', default='{}', help='JSON map file-stem -> cover name')
    ap.add_argument('--dsmap', default='{}', help='JSON map folder name -> dataset name')
    ap.add_argument('--stage', default='all',
                    choices=['all', 'main', 'ablation', 'spec_all', 'extra', 'matched_db', 'xiao', 'figs', 'report'])
    ap.add_argument('--force', action='store_true')
    cfg = ap.parse_args(argv)
    os.makedirs(cfg.out, exist_ok=True)
    # results computed with another block size must never be mixed with these
    mk = os.path.join(cfg.out, 'BLOCK_SIZE')
    if os.path.exists(mk):
        if int(open(mk).read().strip()) != S.BLOCK:
            raise SystemExit('%s holds results for B = %s, this run uses B = %d: use a new output folder'
                             % (cfg.out, open(mk).read().strip(), S.BLOCK))
    elif os.path.isdir(os.path.join(cfg.out, 'cache')):
        raise SystemExit('%s holds results of an earlier run (B = 4): use a new output folder' % cfg.out)
    else:
        open(mk, 'w').write(str(S.BLOCK))
    sets = discover(cfg.data, cfg.n, cfg.seed, json.loads(cfg.rename), json.loads(cfg.dsmap))
    print('datasets found:')
    for ds, m in sets.items():
        print('  %-10s %d images' % (ds, len(m)) + (':  ' + ', '.join(m) if ds == 'USC-SIPI' else ''))
    if 'USC-SIPI' not in sets:
        print('  ! no USC-SIPI covers found: per-cover tables, sweeps and figures need them')
    else:
        miss = [c for c in ORDER if c not in sets['USC-SIPI']]
        if miss:
            print('  ! covers missing: %s (their rows will be absent from Tables 3, 5-8)' % ', '.join(miss))
    if cfg.a1 and cfg.a2:
        a1, a2 = S.load_audio(cfg.a1), S.load_audio(cfg.a2)
        src = '%s, %s' % (os.path.basename(cfg.a1), os.path.basename(cfg.a2))
    else:
        a1, a2 = S.synthetic_audio(1), S.synthetic_audio(2)
        src = 'deterministic synthetic clips (SE_RDHEI.m fallback)'
    st = cfg.stage
    if st in ('all', 'main'):
        print('\n== main measurements'); stage_main(cfg, sets, a1, a2)
    global _R
    _R = load_cache(cfg, sets)
    D = assemble(cfg, sets, _R, src)
    jdump(D, os.path.join(cfg.out, 'data.json'))
    if st in ('all', 'ablation') and 'USC-SIPI' in sets:
        print('\n== ablation at full depth'); stage_ablation(cfg, sets, a1, a2)
    if st in ('all', 'spec_all'):
        print('\n== specified construction, end to end on every image'); stage_spec_all(cfg, sets)
    if st in ('all', 'extra') and 'USC-SIPI' in sets:
        print('\n== manuscript experiment scripts'); stage_extra(cfg, sets)
    if st in ('all', 'matched_db'):
        print('\n== matched comparison on database samples'); stage_matched_db(cfg, sets, _R)
    if st in ('all', 'matched_db', 'xiao'):
        print('\n== identical-image comparison with Xiao et al. (2024), reimplemented'); stage_xiao(cfg, sets, _R)
    if st in ('all', 'figs') and 'USC-SIPI' in sets:
        print('\n== figures'); stage_figs(cfg, sets)
    print('\n== report'); report(cfg, sets, D)
    bundle(cfg)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/se_rdhei/serdhei.py
#!/usr/bin/env python3
"""SE-RDHEI: reversible data hiding in encrypted images by sign-extension room
reservation and deferred synthesis.

REFERENCE IMPLEMENTATION of the specified construction analysed in the paper
(Section 4.6, Algorithm 3, Propositions 1-3): AES-256-CTR keystreams, HKDF key
derivation, nonce-bound rejection-sampled block permutations, Huffman-coded
encrypted t-map, and a data hider holding a key of its own.

Roles (each takes only the keys it holds):
    spec_owner(I, K, nonce)                         -> C, H0, tau1   content owner
    spec_hider(C, H0, tau1, K_tmap, K_H, D)         -> C', H, tau2   data hider
    spec_extract(C', H, tau1, tau2, K_tmap, K_H)    -> D or None     payload receiver
    spec_recover(C', H, tau1, K)                    -> I or None     cover receiver
K_tmap = owner_keys(K, nonce)['tmap'] is the only owner-derived key the data
hider receives. See example.py; spec_test_vector.py checks an implementation
bit for bit.

The module also holds the shared transform and bit-level helpers (5/3 lifting,
band widths, block minimum) and the SHA-256 counter-mode stream of the base
construction (the MATLAB prototype), which the experiments use for comparison.
Pure Python + NumPy; AES-256, SHA-256, HMAC and HKDF are implemented here and
checked against RFC 5869 and NIST SP 800-38A by selftest().
"""
import numpy as np, hashlib
import os as _os
BLOCK = int(_os.environ.get('SE_BLOCK', '2'))   # block size B; the manuscript uses B = 2

# ----------------------------------------------------------- 5/3 lifting
def f1(x):
    """Row-wise integer 5/3 analysis, matching f1() in SE_RDHEI.m."""
    x = x.astype(np.int64)
    e = x[:, 0::2]
    o = x[:, 1::2]
    en = np.concatenate([e[:, 1:], e[:, -1:]], axis=1)
    d = o - np.floor_divide(e + en, 2)
    dp = np.concatenate([d[:, :1], d[:, :-1]], axis=1)
    s = e + np.floor_divide(dp + d + 2, 4)
    return s, d


def i1(s, d):
    dp = np.concatenate([d[:, :1], d[:, :-1]], axis=1)
    e = s - np.floor_divide(dp + d + 2, 4)
    en = np.concatenate([e[:, 1:], e[:, -1:]], axis=1)
    o = d + np.floor_divide(e + en, 2)
    x = np.zeros((e.shape[0], e.shape[1] * 2), dtype=np.int64)
    x[:, 0::2] = e
    x[:, 1::2] = o
    return x


def iwt2(I):
    L, H = f1(I)
    LL, LH = f1(L.T)
    HL, HH = f1(H.T)
    return dict(LL=LL.T, LH=LH.T, HL=HL.T, HH=HH.T)


def iiwt2(B):
    L = i1(B['LL'].T, B['LH'].T).T
    H = i1(B['HL'].T, B['HH'].T).T
    return i1(L, H)


def max_levels(shape, cap=4):
    M, N = shape
    L = 0
    while L < cap and M % 2 == 0 and N % 2 == 0 and min(M, N) >= 2:
        M //= 2
        N //= 2
        L += 1
    return L


def iwt2_multi(I, L):
    bands, cur = {}, I.astype(np.int64)
    for lv in range(1, L + 1):
        B = iwt2(cur)
        bands[f'HL{lv}'] = B['HL']
        bands[f'LH{lv}'] = B['LH']
        bands[f'HH{lv}'] = B['HH']
        cur = B['LL']
    bands['LL'] = cur
    return bands


def iiwt2_multi(bands, L):
    cur = bands['LL']
    for lv in range(L, 0, -1):
        cur = iiwt2(dict(LL=cur, LH=bands[f'LH{lv}'],
                         HL=bands[f'HL{lv}'], HH=bands[f'HH{lv}']))
    return cur


def band_names(L):
    det = []
    for lv in range(1, L + 1):
        det += [f'HL{lv}', f'LH{lv}', f'HH{lv}']
    return ['LL'] + det


def band_id(nm):
    if nm == 'LL':
        return 0
    k = ['HL', 'LH', 'HH'].index(nm[:2]) + 1
    return (int(nm[2:]) << 4) | k


# ------------------------------------------------- widths and reservation
def band_width(v):
    lo, hi = int(v.min()), int(v.max())
    W = 2
    while not (-(2 ** (W - 1)) <= lo and hi <= 2 ** (W - 1) - 1):
        W += 1
    return W


def lead_ident(u, W):
    """Number of leading bits equal to the sign bit, as in lead_ident()."""
    sg = (u >> (W - 1)) & 1
    n = np.ones(u.shape, dtype=np.int64)
    for k in range(W - 1, 0, -1):
        bit = (u >> (k - 1)) & 1
        n = n + ((bit == sg) & (n == (W - k))).astype(np.int64)
    return n


def block_t(u, W, tmax, BLK):
    t = np.minimum(lead_ident(u, W) - 1, tmax)
    hh, ww = t.shape
    bh, bw = hh // BLK, ww // BLK
    if bh == 0 or bw == 0:
        return np.zeros((0, 0), dtype=np.int64)
    t = t[:bh * BLK, :bw * BLK]
    return t.reshape(bh, BLK, bw, BLK).min(axis=(1, 3))


def expand_t(tm, shape, BLK):
    f = np.zeros(shape, dtype=np.int64)
    if tm.size:
        f[:tm.shape[0] * BLK, :tm.shape[1] * BLK] = np.kron(tm, np.ones((BLK, BLK), dtype=np.int64))
    return f


# ----------------------------------------------------- SHA-256 CTR / HKDF
def sha(b):
    return hashlib.sha256(bytes(b)).digest()


def ctr(key, nbytes):
    out = bytearray()
    i = 0
    while len(out) < nbytes:
        out += sha(bytes(key) + i.to_bytes(4, 'big'))
        i += 1
    return bytes(out[:nbytes])


def hmac256(key, msg):
    key = bytes(key)
    if len(key) > 64:
        key = sha(key)
    k = key + b'\x00' * (64 - len(key))
    inner = sha(bytes(x ^ 0x36 for x in k) + bytes(msg))
    return sha(bytes(x ^ 0x5c for x in k) + inner)


def hkdf(prk, info, n):
    out, t, i = b'', b'', 1
    while len(out) < n:
        t = hmac256(prk, t + bytes(info) + bytes([i]))
        out += t
        i += 1
    return out[:n]


def keys(km, nonce):
    prk = hmac256(b'SE-RDHEI-v2', km)
    return dict(enc=hkdf(prk, b'ENC', 32), prm=hkdf(prk, b'PERM', 32),
                nonce=nonce)


def keystream(K, nm, n, W):
    k = hkdf(K['enc'], bytes([band_id(nm)]) + K['nonce'], 32)
    by = ctr(k, 2 * n)
    raw = np.frombuffer(by, dtype='<u2').astype(np.int64)
    return raw[:n] & (2 ** W - 1)


def permutation(K, nm, n):
    raw = ctr(bytes(K['prm']) + b'PERM' + bytes([band_id(nm)]) + K['nonce'], 8 * n)
    r = np.frombuffer(raw, dtype='>u8')
    p = np.arange(n)
    for i in range(n - 1, 0, -1):
        j = int(r[i] % (i + 1))
        p[i], p[j] = p[j], p[i]
    return p


# ------------------------------------------------------------- pipeline
def encrypt(I, km=b'\x00' * 32, nonce=b'\x01' * 12, BLK=BLOCK, Lcap=4):
    I = I.astype(np.int64)
    L = max_levels(I.shape, Lcap)
    bands = iwt2_multi(I, L)
    names = band_names(L)
    Wd, C, room = {}, {}, 0
    for nm in names:
        v = bands[nm]
        W = band_width(v)
        Wd[nm] = W
        u = v % (2 ** W)
        if nm != 'LL':                     # LL carries no room
            tm = block_t(u, W, W - 2, BLK)
            t = expand_t(tm, u.shape, BLK)
            u = u & (2 ** (W - t) - 1)     # vacate
            room += int(tm.sum()) * BLK * BLK
        n = u.size
        ks = keystream({'enc': km if False else _K['enc'], 'nonce': nonce}, nm, n, W) \
            if False else keystream(_K, nm, n, W)
        flat = u.T.reshape(-1)             # MATLAB column-major flatten
        diff = (flat ^ ks)
        p = permutation(_K, nm, n)
        C[nm] = diff[p]
    return C, Wd, names, room, L


def packbits_cipher(C, Wd, names):
    bits = []
    for nm in names:
        W, v = Wd[nm], C[nm]
        m = np.zeros((W, v.size), dtype=np.uint8)
        for k in range(W):
            m[k] = (v >> (W - 1 - k)) & 1
        bits.append(m.T.reshape(-1))       # per coefficient, MSB first
    b = np.concatenate(bits)
    b = b[:len(b) // 8 * 8].reshape(-1, 8)
    return (b * (1 << np.arange(7, -1, -1))).sum(axis=1).astype(np.uint8)


# ------------------------------------------------------------ statistics
def entropy8(x):
    p = np.bincount(np.asarray(x).ravel(), minlength=256) / np.asarray(x).size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def chi2stat(x):
    o = np.bincount(np.asarray(x).ravel(), minlength=256)
    e = np.asarray(x).size / 256
    return float(((o - e) ** 2 / e).sum())


def corr3(x):
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        s = int(np.sqrt(x.size))
        x = x[:s * s].reshape(s, s)
    out = []
    for dy, dx in [(0, 1), (1, 0), (1, 1)]:
        a = x[:x.shape[0] - dy, :x.shape[1] - dx].ravel()
        b = x[dy:, dx:].ravel()
        out.append(float((np.mean(a * b) - a.mean() * b.mean()) /
                         (a.std() * b.std() + 1e-300)))
    return out


_K = None


def run(path, km=b'\x00' * 32, nonce=b'\x01' * 12, BLK=BLOCK):
    global _K
    _K = keys(km, nonce)
    from PIL import Image
    I = np.array(Image.open(path).convert('L'))
    C, Wd, names, room, L = encrypt(I, km, nonce, BLK)
    E = packbits_cipher(C, Wd, names)
    cont = sum(Wd[nm] * C[nm].size for nm in names) / I.size
    return dict(I=I, E=E, Hp=entropy8(I), Hc=entropy8(E),
                chi=chi2stat(E), cp=corr3(I),
                cc=corr3(E[:I.size].reshape(I.shape) if E.size >= I.size else E),
                raw=room / I.size, cont=cont, L=L, Wd=Wd)


# ------------------------------------------- full embed / extract / recover
def roundtrip(path, km=b'\x00' * 32, nonce=b'\x01' * 12, BLK=BLOCK, Lcap=4, seed=11):
    """Encrypt, embed a random payload filling the room, extract it and
    recover the cover. Returns arrays for display plus exactness flags."""
    global _K
    _K = keys(km, nonce)
    from PIL import Image
    I = np.array(Image.open(path).convert('L')).astype(np.int64)
    L = max_levels(I.shape, Lcap)
    bands = iwt2_multi(I, L)
    names = band_names(L)
    rng = np.random.default_rng(seed)
    Wd, Cenc, Cmrk, meta, sent = {}, {}, {}, {}, []
    for nm in names:
        v = bands[nm]; W = band_width(v); Wd[nm] = W
        u = v % (2 ** W)
        t = np.zeros(u.shape, dtype=np.int64)
        if nm != 'LL':
            t = expand_t(block_t(u, W, W - 2, BLK), u.shape, BLK)
            u = u & (2 ** (W - t) - 1)
        n = u.size
        ks = keystream(_K, nm, n, W)
        p = permutation(_K, nm, n)
        tf = t.T.reshape(-1)
        diff = u.T.reshape(-1) ^ ks
        pay = np.array([int(rng.integers(0, 2 ** int(k))) if k else 0 for k in tf],
                       dtype=np.int64)
        mark = (diff & (2 ** (W - tf) - 1)) | (pay << (W - tf))
        Cenc[nm] = diff[p]; Cmrk[nm] = mark[p]
        meta[nm] = (W, tf, ks, p, v.shape); sent.append((nm, pay))
    # receiver
    rec, pay_ok, bits = {}, True, 0
    for nm, pay in sent:
        W, tf, ks, p, shp = meta[nm]
        inv = np.empty_like(p); inv[p] = np.arange(p.size)
        m = Cmrk[nm][inv]
        pay_ok &= bool(np.array_equal(m >> (W - tf), pay))
        bits += int(tf.sum())
        x = (m ^ ks) & (2 ** (W - tf) - 1)
        carrier = (x >> (W - tf - 1)) & 1
        u = x + carrier * (2 ** tf - 1) * 2 ** (W - tf)
        v = np.where(u >= 2 ** (W - 1), u - 2 ** W, u)
        rec[nm] = v.reshape(shp[::-1]).T
    R = iiwt2_multi(rec, L)
    def img(C):
        E = packbits_cipher(C, Wd, names)
        s = min(I.shape[0], int(np.sqrt(E.size)))
        return E[:s * s].reshape(s, s)
    return dict(cover=I.astype(np.uint8), enc=img(Cenc), mark=img(Cmrk),
                rec=R, exact=bool(np.array_equal(R, I)), pay_ok=pay_ok,
                bits=bits, rate=bits / I.size)


# ------------------------------- leakage-reduced variant (block permutation)
def block_perm(key, tag, n):
    """Pseudorandom permutation of n blocks from a key."""
    raw = ctr(bytes(key) + tag, 8 * max(n, 1))
    r = np.frombuffer(raw, dtype='>u8')
    p = np.arange(n)
    for i in range(n - 1, 0, -1):
        j = int(r[i] % (i + 1)); p[i], p[j] = p[j], p[i]
    return p


def owner_encrypt_lr(I, Kown, BLK=BLOCK, Lcap=4):
    """Owner side. Coefficients are grouped by block, blocks are permuted by
    sigma (key K_prm, never given to the data hider), and the t-map is
    published in the same permuted order."""
    I = I.astype(np.int64); L = max_levels(I.shape, Lcap)
    bands = iwt2_multi(I, L); names = band_names(L); out = {}
    for nm in names:
        v = bands[nm]; W = band_width(v); u = v % (2 ** W)
        h, w = u.shape
        bh, bw = (h // BLK, w // BLK) if nm != 'LL' else (0, 0)
        tm = block_t(u, W, W - 2, BLK) if nm != 'LL' and bh and bw else np.zeros((0, 0), np.int64)
        t = expand_t(tm, u.shape, BLK)
        uv = u & (2 ** (W - t) - 1)
        # groups: full blocks first (B*B coefficients each), then remainder
        inblk = np.zeros(u.shape, bool); inblk[:bh * BLK, :bw * BLK] = True
        blocks = uv[:bh * BLK, :bw * BLK].reshape(bh, BLK, bw, BLK).transpose(0, 2, 1, 3).reshape(bh * bw, BLK * BLK)
        rest = uv[~inblk]
        sig = block_perm(Kown['prm'], b'BLK' + nm.encode(), bh * bw)
        rho = block_perm(Kown['prm'], b'REST' + nm.encode(), rest.size)
        flat = np.concatenate([blocks[sig].reshape(-1), rest[rho]])
        ks = keystream(Kown, nm, flat.size, W)
        out[nm] = dict(C=flat ^ ks, W=W, Tperm=tm.reshape(-1)[sig] if tm.size else np.zeros(0, np.int64),
                       shape=u.shape, bh=bh, bw=bw, nrest=rest.size)
    return out, L, names


def hider_embed_lr(ct, names, payload_bits, K_hide, nonce, BLK=BLOCK):
    """Data hider: sees only C, the permuted t-map and W_b. No K_prm, no K_enc."""
    tot = sum(int(ct[nm]['Tperm'].sum()) * BLK * BLK for nm in names)
    assert payload_bits.size <= tot
    stream = np.unpackbits(np.frombuffer(ctr(bytes(K_hide) + nonce, (tot + 7) // 8), np.uint8))[:tot]
    pad = np.zeros(tot, np.uint8); pad[:payload_bits.size] = payload_bits
    bits = pad ^ stream                               # payload encrypted under the hider's own key
    k = 0
    for nm in names:
        c = ct[nm]; W = c['W']; C = c['C'].copy()
        for g, T in enumerate(c['Tperm']):
            for j in range(BLK * BLK):
                i = g * BLK * BLK + j
                for q in range(T):                   # top T bits, MSB first
                    p = W - 1 - q
                    C[i] = (C[i] & ~(1 << p)) | (int(bits[k]) << p); k += 1
        c['Cm'] = C
    return tot


def receiver_extract_lr(ct, names, n_payload, K_hide, nonce, BLK=BLOCK):
    """Path a: needs only the permuted t-map and K_hide."""
    bits = []
    for nm in names:
        c = ct[nm]; W = c['W']
        for g, T in enumerate(c['Tperm']):
            for j in range(BLK * BLK):
                i = g * BLK * BLK + j
                for q in range(T):
                    bits.append((int(c['Cm'][i]) >> (W - 1 - q)) & 1)
    bits = np.array(bits, np.uint8)
    stream = np.unpackbits(np.frombuffer(ctr(bytes(K_hide) + nonce, (bits.size + 7) // 8), np.uint8))[:bits.size]
    return (bits ^ stream)[:n_payload]


def receiver_recover_lr(ct, names, L, Kown, BLK=BLOCK):
    """Path b: needs the master key."""
    rec = {}
    for nm in names:
        c = ct[nm]; W = c['W']; bh, bw = c['bh'], c['bw']; h, w = c['shape']
        ks = keystream(Kown, nm, c['Cm'].size, W)
        x = c['Cm'] ^ ks
        nb = bh * bw
        tb = np.repeat(c['Tperm'], BLK * BLK) if nb else np.zeros(0, np.int64)
        tf = np.concatenate([tb, np.zeros(c['nrest'], np.int64)])
        x = x & (2 ** (W - tf) - 1)
        car = (x >> (W - tf - 1)) & 1
        uu = x + car * (2 ** tf - 1) * 2 ** (W - tf)
        sig = block_perm(Kown['prm'], b'BLK' + nm.encode(), nb)
        rho = block_perm(Kown['prm'], b'REST' + nm.encode(), c['nrest'])
        blocks = np.empty((nb, BLK * BLK), np.int64); blocks[sig] = uu[:nb * BLK * BLK].reshape(nb, BLK * BLK)
        rest = np.empty(c['nrest'], np.int64); rest[rho] = uu[nb * BLK * BLK:]
        U = np.zeros((h, w), np.int64)
        if nb:
            U[:bh * BLK, :bw * BLK] = blocks.reshape(bh, bw, BLK, BLK).transpose(0, 2, 1, 3).reshape(bh * BLK, bw * BLK)
        m = np.zeros((h, w), bool); m[:bh * BLK, :bw * BLK] = True
        U[~m] = rest
        rec[nm] = np.where(U >= 2 ** (W - 1), U - 2 ** W, U)
    return iiwt2_multi(rec, L)


def huff_len(sym):
    """Total Huffman code length of a symbol sequence (zeroth order)."""
    import heapq
    f = np.bincount(sym) if sym.size else np.zeros(0, int)
    f = [int(x) for x in f if x > 0]
    if len(f) <= 1:
        return int(sum(f))
    h = list(f); heapq.heapify(h); tot = 0
    while len(h) > 1:
        a = heapq.heappop(h); b = heapq.heappop(h); tot += a + b; heapq.heappush(h, a + b)
    return tot


def huff_code_lengths(cnt):
    """Optimal (Huffman) code length per symbol; 0 for unused symbols."""
    import heapq
    act = [i for i, c in enumerate(cnt) if c > 0]
    L = [0] * len(cnt)
    if len(act) == 1:
        L[act[0]] = 1
        return L
    h = [(int(cnt[i]), k, [i]) for k, i in enumerate(act)]
    heapq.heapify(h); k = len(h)
    while len(h) > 1:
        a = heapq.heappop(h); b = heapq.heappop(h)
        for i in a[2] + b[2]:
            L[i] += 1
        heapq.heappush(h, (a[0] + b[0], k, a[2] + b[2])); k += 1
    return L


def _canonical(L):
    """Canonical codewords from code lengths: symbol -> (code, length)."""
    code, prev, out = 0, 0, {}
    for ln, sym in sorted((ln, s) for s, ln in enumerate(L) if ln > 0):
        code <<= ln - prev; out[sym] = (code, ln); code += 1; prev = ln
    return out


def tmap_encode(syms):
    """Specified t-map descriptor, Eq. (h0): one byte alpha, a 4-bit code length
    per symbol (the code table), then the Huffman-coded symbols; zero-padded to
    whole bytes.  Returns (bytes, number of meaningful bits)."""
    syms = np.asarray(syms, np.int64)
    alpha = max(2, int(syms.max(initial=0))) + 2
    cnt = np.bincount(syms, minlength=alpha)
    L = huff_code_lengths(cnt)
    assert max(L) <= 15 and alpha <= 255
    bits = [(alpha >> (7 - q)) & 1 for q in range(8)]
    for ln in L:
        bits += [(ln >> (3 - q)) & 1 for q in range(4)]
    cw = _canonical(L)
    for x in syms.tolist():
        c, ln = cw[x]
        bits += [(c >> (ln - 1 - q)) & 1 for q in range(ln)]
    nbits = len(bits)
    return np.packbits(np.array(bits + [0] * (-nbits % 8), np.uint8)).tobytes(), nbits


def tmap_decode(data, nsym):
    bits = np.unpackbits(np.frombuffer(data, np.uint8)).tolist()
    alpha = int(''.join(map(str, bits[:8])), 2); pos = 8
    L = []
    for _ in range(alpha):
        L.append(int(''.join(map(str, bits[pos:pos + 4])), 2)); pos += 4
    inv = {(c, ln): s for s, (c, ln) in _canonical(L).items()}
    out = []
    while len(out) < nsym:
        c = ln = 0
        while (c, ln) not in inv:
            c = (c << 1) | bits[pos]; pos += 1; ln += 1
            if ln > 15:
                raise ValueError('invalid t-map code')
        out.append(inv[(c, ln)])
    return np.array(out, np.int64)


def _nblocks(H, nm):
    if nm == 'LL':
        return 0
    lv = int(nm[2:]); B = H['B']
    return ((H['M'] >> lv) // B) * ((H['N'] >> lv) // B)


# --------------------------- specified stream: AES-256-CTR (SP 800-38A)
CTR_MAX_BYTES = 16 * 2 ** 24        # 2^24 counter blocks per stream


def aes_ctr(key32, iv16, nbytes):
    """AES-256-CTR keystream (SP 800-38A). The 16-byte counter block is
    nonce (12 B) || domain byte || 24-bit block counter, big-endian, starting
    at 0: the first keystream block is AES_K(nonce || dom || 0x000000). The
    counter is incremented by one per block as a big-endian integer; a request
    that would wrap it (more than 2^24 blocks) is rejected rather than allowed
    to carry into the domain byte. The last block is truncated to nbytes."""
    iv16 = bytes(iv16)
    if len(iv16) != 16 or iv16[13:] != b'\x00\x00\x00':
        raise ValueError('counter block must end in a zero 24-bit counter')
    if not 0 <= nbytes <= CTR_MAX_BYTES:
        raise ValueError('keystream request exceeds 2^24 counter blocks')
    return _aes_ctr_raw(key32, iv16, nbytes)


def _aes_ctr_raw(key32, iv16, nbytes):
    """Plain SP 800-38A CTR from an arbitrary counter block (self-test only)."""
    from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
    enc = Cipher(algorithms.AES(bytes(key32)), modes.CTR(bytes(iv16))).encryptor()
    return enc.update(b'\x00' * nbytes) + enc.finalize()


def hkdf_extract(salt, ikm):
    return hmac256(salt, ikm)


def iv_for(nonce, dom):
    """Initial counter block: 96-bit nonce || one domain byte || 24-bit counter = 0.
    Matches aes_ctr() in SE_RDHEI.m, which pads [nonce band_id] with zeros."""
    return bytes(nonce) + bytes([dom]) + b'\x00\x00\x00'


DOM_TMAP, DOM_HIDE = 0xF0, 0xF1          # domain bytes outside the band_id range


def spec_perm(K_prm, tag, nm, nonce, n):
    """sigma_b / rho_b of the specified construction: Fisher-Yates driven by
    64-bit big-endian words of AES-256-CTR under a key bound to K_prm, the
    purpose tag, the band and the nonce, so every image gets fresh permutations.

    Each index is drawn without modulo bias by rejection sampling: for step i
    (i = n-1 down to 1, m = i+1) the next unused word w is accepted iff
    w < 2^64 - (2^64 mod m), and then j = w mod m. Words are consumed strictly
    in stream order; a rejected word is skipped. With a uniform stream the
    result is exactly uniform over all n! permutations."""
    k = hkdf(K_prm, tag + bytes([band_id(nm)]) + bytes(nonce), 32)
    iv = iv_for(nonce, band_id(nm))
    avail = n + 64                                   # words generated so far
    r = np.frombuffer(aes_ctr(k, iv, 8 * avail), dtype='>u8')
    p = np.arange(n); pos = 0; R = 1 << 64
    for i in range(n - 1, 0, -1):
        m = i + 1; lim = R - (R % m)
        while True:
            if pos == avail:                         # extend; CTR output is prefix-stable
                avail *= 2
                r = np.frombuffer(aes_ctr(k, iv, 8 * avail), dtype='>u8')
            w = int(r[pos]); pos += 1
            if w < lim:
                break
        j = w % m; p[i], p[j] = p[j], p[i]
    return p


def keystream_aes(K, nm, n, W):
    """Coefficient keystream of the specification (= SE_RDHEI.m with useAES)."""
    assert 2 <= W <= 16, 'a band width above 16 bits needs a wider keystream word'
    k = hkdf(K['enc'], bytes([band_id(nm)]) + K.get('h1', b'') + K.get('h2', b'')
             + K.get('F', {}).get(nm, b'') + K['nonce'], 32)
    by = aes_ctr(k, iv_for(K['nonce'], band_id(nm)), 2 * n)
    raw = np.frombuffer(by, dtype='<u2').astype(np.int64)
    return raw[:n] & (2 ** W - 1)


def selftest():
    # RFC 5869, Test Case 1 (SHA-256)
    ikm = bytes([0x0b] * 22); salt = bytes(range(13)); info = bytes(range(0xf0, 0xfa))
    okm = hkdf(hkdf_extract(salt, ikm), info, 42).hex()
    ok1 = okm == ('3cb25f25faacd57a90434f64d0362f2a2d2d0a90cf1a5a4c5db02d56ecc4c5bf'
                  '34007208d5b887185865')
    # NIST SP 800-38A F.5.5 CTR-AES256.Encrypt, first block
    key = bytes.fromhex('603deb1015ca71be2b73aef0857d77811f352c073b6108d72d9810a30914dff4')
    ctr0 = bytes.fromhex('f0f1f2f3f4f5f6f7f8f9fafbfcfdfeff')
    pt = bytes.fromhex('6bc1bee22e409f96e93d7e117393172a')
    ct = bytes(a ^ b for a, b in zip(_aes_ctr_raw(key, ctr0, 16), pt)).hex()
    ok2 = ct == '601ec313775789a5b7a7f504bbf3d228'
    return ok1, ok2


# ============================================================================
# SPECIFIED CONSTRUCTION: leakage-reduced descriptor + AES-256-CTR + key split
# Each role is a separate function that receives only the keys that role holds.
# ============================================================================
def owner_keys(K, nonce):
    prk = hkdf_extract(b'SE-RDHEI-v2', K)
    return dict(enc=hkdf(prk, b'ENC', 32), prm=hkdf(prk, b'PERM', 32),
                tmap=hkdf(prk, b'TMAP', 32), mac=hkdf(prk, b'MAC', 32), nonce=bytes(nonce))


def hider_keys(KH):
    prk = hkdf_extract(b'SE-RDHEI-hider', KH)
    return dict(hide=hkdf(prk, b'HIDE', 32), tag=hkdf(prk, b'TAG', 32))


def _bits_hi(v, W, k):           # top k bits of a W-bit word, MSB first
    return [(v >> (W - 1 - q)) & 1 for q in range(k)]


def _ser_header(H):
    out = bytearray(H['nonce'] + H['M'].to_bytes(4, 'big') + H['N'].to_bytes(4, 'big')
                    + bytes([H['L'], H['B']]))
    for nm in H['names']:
        out += bytes([H['W'][nm]])
    out += H['Tcode']
    if 'nD' in H:
        out += H['nD'].to_bytes(3, 'big')
    return bytes(out)


def _tfield(H, nm, n):
    B2 = H['B'] ** 2
    t = np.repeat(H['Tplain'][nm], B2) if H['Tplain'][nm].size else np.zeros(0, np.int64)
    return np.concatenate([t, np.zeros(n - t.size, np.int64)])


def _ser_nonreserved(C, H):
    bits = []
    for nm in H['names']:
        W = H['W'][nm]; c = C[nm]; tf = _tfield(H, nm, c.size)
        for v, t in zip(c.tolist(), tf.tolist()):
            low = W - t
            bits += [(v >> (low - 1 - q)) & 1 for q in range(low)]
    return np.packbits(np.array(bits, np.uint8)).tobytes()


def _ser_all(C, H):
    out = bytearray()
    for nm in H['names']:
        out += C[nm].astype('>u2').tobytes()
    return bytes(out)


def spec_owner(I, K, nonce, BLK=BLOCK, Lcap=4):
    """Content owner. Holds K. Emits (C, H0, tau1); gives the data hider K_tmap only."""
    k = owner_keys(K, nonce)
    I = I.astype(np.int64); L = max_levels(I.shape, Lcap)
    bands = iwt2_multi(I, L); names = band_names(L)
    C, H = {}, dict(nonce=bytes(nonce), M=I.shape[0], N=I.shape[1], L=L, B=BLK,
                    names=names, W={}, Tplain={})
    for nm in names:
        v = bands[nm]; W = band_width(v); u = v % (2 ** W); h, w = u.shape
        bh, bw = (h // BLK, w // BLK) if nm != 'LL' else (0, 0)
        tm = block_t(u, W, W - 2, BLK) if (nm != 'LL' and bh and bw) else np.zeros((0, 0), np.int64)
        uv = u & (2 ** (W - expand_t(tm, u.shape, BLK)) - 1)
        inb = np.zeros(u.shape, bool); inb[:bh * BLK, :bw * BLK] = True
        blocks = uv[:bh * BLK, :bw * BLK].reshape(bh, BLK, bw, BLK).transpose(0, 2, 1, 3).reshape(bh * bw, BLK * BLK)
        rest = uv[~inb]
        sig = spec_perm(k['prm'], b'BLK', nm, k['nonce'], bh * bw)
        rho = spec_perm(k['prm'], b'REST', nm, k['nonce'], rest.size)
        flat = np.concatenate([blocks[sig].reshape(-1), rest[rho]])
        C[nm] = flat ^ keystream_aes(k, nm, flat.size, W)
        tp = tm.reshape(-1)[sig].astype(np.int64) if tm.size else np.zeros(0, np.int64)
        H['W'][nm] = W; H['Tplain'][nm] = tp
    code, nbits = tmap_encode(np.concatenate([H['Tplain'][nm] for nm in names]))
    ks = np.frombuffer(aes_ctr(k['tmap'], iv_for(nonce, DOM_TMAP), len(code)), np.uint8)
    H['Tcode'] = bytes(np.frombuffer(code, np.uint8) ^ ks); H['Tbits'] = nbits
    H0 = {kk: vv for kk, vv in H.items() if kk != 'Tplain'}
    tau1 = hmac256(k['mac'], _ser_header(H0) + _ser_nonreserved(C, H))
    return C, H0, tau1


def _decrypt_tmap(H0, K_tmap):
    e = np.frombuffer(H0['Tcode'], np.uint8)
    ks = np.frombuffer(aes_ctr(K_tmap, iv_for(H0['nonce'], DOM_TMAP), max(e.size, 1)), np.uint8)[:e.size]
    n = [_nblocks(H0, nm) for nm in H0['names']]
    syms = tmap_decode(bytes(e ^ ks), sum(n))
    T, o = {}, 0
    for nm, k in zip(H0['names'], n):
        T[nm] = syms[o:o + k]; o += k
    return T


def spec_recover(Cm, H1, tau1, K):
    """Cover receiver. Holds K. Verifies tau1 over H0 and non-reserved bits, then recovers."""
    k = owner_keys(K, H1['nonce'])
    H0 = {kk: vv for kk, vv in H1.items() if kk != 'nD'}
    H = dict(H0)
    try:                                    # a corrupt descriptor cannot authenticate
        H['Tplain'] = _decrypt_tmap(H0, k['tmap'])
    except (ValueError, IndexError):
        return None
    if hmac256(k['mac'], _ser_header(H0) + _ser_nonreserved(Cm, H)) != tau1:
        return None
    B = H['B']; rec = {}
    for nm in H['names']:
        W = H['W'][nm]; c = Cm[nm]; tf = _tfield(H, nm, c.size)
        x = (c ^ keystream_aes(k, nm, c.size, W)) & (2 ** (W - tf) - 1)
        car = (x >> (W - tf - 1)) & 1
        uu = x + car * (2 ** tf - 1) * 2 ** (W - tf)
        # shape of the band
        lv = 0 if nm == 'LL' else int(nm[2:])
        lvl = H['L'] if nm == 'LL' else lv
        h, w = H['M'] >> lvl, H['N'] >> lvl
        bh, bw = (h // B, w // B) if nm != 'LL' else (0, 0)
        nb = bh * bw; nrest = h * w - nb * B * B
        sig = spec_perm(k['prm'], b'BLK', nm, k['nonce'], nb)
        rho = spec_perm(k['prm'], b'REST', nm, k['nonce'], nrest)
        blocks = np.empty((nb, B * B), np.int64); blocks[sig] = uu[:nb * B * B].reshape(nb, B * B)
        rest = np.empty(nrest, np.int64); rest[rho] = uu[nb * B * B:]
        U = np.zeros((h, w), np.int64)
        if nb:
            U[:bh * B, :bw * B] = blocks.reshape(bh, bw, B, B).transpose(0, 2, 1, 3).reshape(bh * B, bw * B)
        m = np.zeros((h, w), bool); m[:bh * B, :bw * B] = True; U[~m] = rest
        rec[nm] = np.where(U >= 2 ** (W - 1), U - 2 ** W, U)
    return iiwt2_multi(rec, H['L'])


# ---- vectorised versions of the bit-level helpers (same semantics) ----
def _bitmat(c, W):                 # rows = coefficients, cols = bits MSB first
    return ((c[:, None] >> (W - 1 - np.arange(W))[None, :]) & 1).astype(np.uint8)


def _ser_nonreserved(C, H):
    out = []
    for nm in H['names']:
        W = H['W'][nm]; c = C[nm]; tf = _tfield(H, nm, c.size)
        m = _bitmat(c, W)
        keep = np.arange(W)[None, :] >= tf[:, None]
        out.append(m[keep])
    return np.packbits(np.concatenate(out)).tobytes()


def _reserved_bits(C, H):
    out = []
    for nm in H['names']:
        W = H['W'][nm]; c = C[nm]; tf = _tfield(H, nm, c.size)
        m = _bitmat(c, W)
        out.append(m[np.arange(W)[None, :] < tf[:, None]])
    return np.concatenate(out)


def _write_reserved(C, H, bits):
    Cm, k = {}, 0
    for nm in H['names']:
        W = H['W'][nm]; c = C[nm]; tf = _tfield(H, nm, c.size)
        m = _bitmat(c, W); sel = np.arange(W)[None, :] < tf[:, None]
        n = int(sel.sum()); m[sel] = bits[k:k + n]; k += n
        Cm[nm] = (m.astype(np.int64) << (W - 1 - np.arange(W))[None, :]).sum(axis=1)
    return Cm


def hider_room(H0, K_tmap):
    """Number of reserved bits the data hider can fill, from the encrypted t-map."""
    T = _decrypt_tmap(H0, K_tmap)
    return sum(int(T[nm].sum()) for nm in H0['names']) * H0['B'] ** 2


def spec_hider(C, H0, tau1, K_tmap, KH, D):
    """Data hider. Holds K_tmap (from the owner) and its own K_H; never K, K_enc,
    K_prm or K_mac. Writes the payload D (0/1 array), encrypted under K_hide,
    into the reserved bits and returns (C', H, tau2)."""
    kh = hider_keys(KH)
    H = dict(H0); H['Tplain'] = _decrypt_tmap(H0, K_tmap)
    tot = sum(int(H['Tplain'][nm].sum()) * H['B'] ** 2 for nm in H['names'])
    assert D.size <= tot, 'payload exceeds room'
    st = np.unpackbits(np.frombuffer(aes_ctr(kh['hide'], iv_for(H0['nonce'], DOM_HIDE), (tot + 7) // 8), np.uint8))[:tot]
    bits = np.zeros(tot, np.uint8); bits[:D.size] = D; bits ^= st
    Cm = _write_reserved(C, H, bits)
    H1 = dict(H0); H1['nD'] = int(D.size)
    tau2 = hmac256(kh['tag'], _ser_header(H1) + _ser_all(Cm, H1) + tau1)
    return Cm, H1, tau2


def spec_extract(Cm, H1, tau1, tau2, K_tmap, KH):
    """Payload receiver. Holds K_tmap and K_H. Verifies tau2, then returns the
    payload, or None if the tag does not verify."""
    kh = hider_keys(KH)
    if hmac256(kh['tag'], _ser_header(H1) + _ser_all(Cm, H1) + tau1) != tau2:
        return None
    H = dict(H1); H['Tplain'] = _decrypt_tmap(H1, K_tmap)
    bits = _reserved_bits(Cm, H)
    st = np.unpackbits(np.frombuffer(aes_ctr(kh['hide'], iv_for(H1['nonce'], DOM_HIDE), (bits.size + 7) // 8), np.uint8))[:bits.size]
    return (bits ^ st)[:H1['nD']]

In [ ]:
%%writefile /content/se_rdhei/bpc.py
#!/usr/bin/env python3
"""Reimplementation of Yin, Peng & Xiang, "Reversible data hiding in encrypted
images based on pixel prediction and bit-plane compression" (IEEE TDSC 2022,
arXiv:1911.01699), for a matched-image comparison.

Implemented as described: MED prediction, sign + 7-bit magnitude error planes,
bit-plane rearrangement (four scan types over square blocks, shortest kept),
and bit-stream compression by run-length coding with a fixed raw length Lfix.
Details the paper leaves open (border prediction, block size, Lfix) are
parameters; validate() checks them against the paper's published plane lengths.
"""
import numpy as np
from numba import njit


def med_errors(I, border='reference'):
    I = I.astype(np.int64)
    x2 = np.zeros_like(I); x3 = np.zeros_like(I); x1 = np.zeros_like(I)
    x2[1:, :] = I[:-1, :]; x3[:, 1:] = I[:, :-1]; x1[1:, 1:] = I[:-1, :-1]
    mn = np.minimum(x2, x3); mx = np.maximum(x2, x3)
    p = np.where(x1 <= mn, mx, np.where(x1 >= mx, mn, x2 + x3 - x1))
    if border == 'reference':            # first row and column stored as is (error := 0 here)
        p[0, :] = I[0, :]; p[:, 0] = I[:, 0]
    elif border == 'neighbour':           # first row predicted from left, first column from top
        p[0, 1:] = I[0, :-1]; p[1:, 0] = I[:-1, 0]; p[0, 0] = I[0, 0]
    e = I - p
    over = np.abs(e) > 127
    return e, over


def planes(e):
    """Plane 8 = sign, planes 7..1 = magnitude bits, MSB first. Returns dict k -> 2-D 0/1."""
    m = np.abs(e).clip(0, 127)
    out = {8: (e < 0).astype(np.uint8)}
    for k in range(7, 0, -1):
        out[k] = ((m >> (k - 1)) & 1).astype(np.uint8)
    return out


def rearrange(P, bs, typ):
    h, w = P.shape
    hb, wb = h // bs, w // bs
    core = P[:hb * bs, :wb * bs].reshape(hb, bs, wb, bs)       # (block row, in-row, block col, in-col)
    within_col = typ in (2, 3)          # 10, 11: column-by-column within blocks
    across_col = typ in (1, 3)          # 01, 11: column-by-column across blocks
    blk = core.transpose(0, 2, 1, 3)    # (br, bc, ir, ic)
    if within_col:
        blk = blk.transpose(0, 1, 3, 2)
    if across_col:
        blk = blk.transpose(1, 0, 2, 3)
    s = blk.reshape(-1)
    rest = np.concatenate([P[hb * bs:, :].reshape(-1), P[:hb * bs, wb * bs:].reshape(-1)])
    return np.concatenate([s, rest]).astype(np.uint8)


@njit(cache=True)
def bsc_len(s, Lfix):
    n = s.size
    # run length remaining from each position
    rem = np.empty(n, np.int64)
    rem[n - 1] = 1
    for i in range(n - 2, -1, -1):
        rem[i] = rem[i + 1] + 1 if s[i] == s[i + 1] else 1
    i = 0; tot = 0
    while i < n:
        L = rem[i]
        if L >= 4:
            l = 0
            while (1 << (l + 1)) <= L:
                l += 1
            tot += l + l + 1          # prefix l bits ((l-1) ones + one zero), mid l bits, tail 1
            i += L
        else:
            take = Lfix if i + Lfix <= n else n - i
            tot += 1 + take
            i += take
    return tot


def plane_len(P, bs, Lfix):
    return min(bsc_len(rearrange(P, bs, t), Lfix) for t in range(4)) + 3   # 1 flag + 2-bit type


def compress(I, bs=None, Lfix=None, border='reference', bss=(2, 3, 4, 5, 6, 8), lfs=range(1, 8)):
    """Room in bits: for the run of compressible planes from 7 down, sum (n - compressed length)."""
    e, over = med_errors(I, border)
    Pk = planes(e)
    n = I.size
    best = None
    for b in ([bs] if bs else bss):
        for lf in ([Lfix] if Lfix else lfs):
            lens, room = {}, 0
            for k in range(7, 0, -1):
                c = plane_len(Pk[k], b, lf)
                if c >= n:
                    break
                lens[k] = c; room += n - c
            if best is None or room > best[0]:
                best = (room, b, lf, lens, int(over.sum()))
    return best


# published values, arXiv:1911.01699 Tables I and II
PUBLISHED = {
    'Airplane': dict(lens={7: 5034, 6: 17682, 5: 28408, 4: 71013, 3: 182697}, room=1005781, net=3.8363),
    'Baboon':   dict(lens={7: 39985, 6: 143101, 5: 240610}, room=362673, net=1.3832),
    'Man':      dict(lens={7: 17795, 6: 90490, 5: 346108, 4: 974860}, room=2764959, net=2.6345),
}

In [ ]:
%%writefile /content/se_rdhei/extra_experiments.py
#!/usr/bin/env python3
"""Additional measurements for the revised manuscript, written to extra.json.

  1. block-minimum efficiency against a per-coefficient oracle
  2. leakage-reduced slot descriptor: room, t-map length, exactness, and the
     spatial structure of the t-map the data hider sees
  3. ciphertext statistics under many independent keys and nonces

Runs on any 8-bit greyscale cover. verified_covers.json lists the covers
whose public copies reproduce the recorded run bit-exactly; pass other paths
on the command line to extend the measurement to full databases.
"""
import sys, json, os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
from PIL import Image
from scipy.stats import chi2 as C2
import serdhei as S

HERE = os.path.dirname(os.path.abspath(__file__))
covers = json.load(open(os.path.join(HERE, 'verified_covers.json')))
for p in sys.argv[1:]:
    covers[os.path.splitext(os.path.basename(p))[0]] = p
ORDER = ['Airplane', 'Baboon', 'Jetplane', 'Lena', 'Man', 'Tiffany', 'barbara',
         'boat', 'goldhill', 'lake', 'livingroom', 'peppers']
NAMES = sorted(covers, key=lambda k: ORDER.index(k) if k in ORDER else 99)


def H0(x):
    p = np.bincount(x.ravel()) / x.size
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def adjc(T):
    T = T.astype(float)
    a = np.concatenate([T[:, :-1].ravel(), T[:-1, :].ravel()])
    b = np.concatenate([T[:, 1:].ravel(), T[1:, :].ravel()])
    return float(np.corrcoef(a, b)[0, 1])


out = {'covers': NAMES, 'eff': {}, 'lr': {}, 'multikey': {}}
Kown = S.keys(b'\x00' * 32, b'\x01' * 12)
K_hide, nonce = bytes(range(32)), b'\x01' * 12

for nm in NAMES:
    I = np.array(Image.open(covers[nm]).convert('L')).astype(np.int64)
    MN = I.size
    L = S.max_levels(I.shape, 4)
    bands = S.iwt2_multi(I, L)

    # ---- 1. efficiency
    st = om = lab = 0.0; omB = {2: 0.0, 4: 0.0, 8: 0.0}
    for b in S.band_names(L):
        if b == 'LL':
            continue
        v = bands[b]; W = S.band_width(v); u = v % (2 ** W)
        t = np.minimum(S.lead_ident(u, W) - 1, W - 2)
        st += t.sum(); lab += H0(t) * t.size
        om += S.block_t(u, W, W - 2, S.BLOCK).sum() * S.BLOCK ** 2
        for bb in omB:
            omB[bb] += S.block_t(u, W, W - 2, bb).sum() * bb * bb
    out['eff'][nm] = dict(sum_t=st / MN, room=om / MN, eta=om / st,
                          oracle_net=(st - lab) / MN, block=S.BLOCK,
                          eta_by_B={str(bb): v / st for bb, v in omB.items()})

    # ---- 2. leakage-reduced descriptor
    ct, L2, names = S.owner_encrypt_lr(I, Kown)
    tot = sum(int(ct[n]['Tperm'].sum()) * S.BLOCK ** 2 for n in names)
    pay = np.random.default_rng(3).integers(0, 2, tot - 1000).astype(np.uint8)
    S.hider_embed_lr(ct, names, pay, K_hide, nonce)
    got = S.receiver_extract_lr(ct, names, pay.size, K_hide, nonce)
    R = S.receiver_recover_lr(ct, names, L2, Kown)
    same_len = all(S.huff_len(ct[n]['Tperm']) == S.huff_len(np.sort(ct[n]['Tperm']))
                   for n in names if n != 'LL' and ct[n]['Tperm'].size)
    sp, pm = [], []
    for b in ['HL1', 'LH1', 'HH1']:
        v = bands[b]; W = S.band_width(v)
        T = S.block_t(v % (2 ** W), W, W - 2, S.BLOCK)
        sp.append(adjc(T)); pm.append(adjc(ct[b]['Tperm'].reshape(T.shape)))
    out['lr'][nm] = dict(room=tot / MN, payload_exact=bool(np.array_equal(got, pay)),
                         cover_exact=bool(np.array_equal(R, I)), tmap_len_same=bool(same_len),
                         rho_spatial=sp, rho_permuted=pm)

    # ---- 3. many keys
    rng = np.random.default_rng(1)
    Hc, P, Rh = [], [], []
    for k in range(20):
        r = S.run(covers[nm], km=rng.bytes(32), nonce=rng.bytes(12))
        Hc.append(r['Hc']); P.append(float(1 - C2.cdf(r['chi'], 255)))
        Rh.append(max(abs(x) for x in r['cc']))
    out['multikey'][nm] = dict(Hc=Hc, p=P, rho=Rh)

json.dump(out, open(os.path.join(HERE, 'extra.json'), 'w'), indent=1)
for nm in NAMES:
    e, l, m = out['eff'][nm], out['lr'][nm], out['multikey'][nm]
    print('%-11s eta %.3f oracle-net %.4f | LR room %.4f exact %s/%s len %s rho %s->%s | '
          'keys Hc %.4f-%.4f rej %d/20 max|rho| %.4f'
          % (nm, e['eta'], e['oracle_net'], l['room'], l['payload_exact'], l['cover_exact'],
             l['tmap_len_same'], [round(x, 3) for x in l['rho_spatial']],
             [round(x, 3) for x in l['rho_permuted']], min(m['Hc']), max(m['Hc']),
             sum(p < 0.05 for p in m['p']), max(m['rho'])))

In [ ]:
%%writefile /content/se_rdhei/spec_experiments.py
#!/usr/bin/env python3
"""Checks of the SPECIFIED construction (Section 4.6, Algorithm 3), written to spec.json.

 1. role-separated round trip on every verified cover: room, payload, cover
 2. integrity: what each kind of tampering does to tau1, tau2 and recovery,
    including a malicious data hider that writes arbitrary reserved bits
 3. ciphertext statistics under AES-256-CTR, 20 keys x every verified cover
"""
import sys, os, json, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
from PIL import Image
from scipy.stats import chi2 as C2
import serdhei as S

HERE = os.path.dirname(os.path.abspath(__file__))
V = json.load(open(os.path.join(HERE, 'verified_covers.json')))
D = json.load(open(os.path.join(HERE, 'data.json')))
ORDER = ['Airplane', 'Baboon', 'Jetplane', 'Lena', 'Man', 'Tiffany', 'barbara',
         'boat', 'goldhill', 'lake', 'livingroom', 'peppers']
names = sorted(V, key=lambda k: ORDER.index(k) if k in ORDER else 99)
ok = S.selftest(); assert all(ok), ok
out = dict(selftest=dict(hkdf_rfc5869=ok[0], aes_sp800_38a=ok[1]), covers=names,
           roundtrip={}, tamper={}, stats={})
rng = np.random.default_rng(2026)

def img(nm): return np.array(Image.open(os.path.join(HERE, V[nm]) if not os.path.isabs(V[nm]) else V[nm]).convert('L')).astype(np.int64)

for nm in names:
    I = img(nm); K = rng.bytes(32); KH = rng.bytes(32); N = rng.bytes(12)
    t0 = time.time()
    C, H0, t1 = S.spec_owner(I, K, N)
    kt = S.owner_keys(K, N)['tmap']
    T = S._decrypt_tmap(H0, kt)
    room = sum(int(T[b].sum()) for b in names and H0['names']) * H0['B'] ** 2
    Dp = rng.integers(0, 2, room).astype(np.uint8)                      # fill the room
    Cm, H1, t2 = S.spec_hider(C, H0, t1, kt, KH, Dp)
    got = S.spec_extract(Cm, H1, t1, t2, kt, KH)
    R = S.spec_recover(Cm, H1, t1, K)
    out['roundtrip'][nm] = dict(room_bpp=room / I.size, recorded_raw=D['cap'][nm][1],
                                payload_exact=bool(got is not None and np.array_equal(got, Dp)),
                                cover_exact=bool(R is not None and np.array_equal(R, I)),
                                seconds=round(time.time() - t0, 1))
    # ---- tampering, one change at a time, on a fresh copy each time
    H = dict(H0); H['Tplain'] = T
    def tamper(kind):
        Cx = {b: v.copy() for b, v in Cm.items()}; Hx = dict(H1); Hx['W'] = dict(H1['W']); Hx['Tcode'] = H1['Tcode']
        t2x = t2
        if kind == 'hider_rewrites_reserved':        # malicious hider: arbitrary payload, valid tau2
            bits = rng.integers(0, 2, room).astype(np.uint8)
            Cx = S._write_reserved(Cm, H, bits)
            t2x = S.hmac256(S.hider_keys(KH)['tag'], S._ser_header(Hx) + S._ser_all(Cx, Hx) + t1)
        elif kind == 'outsider_flips_reserved_bit':
            b = 'HL1'; tf = S._tfield(H, b, Cx[b].size); i = int(np.nonzero(tf)[0][0])
            Cx[b][i] ^= 1 << (H['W'][b] - 1)
        elif kind == 'outsider_flips_nonreserved_bit':
            b = 'HL1'; Cx[b][0] ^= 1
        elif kind == 'header_tmap_byte':
            e = bytearray(Hx['Tcode']); e[len(e) // 2] ^= 1; Hx['Tcode'] = bytes(e)
        elif kind == 'header_payload_length':
            Hx['nD'] = Hx['nD'] - 8
        p = S.spec_extract(Cx, Hx, t1, t2x, kt, KH)
        r = S.spec_recover(Cx, Hx, t1, K)
        return dict(tau2_accepts=p is not None, tau1_accepts=r is not None,
                    cover_exact_if_accepted=(None if r is None else bool(np.array_equal(r, I))))
    out['tamper'][nm] = {k: tamper(k) for k in ['hider_rewrites_reserved', 'outsider_flips_reserved_bit',
                         'outsider_flips_nonreserved_bit', 'header_tmap_byte', 'header_payload_length']}
    # ---- statistics under AES-256-CTR, 20 independent keys and nonces
    Hc, P, Rh = [], [], []
    for _ in range(20):
        Cs, Hs, _t = S.spec_owner(I, rng.bytes(32), rng.bytes(12))
        E = S.packbits_cipher(Cs, Hs['W'], Hs['names'])
        s = min(I.shape[0], int(np.sqrt(E.size)))
        Hc.append(S.entropy8(E)); P.append(float(1 - C2.cdf(S.chi2stat(E), 255)))
        Rh.append(max(abs(x) for x in S.corr3(E[:s * s].reshape(s, s))))
    out['stats'][nm] = dict(Hc=Hc, p=P, rho=Rh)
    r = out['roundtrip'][nm]
    print('%-11s room %.4f/%.4f payload %s cover %s %5.1fs | tamper %s | Hc %.4f-%.4f rej %d/20 max|rho| %.4f' % (
        nm, r['room_bpp'], r['recorded_raw'], r['payload_exact'], r['cover_exact'], r['seconds'],
        {k: (v['tau2_accepts'], v['tau1_accepts'], v['cover_exact_if_accepted']) for k, v in out['tamper'][nm].items()}
        if nm == names[0] else '...', min(Hc), max(Hc), sum(p < 0.05 for p in P), max(Rh)))
json.dump(out, open(os.path.join(HERE, 'spec.json'), 'w'), indent=1)

In [ ]:
%%writefile /content/se_rdhei/matched_comparison.py
#!/usr/bin/env python3
"""Matched-image comparison: the proposed scheme and the reimplemented
bit-plane-compression method of Yin et al. [arXiv:1911.01699] on identical
covers, written to matched.json. Pass extra image paths to extend it, e.g. the
100-image database samples used in Section 6.1."""
import sys, os, json, math
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
from PIL import Image
import bpc

HERE = os.path.dirname(os.path.abspath(__file__))
V = json.load(open(os.path.join(HERE, 'verified_covers.json')))
D = json.load(open(os.path.join(HERE, 'data.json')))
for p in sys.argv[1:]:
    V[os.path.splitext(os.path.basename(p))[0]] = p
ORDER = ['Airplane', 'Baboon', 'Jetplane', 'Lena', 'Man', 'Tiffany', 'barbara',
         'boat', 'goldhill', 'lake', 'livingroom', 'peppers']
names = sorted(V, key=lambda k: ORDER.index(k) if k in ORDER else 99)
out = {'validation': {}, 'rows': {}}
for nm in names:
    p = V[nm] if os.path.isabs(V[nm]) else os.path.join(HERE, V[nm])
    I = np.array(Image.open(p).convert('L')).astype(np.int64)
    room, bs, lf, lens, ov = bpc.compress(I, border='reference')
    aux = 18 + math.ceil(math.log2(I.size)) * ov + 4 + 3 + 3      # count, locations, bs, Lfix, s
    bpc_net = (room - aux) / I.size
    row = dict(bpc_net=bpc_net, bpc_room=room, overflow=ov, bs=bs, Lfix=lf)
    if nm in D['cap']:
        row.update(prop_net=D['cap'][nm][3], prop_eff=D['cap'][nm][5])
    out['rows'][nm] = row
    if nm in bpc.PUBLISHED:
        pub = bpc.PUBLISHED[nm]
        out['validation'][nm] = dict(published_net=pub['net'], reimpl_net=bpc_net,
                                     published_room=pub['room'], reimpl_room=room,
                                     room_dev_pct=100 * (room - pub['room']) / pub['room'])
json.dump(out, open(os.path.join(HERE, 'matched.json'), 'w'), indent=1)
print('validation:')
for nm, v in out['validation'].items():
    print('  %-9s published %.4f  reimplemented %.4f  room %+.2f%%' % (nm, v['published_net'], v['reimpl_net'], v['room_dev_pct']))
print('%-11s %8s %8s %8s' % ('cover', 'BPC', 'R_net', 'R_eff'))
for nm, r in out['rows'].items():
    print('%-11s %8.4f %8.4f %8.4f' % (nm, r['bpc_net'], r.get('prop_net', float('nan')), r.get('prop_eff', float('nan'))))
rs = [r for r in out['rows'].values() if 'prop_net' in r]
print('mean       %8.4f %8.4f %8.4f' % tuple(np.mean([r[k] for r in rs]) for k in ['bpc_net', 'prop_net', 'prop_eff']))

In [ ]:
%%writefile /content/se_rdhei/mkfigs.py
"""Generate the measured-data figures for the CAEE manuscript.
Every value is read from data.json, which was parsed from the B=4 run."""
import json, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

D = json.load(open('data.json'))

# ---- house style -----------------------------------------------------------
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['TeX Gyre Termes', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8.5,
    'axes.labelsize': 8.5,
    'axes.titlesize': 9,
    'xtick.labelsize': 7.8,
    'ytick.labelsize': 7.8,
    'legend.fontsize': 7.6,
    'axes.linewidth': 0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.linewidth': 0.4,
    'grid.alpha': 0.35,
    'grid.color': '#999999',
    'legend.frameon': False,
    'figure.dpi': 200,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.02,
})
# Okabe-Ito, colour-blind safe
C = {'k':'#000000','orange':'#E69F00','sky':'#56B4E9','green':'#009E73',
     'yellow':'#F0E442','blue':'#0072B2','verm':'#D55E00','purple':'#CC79A7'}
GREY = '#BBBBBB'

ORDER = ['Airplane','Baboon','Jetplane','Lena','Man','Tiffany',
          'barbara','boat','goldhill','lake','livingroom','peppers']
COVERS = [c for c in ORDER if c in D['cap']] + sorted(c for c in D['cap'] if c not in ORDER)
DS = [d for d in ['USC-SIPI','BOSSbase','BOWS-2','UCID'] if d in D['ds']]
REF = next((c for c in COVERS if D.get('size', {}).get(c) == [512, 512]), COVERS[0])
import os as _os
_NPZ = set(np.load('secdata.npz').files) if _os.path.exists('secdata.npz') else set()
def _three(pref):
    have = [c for c in pref if c + '_cover' in _NPZ]
    rest = [c for c in COVERS if c + '_cover' in _NPZ and c not in have]
    return (have + rest)[:3]

# ===========================================================================
# Fig: comparison with published methods
# ===========================================================================
def fig_sota():
    meth = [   # published database means, in Table 6 order (names, not citation numbers)
        ('Zhang 2012',       [None,   None,   0.033 ]),
        ('Puteaux 2018',     [None,   None,   0.8930]),
        ('Yi 2019',          [1.9656, 1.8808, 1.5860]),
        ('Wu 2020',          [2.5613, 2.5194, 2.2751]),
        ('Yin 2020',         [None,   None,   2.6880]),
        ('Yin BPC 2022',     [3.6248, 3.4948, 2.8990]),
        ('Ankur 2023',       [3.8977, 3.7863, None  ]),
        ('Xiao 2024',        [4.041,  3.929,  3.181 ]),
        ('Dong 2025',        [2.9652, 2.7615, None  ]),
        ('Ren 2025',         [3.177,  3.098,  2.722 ]),
        ('Ankur 2025',       [3.9036, 3.8431, None  ]),
        ('Pham Van 2025',    [3.0163, 3.1643, 2.4741]),
        ('Ren 2026',         [2.4938, 2.3508, None  ]),
    ]
    prop_net = [D['ds'][d]['net'] if d in D['ds'] else np.nan for d in ['BOSSbase','BOWS-2','UCID']]
    prop_eff = [D['ds'][d]['eff'] if d in D['ds'] else np.nan for d in ['BOSSbase','BOWS-2','UCID']]
    dbs = ['BOSSbase','BOWS-2','UCID']
    fig, axs = plt.subplots(1, 3, figsize=(6.6, 2.75), sharex=True)
    for k, (ax, db) in enumerate(zip(axs, dbs)):
        rows = [(nm, v[k]) for nm, v in meth if v[k] is not None]
        rows += [(r'Proposed $R_{\mathrm{net}}$', prop_net[k]), (r'Proposed $R_{\mathrm{eff}}$', prop_eff[k])]
        rows.sort(key=lambda r: r[1])
        y = np.arange(len(rows))
        col = [C['blue'] if 'net' in nm else C['orange'] if 'eff' in nm else '#BBBBBB' for nm, _ in rows]
        ax.barh(y, [r[1] for r in rows], 0.72, color=col, edgecolor='#444444', linewidth=0.4)
        ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows], fontsize=6.4)
        ax.set_title(db, fontsize=8)
        ax.set_xlim(0, 4.4); ax.xaxis.set_major_locator(MultipleLocator(1.0))
        ax.tick_params(axis='x', labelsize=6.8)
        ax.set_xlabel('rate (bpp)', fontsize=7)
        ax.grid(axis='y', visible=False)
    fig.tight_layout(w_pad=0.6)
    fig.savefig('fig_sota.pdf'); plt.close(fig)
    print('  fig_sota.pdf')

# ===========================================================================
# Fig: capacity breakdown, per cover and per dataset
# ===========================================================================
def fig_capacity():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.8, 2.25),
                                 gridspec_kw={'width_ratios':[2.1,1],'wspace':0.42})
    raw = [D['cap'][c][1] for c in COVERS]
    tm  = [D['cap'][c][2] for c in COVERS]
    net = [D['cap'][c][3] for c in COVERS]
    eff = [D['cap'][c][5] for c in COVERS]
    x = np.arange(len(COVERS)); w = 0.27
    a1.bar(x-w, raw, w, label=r'$R_{\mathrm{raw}}$', color=GREY,
           edgecolor='#555555', linewidth=0.4)
    a1.bar(x,   net, w, label=r'$R_{\mathrm{net}}$', color=C['blue'],
           edgecolor='black', linewidth=0.4)
    a1.bar(x+w, eff, w, label=r'$R_{\mathrm{eff}}$', color=C['orange'],
           edgecolor='black', linewidth=0.4)
    a1.set_xticks(x)
    a1.set_xticklabels(COVERS, rotation=38, ha='right')
    a1.set_ylabel('rate (bpp)')
    a1.set_ylim(0, 5.0)
    a1.legend(ncol=3, loc='upper right', fontsize=7.4,
              columnspacing=1.0, handlelength=1.1)
    a1.set_title('(a) per cover, USC-SIPI', fontsize=8.5, pad=4)

    mn  = [D['ds'][d]['net'] for d in DS]
    msd = [D['ds'][d]['nsd'] for d in DS]
    me  = [D['ds'][d]['eff'] for d in DS]
    esd = [D['ds'][d]['esd'] for d in DS]
    y = np.arange(len(DS))
    a2.errorbar(mn, y+0.12, xerr=msd, fmt='o', ms=4, color=C['blue'],
                ecolor=C['blue'], elinewidth=1.0, capsize=2.5,
                label=None)
    a2.errorbar(me, y-0.12, xerr=esd, fmt='s', ms=4, color=C['orange'],
                ecolor=C['orange'], elinewidth=1.0, capsize=2.5,
                label=r'$R_{\mathrm{eff}}$')
    a2.set_yticks(y); a2.set_yticklabels(DS)
    a2.invert_yaxis()
    a2.set_xlabel('rate (bpp)')
    a2.set_xlim(1.4, 4.8)
    a2.set_title(r'(b) dataset mean $\pm$ s.d.', fontsize=8.5, pad=4)
    fig.savefig('fig_capacity.pdf'); plt.close(fig)
    print('  fig_capacity.pdf')

# ===========================================================================
# Fig: parameter sweeps
# ===========================================================================
def fig_sweeps():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.10))
    fig.subplots_adjust(wspace=0.28, bottom=0.27)
    B = [2,4,8,16,32][-len(D['blk'][COVERS[0]]):]    # 2-32 in the final run, 4-32 in earlier ones
    sel = [c for c in ['Jetplane','Tiffany','Man','peppers','Lena','Baboon'] if c in COVERS] or COVERS[:6]
    mk = ['o','s','^','D','v','P']
    cols = [C['blue'],C['orange'],C['sky'],C['green'],C['purple'],C['verm']]
    for i,c in enumerate(sel):
        a1.plot(B, D['blk'][c], marker=mk[i], ms=3.6, lw=1.0,
                color=cols[i], label=c)
    allb = np.array([D['blk'][c] for c in COVERS])
    a1.plot(B, allb.mean(0), 'k--', lw=1.2, label='mean (12)')
    a1.set_xscale('log', base=2); a1.set_xticks(B)
    a1.set_xticklabels([str(b) for b in B])
    a1.set_xlabel(r'block size $B$'); a1.set_ylabel(r'$R_{\mathrm{net}}$ (bpp)')

    a1.set_title(r'(a) block size', fontsize=8.5)

    L = [1,2,3,4]
    for i,c in enumerate(sel):
        a2.plot(L, D['lvl'][c], marker=mk[i], ms=3.6, lw=1.0,
                color=cols[i], label=c)
    alll = np.array([D['lvl'][c] for c in COVERS])
    a2.plot(L, alll.mean(0), 'k--', lw=1.2, label='mean (12)')
    a2.set_xticks(L)
    a2.set_xlabel(r'decomposition level $L$')
    a2.set_ylabel(r'$R_{\mathrm{eff}}$ (bpp)')
    a2.set_title(r'(b) decomposition depth', fontsize=8.5)
    h,l = a1.get_legend_handles_labels()
    fig.legend(h, l, ncol=7, fontsize=7.2, loc='lower center',
               bbox_to_anchor=(0.5,-0.12), handlelength=1.4,
               columnspacing=1.1, frameon=False)
    fig.savefig('fig_sweeps.pdf'); plt.close(fig)
    print('  fig_sweeps.pdf')

# ===========================================================================
# Fig: ciphertext statistics
# ===========================================================================
def fig_stats():
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(6.8, 2.15))
    fig.subplots_adjust(wspace=0.46)
    Hp = [D['stats'][c]['Hp'] for c in COVERS]
    Hc = [D['stats'][c]['Hc'] for c in COVERS]
    y = np.arange(len(COVERS))
    a1.barh(y, Hp, 0.62, color=GREY, edgecolor='#555555', linewidth=0.4,
            label='cover')
    a1.plot(Hc, y, 'o', ms=3.4, color=C['verm'], label='ciphertext')
    a1.axvline(8, color='k', ls=':', lw=0.9)
    a1.set_yticks(y); a1.set_yticklabels(COVERS, fontsize=6.8)
    a1.invert_yaxis(); a1.set_xlim(3.5, 8.45)
    a1.set_xlabel('entropy (bits/byte)')
    a1.set_title('(a) entropy: bars cover, dots cipher', fontsize=8.0)

    pl = np.array([D['stats'][c]['plain'] for c in COVERS])
    ci = np.array([D['stats'][c]['ciph']  for c in COVERS])
    lbl = ['H','V','D']
    for j in range(3):
        a2.plot(np.abs(pl[:,j]), np.full(len(COVERS), 1.0+0.12*(j-1)), 'o', ms=3.2,
                color=GREY, markeredgecolor='#555555', markeredgewidth=0.3)
        a2.plot(np.abs(ci[:,j]), np.full(len(COVERS), 0.0+0.12*(j-1)), 'o', ms=3.2,
                color=[C['blue'],C['green'],C['verm']][j], label=lbl[j])
    a2.set_yticks([0,1]); a2.set_yticklabels(['cipher','cover'])
    a2.set_xlim(-0.05, 1.05); a2.set_ylim(-0.5, 1.6)
    a2.set_xlabel(r'$|\rho|$, adjacent samples')
    a2.legend(ncol=3, fontsize=6.9, loc='upper center',
              bbox_to_anchor=(0.5,1.03), handlelength=0.8, columnspacing=0.8)
    a2.set_title('(b) correlation', fontsize=8.5)

    pv = sorted(D['stats'][c]['p'] for c in COVERS)
    a3.plot(pv, np.arange(1,len(COVERS)+1)/float(len(COVERS)), 'o-', ms=3.4, lw=1.0, color=C['blue'],
            label='observed')
    a3.plot([0,1],[0,1],'k--',lw=0.9, label='Uniform(0,1)')
    a3.axvline(0.05, color=C['verm'], ls=':', lw=1.0)
    a3.text(0.075, 0.90, r'$\alpha=0.05$', fontsize=6.8, color=C['verm'])
    a3.set_xlim(0,1); a3.set_ylim(0,1.02)
    a3.set_xlabel(r'$\chi^2$ $p$-value'); a3.set_ylabel('empirical CDF')
    a3.legend(loc='lower right', fontsize=6.9, handlelength=1.3)
    a3.set_title('(c) uniformity', fontsize=8.5)
    fig.savefig('fig_stats.pdf'); plt.close(fig)
    print('  fig_stats.pdf')

# ===========================================================================
# Fig: ablation
# ===========================================================================
def fig_ablation():
    # legends sit above the axes so they cannot collide with the bars, and
    # the ratio statement lives in the body text, not on the figure
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(6.6, 2.45))
    ovf = [D['abl'][c][0] for c in COVERS]
    x = np.arange(len(COVERS)); w = 0.62
    a1.bar(x, ovf, w, color=C['verm'], edgecolor='black', linewidth=0.4,
           label='classic ordering')
    a1.plot(x, [1.35]*len(COVERS), 'o', ms=4.2, color=C['blue'],
            markeredgecolor='black', markeredgewidth=0.4,
            label='deferred synthesis (exactly 0)', clip_on=False, zorder=5)
    a1.set_yscale('log'); a1.set_ylim(1.0, 3e6)
    a1.set_xticks(x); a1.set_xticklabels(COVERS, rotation=38, ha='right')
    a1.set_ylabel('pixels outside $[0,255]$')
    a1.set_xlabel('(a) inverse-transform overflow', fontsize=8.5, labelpad=2)
    a1.legend(fontsize=7.0, handlelength=1.1, ncol=1,
              loc='lower left', bbox_to_anchor=(0.0, 1.02), borderaxespad=0)

    rm_mod = [D['abl'][c][2] for c in COVERS]
    rm_our = [D['abl'][c][3] for c in COVERS]
    a2.bar(x-0.175, rm_mod, 0.34, color=C['verm'],
           edgecolor='black', linewidth=0.4, label=r'modular $\mathbb{Z}_{256}$')
    a2.bar(x+0.175, rm_our, 0.34, color=C['blue'], edgecolor='black',
           linewidth=0.4, label='proposed')
    a2.set_yscale('log'); a2.set_ylim(1e2, 3e7)
    a2.set_xticks(x); a2.set_xticklabels(COVERS, rotation=38, ha='right')
    a2.set_ylabel('exploitable room (bits)')
    a2.set_xlabel('(b) transform choice', fontsize=8.5, labelpad=2)
    a2.legend(fontsize=7.0, handlelength=1.1, ncol=2,
              loc='lower left', bbox_to_anchor=(0.0, 1.02), borderaxespad=0)
    fig.tight_layout(pad=0.3, w_pad=1.2)
    fig.savefig('fig_ablation.pdf')
    fig.savefig('p_fig_ablation.png', dpi=150)
    plt.close(fig)
    print('  fig_ablation.pdf')

# ===========================================================================
# Fig: key sensitivity
# ===========================================================================
def fig_sensitivity():
    """Each cover against its own Wu-Noonan-Agaian region (T = compared bytes)."""
    fig, ax = plt.subplots(figsize=(3.3, 2.7))
    mu = 257 / 768 * 100
    x = [D['npcr'][c]['n'] - D['npcr'][c]['nstar'] for c in COVERS]
    y = [(D['npcr'][c]['u'] - mu) / ((D['npcr'][c]['uhi'] - D['npcr'][c]['ulo']) / 2) for c in COVERS]
    ax.axhspan(-1, 1, xmin=0, xmax=1, color=C['sky'], alpha=0.16, lw=0)
    ax.axvline(0, color=C['verm'], ls='--', lw=0.9)
    xe = [D['npcr'][c].get('nsd1', D['npcr'][c]['nsd']) for c in COVERS]
    ye = [D['npcr'][c].get('usd1', D['npcr'][c]['usd']) / ((D['npcr'][c]['uhi'] - D['npcr'][c]['ulo']) / 2) for c in COVERS]
    ax.errorbar(x, y, xerr=xe, yerr=ye, fmt='none', ecolor='#666666', elinewidth=0.6, capsize=1.5, zorder=1)
    big = [D.get('size', {}).get(c) != [512, 512] for c in COVERS]
    ax.plot([a for a, b in zip(x, big) if not b], [a for a, b in zip(y, big) if not b], 'o', ms=4.2,
            color=C['blue'], markeredgecolor='black', markeredgewidth=0.4, label=r'$512\times512$')
    ax.plot([a for a, b in zip(x, big) if b], [a for a, b in zip(y, big) if b], 's', ms=4.2,
            color=C['green'], markeredgecolor='black', markeredgewidth=0.4, label=r'\emph{Man}, $1024^2$' if plt.rcParams.get('text.usetex') else r'Man, $1024\times1024$')
    ax.set_xlabel(r'NPCR $-\ N^{*}$ (percentage points)'); ax.set_ylabel('UACI within its interval')
    ax.set_xlim(-0.008, 0.040); ax.set_ylim(-1.35, 1.35)
    ax.text(0.0008, -1.25, r'$N^*$', fontsize=7.2, color=C['verm'])
    ax.text(0.0145, 1.07, 'UACI acceptance region', fontsize=6.8, color='#30627f')
    ax.legend(loc='lower right', fontsize=7.0, handlelength=1.0)
    ax.xaxis.set_major_locator(MultipleLocator(0.01))
    fig.savefig('fig_sensitivity.pdf'); plt.close(fig)
    print('  fig_sensitivity.pdf')

import traceback
for f in (fig_sota, fig_capacity, fig_sweeps, fig_stats, fig_ablation,
          fig_sensitivity):
    try:
        f()
    except Exception:
        print('  ! %s failed:' % f.__name__); traceback.print_exc()
print('done')


# ===========================================================================
# Fig: histograms, cover against ciphertext
# ===========================================================================
def fig_hist():
    Z = np.load('secdata.npz')
    fig, ax = plt.subplots(2, 2, figsize=(6.6, 2.75), sharex=True)
    rows = [(c, c) for c in _three(['Airplane', 'barbara'])[:2]]
    for r, (key, lab) in enumerate(rows):
        cov = Z[key + '_cover'].ravel()
        cip = Z[key + '_ciph'].ravel()
        for c, (dat, ttl, col) in enumerate(
                [(cov, 'cover', GREY), (cip, 'ciphertext', C['blue'])]):
            a = ax[r][c]
            a.hist(dat, bins=256, range=(0, 256), color=col,
                   edgecolor='none', linewidth=0)
            a.set_xlim(0, 255)
            a.grid(axis='y')
            a.set_axisbelow(True)
            if r == 0:
                a.set_title('%s' % ttl, fontsize=8.5, pad=4)
            if c == 0:
                a.set_ylabel('%s\ncount' % lab, fontsize=8.0)
            if r == 1:
                a.set_xlabel('byte value', fontsize=8.5)
    # a flat reference line on each ciphertext panel
    for r, (key, _) in enumerate(rows):
        n = Z[key + '_ciph'].size
        ax[r][1].axhline(n / 256.0, color=C['verm'], lw=0.9, ls='--',
                         zorder=5)
    fig.tight_layout(pad=0.3, h_pad=0.6, w_pad=1.0)
    fig.savefig('fig_hist.pdf')
    fig.savefig('p_fig_hist.png', dpi=150)
    plt.close(fig)
    return 'fig_hist.pdf'


# ===========================================================================
# Fig: adjacent-sample correlation, stacked by image, cover against ciphertext
# ===========================================================================
CORR_IMGS = _three(['Airplane', 'barbara', 'livingroom'])   # image index 1..3
CORR_COLS = ['#5B2A86', '#D6589F', '#6F9BC6']

def fig_corr():
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    Z = np.load('secdata.npz')
    rng = np.random.default_rng(7)
    dirs = [('horizontal', (0, 1)), ('vertical', (1, 0)), ('diagonal', (1, 1))]
    fig = plt.figure(figsize=(7.0, 4.9))
    for r, kind in enumerate(['cover', 'ciph']):
        for c, (dname, (dy, dx)) in enumerate(dirs):
            ax = fig.add_subplot(2, 3, r * 3 + c + 1, projection='3d')
            for k, nm in enumerate(CORR_IMGS):
                M = Z['%s_%s' % (nm, kind)].astype(float)
                a = M[:M.shape[0] - dy, :M.shape[1] - dx].ravel()
                b = M[dy:, dx:].ravel()
                idx = rng.choice(a.size, 2500, replace=False)
                ax.scatter(a[idx], b[idx], np.full(idx.size, k + 1.0), s=0.9,
                           c=CORR_COLS[k], alpha=0.55, linewidths=0,
                           depthshade=False, rasterized=True)
            ax.set_xlim(0, 255); ax.set_ylim(0, 255); ax.set_zlim(0.6, 3.4)
            ax.set_xticks([0, 128, 255]); ax.set_yticks([0, 128, 255])
            ax.set_zticks([1, 2, 3])
            ax.tick_params(labelsize=6.2, pad=-2)
            ax.set_xlabel('sample', fontsize=7, labelpad=-7)
            ax.set_ylabel('adjacent sample', fontsize=7, labelpad=-7)
            ax.set_zlabel('image index', fontsize=7, labelpad=-10)
            ax.view_init(elev=22, azim=-72)
            ax.set_box_aspect((1, 1, 0.82))
            for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
                axis.pane.set_facecolor('white'); axis.pane.set_edgecolor('#BBBBBB')
                axis._axinfo['grid'].update(linewidth=0.3, color='#DDDDDD')
            ax.set_title('%s, %s' % ('cover' if kind == 'cover' else 'ciphertext', dname),
                         fontsize=8.2, pad=-4)
    fig.subplots_adjust(left=0.0, right=0.94, bottom=0.04, top=0.97,
                        wspace=0.13, hspace=0.14)
    fig.savefig('fig_corr.pdf', dpi=300)
    fig.savefig('p_fig_corr.png', dpi=150)
    plt.close(fig)
    return 'fig_corr.pdf'


# ===========================================================================
# Fig: visual results -- cover, encrypted, marked encrypted, recovered
# ===========================================================================
def fig_visual():
    Z = np.load('montage.npz')
    rows = _three(['Airplane', 'barbara', 'livingroom'])
    cols = [('cover', 'cover'), ('enc', 'encrypted'),
            ('mark', 'marked encrypted'), ('rec', 'recovered')]
    fig, ax = plt.subplots(3, 4, figsize=(6.4, 4.95))
    for r, nm in enumerate(rows):
        for c, (key, ttl) in enumerate(cols):
            a = ax[r][c]
            a.imshow(Z['%s_%s' % (nm, key)], cmap='gray', vmin=0, vmax=255,
                     interpolation='nearest')
            a.set_xticks([]); a.set_yticks([]); a.grid(False)
            for sp in a.spines.values():
                sp.set_visible(True); sp.set_linewidth(0.5); sp.set_color('#444444')
            if r == 0:
                a.set_title(ttl, fontsize=8.5, pad=4)
            if c == 0:
                a.set_ylabel(nm, fontsize=8.5)
    fig.subplots_adjust(left=0.04, right=0.995, top=0.95, bottom=0.01,
                        wspace=0.04, hspace=0.04)
    fig.savefig('fig_visual.pdf', dpi=220)
    fig.savefig('p_fig_visual.png', dpi=110)
    plt.close(fig)
    return 'fig_visual.pdf'


if __name__ == '__main__' or True:
    if _NPZ:
        for f in (fig_hist, fig_corr, fig_visual):
            try:
                print(' ', f())
            except Exception:
                print('  ! %s failed:' % f.__name__); traceback.print_exc()

In [ ]:
%%writefile /content/se_rdhei/recorded_matlab.json
{"cap":{"Airplane":[4.0,4.1379,0.142,3.9959,8.355,3.6404],"Baboon":[4.0,2.1608,0.1379,2.0229,8.477,1.5463],"Jetplane":[4.0,3.9614,0.1597,3.8017,8.438,3.3642],"Lena":[4.0,2.976,0.1435,2.8325,7.652,2.8325],"Man":[4.0,4.0799,0.1416,3.9383,9.254,2.6844],"Tiffany":[4.0,4.129,0.1679,3.961,8.762,3.1993],"barbara":[4.0,3.1223,0.156,2.9663,8.148,2.8179],"boat":[4.0,2.8157,0.1407,2.675,8.113,2.5617],"goldhill":[4.0,2.6426,0.1402,2.5024,7.824,2.5024],"lake":[4.0,2.4476,0.1488,2.2989,7.938,2.2989],"livingroom":[4.0,2.9788,0.138,2.8408,8.246,2.5947],"peppers":[4.0,3.1505,0.1365,3.014,8.188,2.8265]},"blk":{"Airplane":[3.9959,3.4414,2.6961,1.8218],"Baboon":[2.0229,1.645,1.223,0.7756],"Jetplane":[3.8017,3.2982,2.6744,1.9584],"Lena":[2.8325,2.3721,1.8399,1.2519],"Man":[3.9383,3.4608,2.9264,2.3082],"Tiffany":[3.961,3.4403,2.7217,1.8685],"barbara":[2.9663,2.5167,1.9656,1.3767],"boat":[2.675,2.1857,1.6666,1.2011],"goldhill":[2.5024,2.0547,1.5624,1.1153],"lake":[2.2989,1.8128,1.2715,0.7366],"livingroom":[2.8408,2.3079,1.7203,1.1231],"peppers":[3.014,2.4847,1.8456,1.1545]},"lvl":{"Airplane":[2.5884,3.4086,3.598,3.6404],"Baboon":[1.0418,1.4349,1.524,1.5463],"Jetplane":[2.5154,3.1389,3.3224,3.3642],"Lena":[2.0576,2.7015,2.8113,2.8325],"Man":[1.7011,2.489,2.667,2.6844],"Tiffany":[2.048,2.9562,3.1584,3.1993],"barbara":[2.0444,2.6626,2.794,2.8179],"boat":[1.5877,2.3634,2.5252,2.5617],"goldhill":[1.9344,2.4064,2.4806,2.5024],"lake":[1.6866,2.1815,2.2817,2.2989],"livingroom":[1.6317,2.3913,2.5721,2.5947],"peppers":[1.9853,2.6129,2.791,2.8265]},"modroom":{"Airplane":448,"Baboon":688,"Jetplane":3824,"Lena":3264,"Man":10896,"Tiffany":1392,"barbara":9616,"boat":1104,"goldhill":3296,"lake":560,"livingroom":8176,"peppers":320},"ourroom":{"Airplane":1084720,"Baboon":566432,"Jetplane":1038464,"Lena":780128,"Man":4278064,"Tiffany":1082384,"barbara":818496,"boat":738112,"goldhill":692752,"lake":641632,"livingroom":780864,"peppers":825888},"tleak":{"Airplane":[2.2079,7.9899],"Baboon":[2.1264,7.9888],"Jetplane":[2.5155,7.9885],"Lena":[2.2315,7.9897],"Man":[2.1835,7.9971],"Tiffany":[2.6565,7.9887],"barbara":[2.4189,7.9885],"boat":[2.1818,7.9894],"goldhill":[2.14,7.989],"lake":[2.3485,7.9902],"livingroom":[2.1171,7.9889],"peppers":[2.121,7.9892]}}

In [ ]:
%%writefile /content/se_rdhei/spec_test_vector.py
"""Reproducible test vector for the specified construction (Section 4.6).

Inputs are fixed and derivable without any image file:
  cover   I[y, x] = (7x + 13y + (x*y mod 31)) mod 256,  64 x 64, uint8
  K       bytes 0x00..0x1f            (owner master key)
  K_H     bytes 0x20..0x3f            (data hider's own key)
  N       bytes 0x40..0x4b            (96-bit nonce)
  D       the first min(4096, |Omega|) bits of SHA-256(b'SE-RDHEI test payload' || <i>_32), i = 0, 1, ...
The optional credentials A1, A2 are disabled (Section 7.6), and the permuted
t-map is carried uncoded, one byte per block, as in the reference code.
Outputs: room, SHA-256 of the serialised ciphertext before and after embedding
and of the owner header, both tags, and the two exactness checks.
Writes spec_test_vector.json next to this file; with --check it compares a
fresh run against the stored vector and exits non-zero on any mismatch.
"""
import hashlib, json, os, sys
import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import serdhei as S


def cover():
    y, x = np.mgrid[0:64, 0:64]
    return ((7 * x + 13 * y + (x * y) % 31) % 256).astype(np.uint8)


def payload(n):
    out, i = b'', 0
    while len(out) * 8 < n:
        out += hashlib.sha256(b'SE-RDHEI test payload' + i.to_bytes(4, 'big')).digest(); i += 1
    return np.unpackbits(np.frombuffer(out, np.uint8))[:n]


def run():
    I = cover()
    K, KH, N = bytes(range(32)), bytes(range(32, 64)), bytes(range(64, 76))
    C, H0, t1 = S.spec_owner(I.astype(np.int64), K, N)
    kt = S.owner_keys(K, N)['tmap']
    T = S._decrypt_tmap(H0, kt)
    room = int(sum(int(T[b].sum()) * H0['B'] ** 2 for b in T))
    D = payload(min(4096, room))
    Cm, H1, t2 = S.spec_hider(C, H0, t1, kt, KH, D)
    got = S.spec_extract(Cm, H1, t1, t2, kt, KH)
    R = S.spec_recover(Cm, H1, t1, K)
    h = lambda b: hashlib.sha256(b).hexdigest()
    return dict(
        cover_sha256=h(I.tobytes()), room_bits=room, payload_bits=int(D.size),
        C_sha256=h(S._ser_all(C, H0)), H0_sha256=h(S._ser_header(H0)), tau1=t1.hex(),
        Cmarked_sha256=h(S._ser_all(Cm, H1)), H_sha256=h(S._ser_header(H1)), tau2=t2.hex(),
        payload_exact=bool(got is not None and np.array_equal(np.asarray(got)[:D.size], D)),
        cover_exact=bool(R is not None and np.array_equal(np.asarray(R).astype(np.uint8), I)))


if __name__ == '__main__':
    v = run()
    p = os.path.join(HERE, 'spec_test_vector.json')
    if '--check' in sys.argv:
        ref = json.load(open(p))
        bad = [k for k in ref if ref[k] != v[k]]
        print('test vector', 'MATCHES' if not bad else 'DIFFERS in ' + ', '.join(bad))
        sys.exit(1 if bad else 0)
    json.dump(v, open(p, 'w'), indent=1)
    print(json.dumps(v, indent=1))

In [ ]:
%%writefile /content/se_rdhei/spec_test_vector.json
{
 "cover_sha256": "eaaf0225d6572007ea92fc8367b38a21e5d9268f31bccdb57bdc38ca5ce260bc",
 "room_bits": 12436,
 "payload_bits": 4096,
 "C_sha256": "97b65021d2e324a8f39b6b1a758f0e676194486579bfd2e2468bc053630d9cd1",
 "H0_sha256": "97634399ce2fa8ae649424b7d36a60b865eea3a412d89cd94a947764fb891159",
 "tau1": "5eff72bfc4dc6795e0bf82c23d15340efa3266b75e68def228f95570a3f841d8",
 "Cmarked_sha256": "6b760c2364cbb42bd8b80d7f5d9bd73a3029f34d2032323872f42a9f587f3107",
 "H_sha256": "22c9755daa743e20465373e12e0c20eec9fd50d300cdac2fc5800a77b88329b1",
 "tau2": "4920ddbbea0ca30aa53a26ddf45034bad51fb237e9835b4bc85a12074730260c",
 "payload_exact": true,
 "cover_exact": true
}

In [ ]:
%%writefile /content/se_rdhei/abcs.py
#!/usr/bin/env python3
"""Reimplementation of the capacity of Xiao, Wang, Shen & Yao (2024),
'High-capacity reversible data hiding in encrypted images based on adaptive
block coding selection', J. Vis. Commun. Image R. 104, 104291 (ABCS).

Only the embedding rate is computed. Iterative encryption is constructed so
that every encrypted block has the same prediction errors (PEs) as the
original block (Sec. 3.1.2), and block permutation only reorders blocks, so
the rate is key independent and can be computed from the plaintext blocks.

Equations follow the paper: Eq. (1) block PEs (first pixel kept, first row
predicted from the left, first column from above, the rest by MED); Eq. (3)
entropy coding with threshold T; Eqs. (5)-(8) zero-valued high bit-plane
compression; Sec. 3.2.3 per-block selection and global auxiliary information.
"""
import heapq, math
import numpy as np


def block_pes(I, t):
    H, W = I.shape
    bh, bw = H // t, W // t
    B = I[:bh * t, :bw * t].astype(np.int64).reshape(bh, t, bw, t).transpose(0, 2, 1, 3).reshape(-1, t, t)
    P = np.zeros_like(B)
    P[:, 0, 1:] = B[:, 0, :-1]                     # first row: left neighbour (a)
    P[:, 1:, 0] = B[:, :-1, 0]                     # first column: upper neighbour (b)
    a, b, c = B[:, 1:, :-1], B[:, :-1, 1:], B[:, :-1, :-1]
    med = np.where(c <= np.minimum(a, b), np.maximum(a, b),
                   np.where(c >= np.maximum(a, b), np.minimum(a, b), a + b - c))
    P[:, 1:, 1:] = med
    E = (B - P).reshape(len(B), -1)[:, 1:]          # drop the reference pixel
    return E                                        # shape (n, t*t-1)


def huff_lengths(counts):
    """Huffman code lengths for the symbols with nonzero count (dict sym->count)."""
    syms = [s for s, c in counts.items() if c > 0]
    if len(syms) == 1:
        return {syms[0]: 1}
    h = [(counts[s], i, [s]) for i, s in enumerate(syms)]
    heapq.heapify(h); L = {s: 0 for s in syms}; k = len(h)
    while len(h) > 1:
        x = heapq.heappop(h); y = heapq.heappop(h)
        for s in x[2] + y[2]:
            L[s] += 1
        heapq.heappush(h, (x[0] + y[0], k, x[2] + y[2])); k += 1
    return L


def ec_entropy(E, T):
    m = E.shape[1]
    sym = np.where(np.abs(E) > T, T + 1, E)          # T+1 stands for the class |E|>T
    vals, cnt = np.unique(sym, return_counts=True)
    L = huff_lengths(dict(zip(vals.tolist(), cnt.tolist())))
    lut = np.zeros(2 * T + 3, np.int64)
    for v, l in L.items():
        lut[v + T] = l
    NT = (np.abs(E) > T).sum(1)
    ec1 = 8 * (m - NT) - lut[sym + T].sum(1)
    table_bits = (2 * T + 2) * math.ceil(math.log2(2 * T + 2)) + sum(L.values())
    return ec1, table_bits


def ec_bitplanes(E, t):
    n, m = E.shape
    w_pos = math.ceil(math.log2(m))
    # |F|max: largest |F| whose description still saves bits on an m-bit plane (Eqs. 6-7)
    Fmax = 0
    for k in range(1, m):
        if m - (1 + max(1, math.ceil(math.log2(k + 1))) + k * w_pos) > 0:
            Fmax = k
    A = np.abs(E)
    planes = np.stack([(A >> (8 - k)) & 1 for k in range(1, 9)], 1)   # (n, 8, m), k=1 is MSB
    ones = planes.sum(2)
    allzero = ones == 0
    N = np.argmin(np.concatenate([allzero, np.zeros((n, 1), bool)], 1), 1)  # leading all-zero planes
    N = np.where(allzero.all(1), 8, N)
    minority = np.minimum(ones, m - ones)
    usable = minority <= Fmax
    k_idx = np.arange(8)[None, :]
    after = k_idx >= N[:, None]
    use = usable & after
    # |F| is written in a fixed field of ceil(log2(|F|max+1)) bits: the worked
    # example of Sec. 3.2.4 writes |F|=1 as '01' and |F|=0 as '00' for |F|max=2,
    # and only a fixed width can be parsed by the receiver.
    w_cnt = max(1, math.ceil(math.log2(Fmax + 1)))
    inf = np.where(use, 1 + w_cnt + minority * w_pos, 0).sum(1)
    lenM = 8 - N
    freed = N + use.sum(1)
    vals, cnt = np.unique(N, return_counts=True)
    LN = huff_lengths(dict(zip(vals.tolist(), cnt.tolist())))
    lN = np.array([LN[v] for v in N.tolist()])
    # The sign sequence L_sign (m bits, Sec. 3.2.2) must be kept for exact
    # recovery; Eq. (8) as printed omits it, and only with it do the published
    # per-image rates reproduce (within 1.4%, see validate.py).
    ec2 = freed * m - lN - lenM - inf - m
    return ec2


def rate(I, t=8, T=None, Trange=range(2, 160)):
    E = block_pes(I, t)
    n = len(E)
    ec2 = ec_bitplanes(E, t)
    best = None
    for TT in ([T] if T is not None else Trange):
        ec1, tbl = ec_entropy(E, TT)
        tot = np.maximum(ec1, ec2).sum() - (n + 8 + tbl + 72)
        if best is None or tot > best[0]:
            best = (tot, TT, int((ec2 > ec1).sum()))
    tot, TT, n2 = best
    return dict(er=tot / I.size, T=TT, coding2_blocks=n2, blocks=n)


if __name__ == '__main__':
    import sys
    from PIL import Image
    for p in sys.argv[1:]:
        I = np.array(Image.open(p).convert('L'))
        print(p, rate(I))


def selftest():
    """Worked example of Sec. 3.2.4 (t = 4): N = 5, |F|max = 2, planes 6 and 8
    usable, plane 7 not; the paper gives EC2 = 89 with l(N) = 3. The sign
    sequence (15 bits) is added back here because Eq. (8) as printed omits it."""
    E = np.zeros((1, 15), np.int64)
    E[0, 14] = 4                      # one '1' in plane 6, 15 positions from the first pixel
    E[0, [0, 2, 4, 6, 8]] = 2         # five '1's in plane 7
    ec2 = ec_bitplanes(E, 4)[0]
    lN_here = 1                       # a single block gives a 1-bit Huffman code for N
    return int(ec2 + lN_here - 3 + 15) == 89

In [ ]:
%%writefile /content/se_rdhei/validate.py
#!/usr/bin/env python3
"""Validate abcs.py against Table 5 of Xiao et al. (2024): ER (bpp) for square block sizes."""
import json, numpy as np, abcs
from PIL import Image
PUB = {  # Table 5 of the paper, square block sizes only
 'Lena':     {6: 3.2579, 7: 3.3166, 8: 3.3564, 9: 3.2569},
 'Baboon':   {6: 1.5386, 7: 1.5819, 8: 1.6552, 9: 1.5570},
 'Airplane': {6: 3.5717, 7: 3.6338, 8: 3.6758, 9: 3.5581},
 'Man':      {6: 2.8720, 7: 2.9231, 8: 2.9585, 9: 2.9292},
}
# Their 'Airplane' is USC-SIPI 4.2.05 (F-16, entropy 6.7025 in their Table 3), our 'Jetplane'.
FILES = {'Lena': '/home/claude/imgs/lena_gray_512.tif',
         'Baboon': '/home/claude/sipi/gray/4.2.03_matlab_rgb2gray.png',
         'Airplane': '/home/claude/sipi/gray/4.2.05_matlab_rgb2gray.png',
         'Man': '/home/claude/sipi/gray/5.3.01_gray.png'}
out = {}
for nm, p in FILES.items():
    I = np.array(Image.open(p).convert('L'))
    out[nm] = {}
    for t, pub in PUB[nm].items():
        r = abcs.rate(I, t)
        out[nm][t] = dict(ours=round(r['er'], 4), published=pub, rel=round(r['er'] / pub - 1, 4), T=r['T'])
        print('%-9s t=%d ours %.4f published %.4f  %+.2f%%  T=%d' % (nm, t, r['er'], pub, 100 * (r['er'] / pub - 1), r['T']), flush=True)
json.dump(out, open('validation.json', 'w'), indent=1)

In [ ]:
%%writefile /content/se_rdhei/abcs_roundtrip.py
#!/usr/bin/env python3
"""Decodability check for abcs.py: every block is actually encoded to a
bitstring with each of the two codings, decoded back, and compared; the
bitstring length must equal the bits that abcs.py says are NOT freed, i.e.
8(t*t-1) - EC. This shows the rate counts a format a receiver can parse.
Shared side information (Huffman tables, T, |F|max, N codes) is assumed known
to the decoder, as it is carried once in the auxiliary information."""
import math, sys, numpy as np
import abcs


def canon(L):
    """canonical Huffman codewords from {sym: length}"""
    code, out, prev = 0, {}, 0
    for l, s in sorted((l, s) for s, l in L.items()):
        code <<= (l - prev); out[s] = format(code, '0%db' % l); code += 1; prev = l
    return out


def enc1(e, T, cw):
    bits = ''
    for v in e:
        if abs(v) > T:
            bits += cw[T + 1]
        else:
            bits += cw[int(v)]
    raw = ''.join(format(int(v) & 0xFF, '08b') for v in e if abs(v) > T)   # stands for the 8-bit original pixel
    return bits + raw


def dec1(bits, m, T, cw, e_true):
    inv = {c: s for s, c in cw.items()}; out = []; i = 0
    for _ in range(m):
        j = i + 1
        while bits[i:j] not in inv: j += 1
        out.append(inv[bits[i:j]]); i = j
    nout = sum(1 for s in out if s == T + 1)
    # outliers: their original pixels are stored verbatim (8 bits each), from which the
    # receiver recomputes the PE; here we check the count and restore from e_true
    assert len(bits) - i == 8 * nout
    return [e_true[k] if s == T + 1 else s for k, s in enumerate(out)]


def enc2(e, t, Fmax, Ncode):
    m = len(e); wp = math.ceil(math.log2(m)); wc = max(1, math.ceil(math.log2(Fmax + 1)))
    A = np.abs(e); sign = ''.join('0' if v < 0 else '1' for v in e)
    planes = [((A >> (8 - k)) & 1) for k in range(1, 9)]
    N = next((k for k in range(8) if planes[k].any()), 8)
    M = ''; inf = ''; rawp = ''
    for k in range(N, 8):
        p = planes[k]; ones = int(p.sum()); Fb = 1 if ones <= m - ones else 0
        cnt = ones if Fb == 1 else m - ones
        if cnt <= Fmax:
            M += '1'; pos = np.flatnonzero(p == Fb); prev = -1; P = ''
            for q in pos:
                P += format(int(q - prev - 1), '0%db' % wp); prev = q
            inf += str(Fb) + format(cnt, '0%db' % wc) + P
        else:
            M += '0'; rawp += ''.join(map(str, p))
    return Ncode[N] + M + inf + sign + rawp, N


def dec2(bits, m, Fmax, Ncode):
    wp = math.ceil(math.log2(m)); wc = max(1, math.ceil(math.log2(Fmax + 1)))
    inv = {c: s for s, c in Ncode.items()}; j = 1
    while bits[:j] not in inv: j += 1
    N = inv[bits[:j]]; i = j
    M = bits[i:i + 8 - N]; i += 8 - N
    planes = [np.zeros(m, np.int64) for _ in range(8)]
    for idx, k in enumerate(range(N, 8)):
        if M[idx] == '1':
            Fb = int(bits[i]); cnt = int(bits[i + 1:i + 1 + wc], 2); i += 1 + wc
            p = np.full(m, 1 - Fb); prev = -1
            for _ in range(cnt):
                prev = prev + 1 + int(bits[i:i + wp], 2); i += wp; p[prev] = Fb
            planes[k] = p
    sign = bits[i:i + m]; i += m
    for idx, k in enumerate(range(N, 8)):
        if M[idx] == '0':
            planes[k] = np.array([int(c) for c in bits[i:i + m]]); i += m
    assert i == len(bits)
    A = sum(planes[k] << (7 - k) for k in range(8))
    return [int(a) if s == '1' else -int(a) for a, s in zip(A, sign)]


def iterative_encryption_keeps_pes(I, t=8, seed=1, maxit=100000):
    """Sec. 3.1.2: add R mod 256 repeatedly until the block's PEs equal the
    original ones; returns (all blocks terminated with identical PEs, max iterations)."""
    rng = np.random.default_rng(seed); E0 = abcs.block_pes(I, t)
    H, W = I.shape; bh, bw = H // t, W // t
    B = I[:bh * t, :bw * t].astype(np.int64).reshape(bh, t, bw, t).transpose(0, 2, 1, 3).reshape(-1, t, t)
    worst = 0
    for d in range(len(B)):
        R = int(rng.integers(0, 256)); b = B[d].copy(); it = 0
        while True:
            b = (b + R) % 256; it += 1
            if np.array_equal(abcs.block_pes(b, t)[0], E0[d]): break
            if it > maxit: return False, it
        worst = max(worst, it)
    return True, worst


def check(I, t=8, T=None, nblocks=None):
    E = abcs.block_pes(I, t); n, m = E.shape
    r = abcs.rate(I, t, T); T = r['T']
    sym = np.where(np.abs(E) > T, T + 1, E); v, c = np.unique(sym, return_counts=True)
    cw = canon(abcs.huff_lengths(dict(zip(v.tolist(), c.tolist()))))
    ec1, _ = abcs.ec_entropy(E, T); ec2 = abcs.ec_bitplanes(E, t)
    wp = math.ceil(math.log2(m))
    Fmax = max(k for k in range(1, m) if m - (1 + max(1, math.ceil(math.log2(k + 1))) + k * wp) > 0)
    A = np.abs(E); allz = np.stack([((A >> (8 - k)) & 1).sum(1) == 0 for k in range(1, 9)], 1)
    Ns = [next((k for k in range(8) if not allz[d, k]), 8) for d in range(n)]
    vv, cc = np.unique(Ns, return_counts=True)
    Ncode = canon(abcs.huff_lengths(dict(zip(vv.tolist(), cc.tolist()))))
    idx = range(n) if nblocks is None else np.random.default_rng(0).choice(n, nblocks, replace=False)
    ok = True
    for d in idx:
        e = E[d].tolist()
        b1 = enc1(e, T, cw); ok &= dec1(b1, m, T, cw, e) == e and len(b1) == 8 * m - ec1[d]
        b2, _ = enc2(E[d], t, Fmax, Ncode); ok &= dec2(b2, m, Fmax, Ncode) == e and len(b2) == 8 * m - ec2[d]
        if not ok:
            print('mismatch block', d, len(b1), 8 * m - ec1[d], len(b2), 8 * m - ec2[d]); return False
    return ok


if __name__ == '__main__':
    from PIL import Image
    for p in sys.argv[1:]:
        I = np.array(Image.open(p).convert('L'))
        print(p, 'all blocks round-trip with the counted lengths:', check(I),
              '| iterative encryption keeps every block PE (ok, max iterations):', iterative_encryption_keeps_pes(I))

In [ ]:
#@title Self-test of the engine (HKDF RFC 5869, AES SP 800-38A, exact round trip)
import sys; sys.path.insert(0, '/content/se_rdhei')
import numpy as np, serdhei, se_full as S
print('HKDF / AES self-tests:', serdhei.selftest())
I = (np.add.outer(np.arange(64), np.arange(64)) * 3 % 256).astype(np.uint8)
a1, a2 = S.synthetic_audio(1), S.synthetic_audio(2)
C, h, pl = S.encrypt_embed(I, S.DEFAULT_KEY, a1, a2, S.DEFAULT_NONCE)
R, pe = S.extract_decrypt(C, h, S.DEFAULT_KEY, a1, a2)
print('round trip exact:', np.array_equal(R, I) and np.array_equal(pe[:pl.size], pl))
import subprocess
subprocess.run([sys.executable, '/content/se_rdhei/spec_test_vector.py', '--check'], check=True)

## 2. Provide the datasets

Put each dataset in its own folder (sub-folders are fine), named:

```
USC-SIPI/   the 12 standard covers: Airplane, Baboon, Jetplane, Lena, Man, Tiffany,
            barbara, boat, goldhill, lake, livingroom, peppers  (the same files used in MATLAB)
BOSSbase/   BOSSbase 1.01 (all 10 000 or any subset)
BOWS-2/     BOWS-2 original images
UCID/       UCID v2 (colour TIFFs are converted exactly as SE_RDHEI.m does)
```

Any other folder is processed as an additional dataset. If a database holds more images than `N_PER_DB`, a fixed random sample (seed 2026) is drawn, and the list of files used is saved with the results.

Choose **one** option in *Settings* (`DATA_SOURCE`):

* **colab_files (default):** you have already uploaded the data through Colab's *Files* panel. Dataset folders, zip files (named after the dataset), or loose cover images in `/content` are all found automatically. Files there are deleted when the Colab session ends.

* **drive (recommended for large databases):** copy the folders into any Drive folder. If `DRIVE_DATA` below does not point to it, the notebook searches your Drive for the dataset folders and uses the first match.
* **upload:** you will be asked to upload one zip per dataset, named `USC-SIPI.zip`, `BOSSbase.zip`, `BOWS-2.zip`, `UCID.zip`. One zip that contains these folders also works.

**Optional, credentials A1/A2:** to use the same two audio credentials as the MATLAB run, run this in MATLAB and upload the two files when prompted (or place them in the Drive folder):

```matlab
load handel; save('A1_handel.mat','y','Fs');  load gong; save('A2_gong.mat','y','Fs');
```

WAV, FLAC and OGG files are also accepted, named so that they sort A1 first and A2 second. Without them, the deterministic synthetic clips of `SE_RDHEI.m` are used. Capacity results do not depend on the credentials. Statistics agree either way, though not digit for digit, because the payload bits are random.

In [ ]:
#@title Settings
DATA_SOURCE = 'colab_files'   #@param ["colab_files", "drive", "upload"]
# colab_files: datasets already uploaded through the Files panel (folders or zip files in /content)
# drive: folders in Google Drive;  upload: you are asked to upload zip files now
DRIVE_DATA = '/content/drive/MyDrive/SE-RDHEI-data'   #@param {type:"string"}
DRIVE_OUT = '/content/drive/MyDrive/SE-RDHEI-results-B2'   #@param {type:"string"}
N_PER_DB = 500   #@param {type:"integer"}
TRIALS = 10   #@param {type:"integer"}
UPLOAD_AUDIO = False   #@param {type:"boolean"}
# map file names to cover names if automatic matching misses one, e.g. {'4.2.04_gray': 'Lena'}
RENAME = {}
# give a sub-folder its own dataset name, or 'skip' to leave it out.
# A folder named BOSS-2 is read as BOWS-2 (the copy used in the MATLAB run was stored under that name).
DATASET_MAP = {'BOSS-2': 'BOWS-2', 'BOSS2': 'BOWS-2', 'BOSE-2': 'BOWS-2', 'BOWS2': 'BOWS-2'}

In [ ]:
#@title Load the data
import os, glob, zipfile, shutil, json, re
DATA, OUT = '/content/data', '/content/results_B2'
A1 = A2 = None
DS_KEYS = ('usc-sipi', 'usc_sipi', 'uscsipi', 'sipi', 'bossbase', 'boss', 'bows', 'ucid')
_BOWS_ALIASES = ('bows', 'boss-2', 'boss2', 'boss_2', 'bose-2', 'bose2')
CANON = lambda n: ('USC-SIPI' if 'sipi' in n.lower() or n.lower().startswith('usc') else
                   'BOWS-2' if n.lower().startswith(_BOWS_ALIASES) else
                   'BOSSbase' if n.lower().startswith('boss') else
                   'BOWS-2' if n.lower().startswith('bows') else
                   'UCID' if n.lower().startswith('ucid') else None)
AUDIO_EXT = ('.mat', '.wav', '.flac', '.ogg')
SKIP = {'drive', 'sample_data', 'se_rdhei', 'results', 'data', '.config'}

def link_datasets(search_roots, depth=4):
    """Find dataset folders below the roots and link them into /content/data/<name>."""
    os.makedirs(DATA, exist_ok=True)
    found = {}
    for r in search_roots:
        for cur, dirs, _ in os.walk(r):
            if cur.count(os.sep) - r.count(os.sep) >= depth:
                dirs[:] = []; continue
            if cur == r:
                dirs[:] = [d for d in dirs if d not in SKIP]
            dirs[:] = [d for d in dirs if not d.startswith(('.', '__'))]
            keep = []
            for d in dirs:
                c = CANON(d)
                if c and c not in found:
                    found[c] = os.path.join(cur, d)
                else:
                    keep.append(d)
            dirs[:] = keep
    for c, src in found.items():
        dst = os.path.join(DATA, c)
        if os.path.abspath(src) != os.path.abspath(dst) and not os.path.exists(dst):
            os.symlink(src, dst)
        print('  %-9s <- %s' % (c, src))
    return found

def _kind(path):
    """What an archive that is not a valid zip actually is, from its first bytes."""
    with open(path, 'rb') as f:
        h = f.read(512)
    if h.startswith(b'Rar!'): return 'a RAR archive renamed .zip'
    if h.startswith(b"7z\xbc\xaf\x27\x1c"): return 'a 7-Zip archive renamed .zip'
    if h.startswith(b'\x1f\x8b'): return 'a gzip/tar.gz archive renamed .zip'
    if b'<html' in h.lower() or b'<!doctype' in h.lower(): return 'a web page (a failed or blocked download), not the dataset'
    if h.startswith(b'PK'): return 'an incomplete zip (upload or download not finished, or the file is truncated)'
    return 'not a zip file'

def unzip_found(root, depth=2):
    bad = []
    import tarfile
    for t in [f for pat in ('*.tar', '*.tgz', '*.tar.gz', '*.tar.bz2', '*.tar.xz')
              for f in glob.glob(os.path.join(root, pat)) + glob.glob(os.path.join(root, '*', pat))]:
        if '/drive/' in t and root == '/content':
            continue
        c = CANON(os.path.basename(t)) or ('BOWS-2' if 'bows' in t.lower() else None)
        dest = os.path.join(DATA, c) if c else DATA
        mark = os.path.join(dest, '.unzipped_' + os.path.basename(t))
        if os.path.exists(mark):
            continue
        os.makedirs(dest, exist_ok=True)
        print('  extracting', os.path.basename(t), '->', dest, flush=True)
        with tarfile.open(t) as f:
            f.extractall(dest)
        open(mark, 'w').close()
    for z in glob.glob(os.path.join(root, '*.zip')) + glob.glob(os.path.join(root, '*', '*.zip')):
        if '/drive/' in z or '/results' in z or 'results_SE-RDHEI' in z:
            continue
        c = CANON(os.path.basename(z))
        dest = os.path.join(DATA, c) if c else DATA
        mark = os.path.join(dest, '.unzipped_' + os.path.basename(z))
        if os.path.exists(mark):
            continue
        if not zipfile.is_zipfile(z):
            bad.append((z, os.path.getsize(z), _kind(z)))
            continue
        os.makedirs(dest, exist_ok=True)
        print('  extracting', os.path.basename(z), '->', dest, flush=True)
        try:
            with zipfile.ZipFile(z) as f:
                f.extractall(dest)
        except (zipfile.BadZipFile, EOFError, OSError) as e:
            bad.append((z, os.path.getsize(z), 'damaged inside (%s)' % e)); continue
        open(mark, 'w').close()
    if bad:
        print()
        for z, n, k in bad:
            print('  PROBLEM: %s (%.1f MB) is %s.' % (os.path.basename(z), n / 2**20, k))
        raise RuntimeError(
            'Some uploaded archives cannot be opened (listed above). If an upload is still in progress in the '
            'Files panel, wait for it to finish and run this cell again. Otherwise delete the file, re-download '
            'or re-zip the folder, upload it again, and rerun this cell. Nothing has been computed yet.')

def cover_folder(root, depth=3):
    """A folder of standard covers under any name (e.g. 'test image') is linked as USC-SIPI."""
    import sys; sys.path.insert(0, '/content/se_rdhei'); import run_all
    if os.path.exists(os.path.join(DATA, 'USC-SIPI')):
        return
    for cur, dirs, fs in os.walk(root):
        if cur.count(os.sep) - root.count(os.sep) >= depth or '/drive' in cur or cur.startswith(DATA):
            dirs[:] = []; continue
        dirs[:] = [d for d in dirs if d not in SKIP and not d.startswith(('.', '__'))]
        stems = [os.path.splitext(f)[0] for f in fs if f.lower().endswith(run_all.IMG_EXT)]
        if sum(run_all.canon_cover(x, RENAME) in run_all.ORDER for x in stems) >= 3:
            os.makedirs(DATA, exist_ok=True)
            os.symlink(cur, os.path.join(DATA, 'USC-SIPI'))
            print('  USC-SIPI  <- %s (standard covers)' % cur)
            return

def loose_covers(root):
    """Cover images uploaded directly into /content are treated as USC-SIPI."""
    import sys; sys.path.insert(0, '/content/se_rdhei'); import run_all
    imgs = [f for f in glob.glob(os.path.join(root, '*')) if f.lower().endswith(run_all.IMG_EXT)]
    hits = [f for f in imgs if run_all.canon_cover(os.path.splitext(os.path.basename(f))[0], RENAME) in run_all.ORDER
            or os.path.splitext(os.path.basename(f))[0] in RENAME]
    if hits:
        d = os.path.join(DATA, 'USC-SIPI')
        if os.path.islink(d):
            return
        os.makedirs(d, exist_ok=True)
        for f in hits:
            t = os.path.join(d, os.path.basename(f))
            if not os.path.exists(t):
                os.symlink(f, t)
        print('  USC-SIPI  <- %d cover images found directly in %s' % (len(hits), root))

def find_audio(dirs):
    aud = []
    for d in dirs:
        aud += [f for f in glob.glob(os.path.join(d, '*')) if f.lower().endswith(AUDIO_EXT)]
    aud = sorted(set(aud))
    return aud[:2] if len(aud) >= 2 else (None, None)

if DATA_SOURCE == 'colab_files':
    print('Looking for datasets uploaded to /content ...')
    unzip_found('/content')
    link_datasets(['/content'])
    cover_folder('/content')
    loose_covers('/content')
    A1, A2 = find_audio(['/content', DATA])
elif DATA_SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive')
    base = DRIVE_DATA if os.path.isdir(DRIVE_DATA) else '/content/drive/MyDrive'
    print('Looking for datasets in', base, '...')
    unzip_found(base)
    _f = link_datasets([base]); cover_folder(base)
    if not _f:
        print('  nothing found. Top level of your Drive:')
        for e in sorted(os.listdir('/content/drive/MyDrive'))[:60]:
            print('    ', e)
        raise SystemExit('Put USC-SIPI/, BOSSbase/, BOWS-2/, UCID/ in Drive and set DRIVE_DATA.')
    OUT = DRIVE_OUT
    A1, A2 = find_audio([base])
else:
    from google.colab import files
    os.makedirs(DATA, exist_ok=True)
    print('Upload one zip per dataset (USC-SIPI.zip, BOSSbase.zip, BOWS-2.zip, UCID.zip):')
    up = files.upload()
    for name in up:
        c = CANON(name); dest = os.path.join(DATA, c) if c else DATA
        os.makedirs(dest, exist_ok=True)
        with zipfile.ZipFile(name) as z:
            z.extractall(dest)
        os.remove(name); print('  extracted', name, '->', dest)
    if UPLOAD_AUDIO:
        print('Upload the two credential files (A1 first, A2 second):')
        au = files.upload(); os.makedirs('/content/audio', exist_ok=True)
        for n in au:
            shutil.move(n, '/content/audio/' + n)
        A1, A2 = find_audio(['/content/audio'])
os.makedirs(OUT, exist_ok=True)

import sys; sys.path.insert(0, '/content/se_rdhei')
import importlib, run_all; importlib.reload(run_all)
sets = run_all.discover(DATA, N_PER_DB, 2026, RENAME, DATASET_MAP)
from PIL import Image as _Im
for ds, m in sets.items():
    _f = next(iter(m.values()))
    try:
        _sz = '%dx%d' % _Im.open(_f).size
    except Exception:
        _sz = '?'
    print('%-10s %5d images   e.g. %s  (%s)' % (ds, len(m), os.path.relpath(_f, DATA), _sz))
_miss = [d for d in ('USC-SIPI', 'BOSSbase', 'BOWS-2', 'UCID') if d not in sets]
if _miss:
    print('\n' + '!' * 70)
    print('  NOT FOUND: ' + ', '.join(_miss))
    print('  Folders seen in %s:' % DATA)
    for _r, _ds, _fs in os.walk(DATA, followlinks=True):
        _n = sum(f.lower().endswith(run_all.IMG_EXT) for f in _fs)
        if _n:
            print('    %6d images in %s' % (_n, os.path.relpath(_r, DATA)))
    print('  If one of these is the missing dataset, add its folder name to DATASET_MAP,')
    print('  e.g. DATASET_MAP = {"my_bows_folder": "BOWS-2"}, and run this cell again.')
    print('!' * 70)
if 'USC-SIPI' in sets:
    print('\nUSC-SIPI cover names (must read Airplane, Baboon, ... ; fix any miss with RENAME):')
    for nm, p in sets['USC-SIPI'].items():
        print('  %-11s <- %s' % (nm, os.path.basename(p)))
    miss = [c for c in run_all.ORDER if c not in sets['USC-SIPI']]
    if miss:
        print('  missing:', ', '.join(miss))
print('\ncredentials:', (os.path.basename(A1) + ', ' + os.path.basename(A2)) if A1 else 'synthetic (SE_RDHEI.m fallback)')

## 3. Run all experiments
This step streams progress per image. If Colab disconnects, run it again and it continues from the cache. To recompute everything from scratch, delete the output folder first.

In [ ]:
#@title Run
import subprocess, sys
cmd = [sys.executable, '-u', '/content/se_rdhei/run_all.py', '--data', DATA, '--out', OUT,
       '--n', str(N_PER_DB), '--trials', str(TRIALS), '--rename', json.dumps(RENAME),
       '--dsmap', json.dumps(DATASET_MAP)]
if A1 and A2:
    cmd += ['--a1', A1, '--a2', A2]
with open(os.path.join(OUT, 'run_log.txt'), 'a') as log:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end=''); log.write(line)
    p.wait()
print('\nexit code', p.returncode)
# the bundle is rebuilt so it also carries the complete log
if p.returncode == 0:
    subprocess.run([sys.executable, '/content/se_rdhei/run_all.py', '--data', DATA, '--out', OUT,
                    '--n', str(N_PER_DB), '--trials', str(TRIALS), '--rename', json.dumps(RENAME),
                    '--dsmap', json.dumps(DATASET_MAP), '--stage', 'report'] + (['--a1', A1, '--a2', A2] if A1 and A2 else []),
                   stdout=subprocess.DEVNULL)

## 4. Results

In [ ]:
#@title Report
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(OUT, 'REPORT.md')).read()))

In [ ]:
#@title Figures
from IPython.display import Image as _Im
import glob
for f in sorted(glob.glob(os.path.join(OUT, 'work', 'p_fig_*.png'))):
    print(os.path.basename(f)); display(_Im(f, width=900))
print('All figures (PDF) are in', os.path.join(OUT, 'figures'))

## 5. Download and send back
Send **`results_SE-RDHEI.zip`** back. The cell below tries three routes: a browser download, a copy in your Google Drive (*My Drive/results_SE-RDHEI.zip*), and the file in the Files panel (right-click → Download). It holds the report, all JSON result files, the per-image CSV, the figures, the run log and the list of files used.

In [ ]:
#@title Download results_SE-RDHEI.zip
SAVE_COPY_TO_DRIVE = True   #@param {type:"boolean"}
import os, shutil, subprocess, sys
z = os.path.join(OUT, 'results_SE-RDHEI.zip')
if not os.path.exists(z):                       # rebuild the bundle from the cached results
    print('bundle not found, rebuilding from the cache ...')
    subprocess.run([sys.executable, '/content/se_rdhei/run_all.py', '--data', DATA, '--out', OUT,
                    '--n', str(N_PER_DB), '--trials', str(TRIALS), '--rename', json.dumps(RENAME),
                    '--dsmap', json.dumps(DATASET_MAP), '--stage', 'report']
                   + (['--a1', A1, '--a2', A2] if A1 and A2 else []), check=True)
top = '/content/results_SE-RDHEI.zip'
if os.path.abspath(z) != top:
    shutil.copy(z, top)
print('results: %s (%.1f MB)' % (top, os.path.getsize(top) / 2**20))

# 1. browser download (browsers sometimes block this silently)
try:
    from google.colab import files
    files.download(top)
    print('1. browser download started (check your Downloads folder)')
except Exception as e:
    print('1. browser download failed:', e)

# 2. a copy in Google Drive, which always works
if SAVE_COPY_TO_DRIVE:
    from google.colab import drive
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    dst = '/content/drive/MyDrive/results_SE-RDHEI.zip'
    shutil.copy(top, dst)
    print('2. copy saved to Google Drive: My Drive/results_SE-RDHEI.zip')

print('3. or: Files panel on the left -> results_SE-RDHEI.zip -> right-click -> Download')